# Sewer Network Expansion — Non-Budgeted Multi-Heuristic Pipeline
**Larnaca District · Cyprus**

This notebook prepares the wastewater-demand and road-network inputs, reviews existing sewer coverage, selects a candidate WWTP2 location, and compares three non-budgeted routing heuristics.

The notebook is organised as a single numbered sequence, from **Block 1** to **Block 24**. Shared preprocessing outputs are written directly under `NetworkNonBudgeted/`; heuristic-specific outputs are written inside the corresponding subfolder for each routing method.


---
## BLOCK 1 — Setup, Paths and Parameters


In [1]:
# ============================================================
# BLOCK 1 -- SETUP, PATHS AND PARAMETERS
# ============================================================

from pathlib import Path
import json
import math
import gzip
import requests
import warnings
from collections import defaultdict, deque

import pandas as pd
import geopandas as gpd
import numpy as np
import networkx as nx
from shapely.geometry import Point, LineString, shape
from shapely.ops import unary_union, nearest_points

try:
    import osmnx as ox
    HAS_OSMNX = True
except Exception:
    HAS_OSMNX = False

try:
    import folium
    from folium.plugins import MiniMap, Fullscreen
    HAS_FOLIUM = True
except Exception:
    HAS_FOLIUM = False

try:
    import fiona
    HAS_FIONA = True
except Exception:
    HAS_FIONA = False


# ------------------------------------------------------------
# MAIN PROJECT PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(r"C:\Users\lucam\OneDrive\Desktop\Cyprus Project")

AREAS_DIR  = PROJECT_DIR / "areas"
ROADS_PATH = PROJECT_DIR / "requested_shapefiles" / "gis_osm_roads_free_1.shp"

# ------------------------------------------------------------
# OUTPUT DIRECTORY STRUCTURE
# ------------------------------------------------------------
# NetworkNonBudgeted/                          <- shared, heuristic-independent outputs
#   01_boundaries_and_urban_footprint.gpkg
#   02_demand_nodes_population.gpkg
#   03_road_graph.gpkg
#   04_existing_coverage_review.gpkg
#   wwtp2_site_selection.gpkg
#   heuristic_rooted_dijkstra/                 <- heuristic-specific outputs
#     new_pipes_optimisation.gpkg
#     pipe_cost_by_municipality.csv
#     final_network_map.html
#     ...
#   heuristic_prim_steiner/
#     ...

#     ...

OUTPUT_DIR = PROJECT_DIR / "Results" / "NetworkNonBudgeted"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HEURISTICS = ["rooted_dijkstra", "prim_steiner"]

HEURISTIC_DIRS = {}
for h in HEURISTICS:
    d = OUTPUT_DIR / f"heuristic_{h}"
    d.mkdir(parents=True, exist_ok=True)
    HEURISTIC_DIRS[h] = d


def shared_path(filename):
    """Path for an output that is identical across all heuristics."""
    return OUTPUT_DIR / filename


def heuristic_path(heuristic, filename):
    """Path for an output that differs per heuristic."""
    return HEURISTIC_DIRS[heuristic] / filename


# ------------------------------------------------------------
# LOCAL INFRASTRUCTURE DATA
# ------------------------------------------------------------

LOCAL_DATA = {
    "areas":      AREAS_DIR / "areas.gpkg",
    "conduits":   AREAS_DIR / "conduits.gpkg",
    "consumers":  AREAS_DIR / "consumers.gpkg",
    "manholes":   AREAS_DIR / "manholes.gpkg",
    "outfalls":   AREAS_DIR / "outfalls.gpkg",
    "pumps":      AREAS_DIR / "pumps.gpkg",
    "storages":   AREAS_DIR / "storages.gpkg",
    "flows":      AREAS_DIR / "flows.xlsx",
    "ps_inflows": AREAS_DIR / "ps_inflows.xlsx",
}


# ------------------------------------------------------------
# MUNICIPALITIES
# ------------------------------------------------------------

ALL_MUNICIPALITIES = [
    "Larnaca", "Aradippou", "Livadia", "Oroklini", "Kellia",
    "Dromolaxia-Meneou", "Perivolia", "Kiti", "Pyla",
]

NORTH_MUNICIPALITIES = ["Oroklini", "Kellia", "Livadia", "Aradippou", "Pyla"]
SOUTH_MUNICIPALITIES = ["Larnaca", "Dromolaxia-Meneou", "Perivolia", "Kiti"]

# Municipalities where the new WWTP2 is allowed to be physically sited.
# Constraint: must be feasible to build (not deep inside dense urban fabric)
# and reasonably close to the demand it serves.
WWTP2_ADMISSIBLE_MUNICIPALITIES = ["Kellia", "Oroklini", "Aradippou"]

WWTP_ASSIGNMENT = {
    **{m: "WWTP_2_NORTH" for m in NORTH_MUNICIPALITIES},
    **{m: "WWTP_1_SOUTH" for m in SOUTH_MUNICIPALITIES},
}


# ------------------------------------------------------------
# POPULATION ASSUMPTIONS
# IMPORTANT: replace these values with the official/latest values
# ------------------------------------------------------------

MUNICIPALITY_POPULATION = {
    # Census 2021 (CYSTAT) — confirmed:
    "Larnaca": 52046, "Aradippou": 22932, "Livadia": 8581,
    "Oroklini": 7575, "Dromolaxia-Meneou": 6838, "Kiti": 5136,
    # Secondary source (see A4) — CONFIRM these match your cited source:
    "Kellia": 387, "Perivolia": 3920, "Pyla": 2845,
}
# ------------------------------------------------------------
# MODELLING PARAMETERS
# ------------------------------------------------------------

CRS_WGS84     = "EPSG:4326"
CRS_PROJECTED = "EPSG:32636"  # Cyprus / UTM zone 36N

PIPE_COST_EUR_PER_M         = 400.0
WASTEWATER_L_PER_CAPITA_DAY = 120.0

URBAN_BUILDING_BUFFER_M = 80.0
MIN_CLUSTER_BUILDINGS   = 20
MIN_CLUSTER_AREA_M2     = 25_000.0
KEEP_ONLY_LARGEST_URBAN_CLUSTER = False
ROAD_BUFFER_AROUND_URBAN_M = 250.0
USE_URBAN_FOOTPRINT_FOR_DEMAND = True

# ------------------------------------------------------------
# WWTP2 site selection parameters (Block 17)
# ------------------------------------------------------------
# The optimal WWTP2 site is a population-weighted centroid computed only
# over WWTP2_ADMISSIBLE_MUNICIPALITIES, then constrained to fall outside
# the urban footprint (buildability) and inside the admissible territory
# (feasibility), then snapped to the nearest road node.

WWTP2_MIN_DIST_FROM_URBAN_M = 150.0   # must clear the urban footprint by at least this much
WWTP2_MAX_SNAP_DIST_M       = 400.0   # sanity check: site-to-road-node snap should not exceed this

# This will be set by Block 17 -- do not edit directly.
WWTP2_LONLAT = None

print("=" * 70)
print("BLOCK 1  Configuration")
print("=" * 70)
print(f"Output directory : {OUTPUT_DIR}")
print(f"Heuristics        : {HEURISTICS}")
for name, path in LOCAL_DATA.items():
    print(f"  {name:12s} -> {path.exists()} | {path}")
print(f"  roads        -> {ROADS_PATH.exists()} | {ROADS_PATH}")


BLOCK 1  Configuration
Output directory : C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted
Heuristics        : ['rooted_dijkstra', 'prim_steiner']
  areas        -> True | C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\areas.gpkg
  conduits     -> True | C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\conduits.gpkg
  consumers    -> True | C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\consumers.gpkg
  manholes     -> True | C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\manholes.gpkg
  outfalls     -> True | C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\outfalls.gpkg
  pumps        -> True | C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\pumps.gpkg
  storages     -> True | C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\storages.gpkg
  flows        -> True | C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\flows.xlsx
  ps_inflows   -> True | C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\ps_inflows.xlsx
  road

In [2]:
# ============================================================
# BLOCK 2 -- UTILITY FUNCTIONS
# ============================================================

def norm_name(s):
    """Normalise municipality names for robust matching."""
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    replacements = {
        "–": "-",
        "—": "-",
        "_": "-",
        " ": "-",
        "dromolaxia meneou": "dromolaxia-meneou",
        "dromolaxia-meneou": "dromolaxia-meneou",
        "voroklini": "oroklini",
        "oroqlini": "oroklini",
        "larnaka": "larnaca",
    }
    for a, b in replacements.items():
        s = s.replace(a, b)
    return s


def read_gpkg_first_layer(path):
    """Read the first layer of a GeoPackage."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if HAS_FIONA:
        layers = fiona.listlayers(path)
        print(f"{path.name}: layers = {layers}")
        return gpd.read_file(path, layer=layers[0])
    return gpd.read_file(path)


def safe_to_crs(gdf, crs):
    if gdf.crs is None:
        raise ValueError("GeoDataFrame has no CRS. Define CRS before transforming.")
    return gdf.to_crs(crs)


def ensure_polygon_gdf(gdf):
    """Keep only polygonal geometries."""
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    return gdf


def ensure_line_gdf(gdf):
    """Keep only line geometries."""
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
    return gdf


def explode_lines_to_segments(line_geom):
    """Convert LineString/MultiLineString to small consecutive LineString segments."""
    if line_geom is None or line_geom.is_empty:
        return []

    lines = []
    if line_geom.geom_type == "LineString":
        lines = [line_geom]
    elif line_geom.geom_type == "MultiLineString":
        lines = list(line_geom.geoms)
    else:
        return []

    segments = []
    for line in lines:
        coords = list(line.coords)
        for a, b in zip(coords[:-1], coords[1:]):
            if a != b:
                segments.append(LineString([a, b]))
    return segments


def nearest_node_id(points_gdf, nodes_gdf, point_id_col=None):
    """
    Assign each point to nearest road node using geopandas sjoin_nearest.
    points_gdf and nodes_gdf must be in projected CRS.
    """
    pts = points_gdf.copy()
    if point_id_col is None:
        pts["_tmp_pid"] = range(len(pts))
        point_id_col = "_tmp_pid"

    nodes = nodes_gdf[["node_id", "geometry"]].copy()

    joined = gpd.sjoin_nearest(
        pts[[point_id_col, "geometry"]],
        nodes,
        how="left",
        distance_col="snap_dist_m"
    )

    return joined[[point_id_col, "node_id", "snap_dist_m"]]

In [3]:
# ============================================================
# BLOCK 3 -- LOAD INFRASTRUCTURE, FLOW TABLES AND ROAD DATA
# ============================================================

import sys
import subprocess
import importlib

# ------------------------------------------------------------
# Ensure openpyxl is installed for reading .xlsx files
# ------------------------------------------------------------

def ensure_package(package_name):
    """
    Try importing a package. If missing, install it in the current Python environment.
    """
    try:
        return importlib.import_module(package_name)
    except ImportError:
        print(f"{package_name} not found. Installing it now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"{package_name} installed. Importing again...")
        return importlib.import_module(package_name)

openpyxl = ensure_package("openpyxl")


# ------------------------------------------------------------
# Load GeoPackage infrastructure layers
# ------------------------------------------------------------

local_gdfs = {}

for key, path in LOCAL_DATA.items():
    if path.suffix.lower() == ".gpkg" and path.exists():
        try:
            gdf = read_gpkg_first_layer(path)
            print(f"\nLoaded {key}: {len(gdf)} rows | CRS = {gdf.crs}")
            print(gdf.columns.tolist())
            local_gdfs[key] = gdf
        except Exception as e:
            print(f"Could not load {key}: {e}")


# ------------------------------------------------------------
# Load Excel files
# ------------------------------------------------------------

flows_df = None
ps_inflows_df = None

if LOCAL_DATA["flows"].exists():
    flows_df = pd.read_excel(LOCAL_DATA["flows"], engine="openpyxl")
    print("\nflows.xlsx")
    print(flows_df.head())
    print(flows_df.columns.tolist())
else:
    print("\nflows.xlsx not found:", LOCAL_DATA["flows"])

if LOCAL_DATA["ps_inflows"].exists():
    ps_inflows_df = pd.read_excel(LOCAL_DATA["ps_inflows"], engine="openpyxl")
    print("\nps_inflows.xlsx")
    print(ps_inflows_df.head())
    print(ps_inflows_df.columns.tolist())
else:
    print("\nps_inflows.xlsx not found:", LOCAL_DATA["ps_inflows"])


# ------------------------------------------------------------
# Load local road shapefile
# ------------------------------------------------------------

roads = gpd.read_file(ROADS_PATH)

print("\nRoads loaded:")
print(len(roads), roads.crs)
print(roads.columns.tolist())

roads = ensure_line_gdf(roads)

if roads.crs is None:
    print("WARNING: Roads have no CRS. Assuming EPSG:4326.")
    roads = roads.set_crs(CRS_WGS84)

roads_p = roads.to_crs(CRS_PROJECTED)

print("\nRoads projected:")
print(len(roads_p), roads_p.crs)

areas.gpkg: layers = ['areas']

Loaded areas: 25 rows | CRS = EPSG:6312
['Area', 'geometry']
conduits.gpkg: layers = ['edams_sewer_pipesnew']


C:\Users\lucam\anaconda3\envs\Project_Goosee\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\conduits.gpkg' may only be partially supported
  return ogr_read(



Loaded conduits: 6380 rows | CRS = PROJCS["LTM",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",33],PARAMETER["scale_factor",0.99995],PARAMETER["false_easting",200000],PARAMETER["false_northing",-3500000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
['DATA', 'Label', 'Angle', 'geometry']
consumers.gpkg: layers = ['consumers']

Loaded consumers: 3891 rows | CRS = EPSG:6312
['PROP_REF', 'METER_REF', 'BILLGROUP', 'X_COORD', 'Y_COORD', 'PDOP', 'STATUS', 'STATUS_DAT', 'ID1', 'Matched', 'M3', 'Address', 'match', 'layer', 'Area', 'geometry']
manholes.gpkg: layers = ['manholes']

Loaded manholes: 5794 rows | CRS = EPSG:6312
['Name', 'TOPOFMANHO', 'Elevation', 'MaxDepth', 'InitDepth', 'SurDepth', 'Aponded', 'Area', 'Hydraulic'

C:\Users\lucam\anaconda3\envs\Project_Goosee\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\storages.gpkg' may only be partially supported
  return ogr_read(



Loaded storages: 22 rows | CRS = EPSG:6312
['Name', 'Elevation', 'MaxDepth', 'InitDepth', 'SurDepth', 'Type', 'Curve', 'Coeff', 'Exponent', 'MajorAxis', 'MinorAxis', 'SideSlope', 'SurfHeight', 'Fevap', 'Psi', 'Ksat', 'IMD', 'Area', 'Hydraulic', 'Constant', 'geometry']

flows.xlsx
  PS_Out PS_In Node_In  Losses  Delay
0     Z1    A1     NaN       0     15
1      W    A1     NaN       0     15
2      J    A1     NaN       0     15
3      Q    A1     NaN       0     15
4     C1    B1     B29       0     15
['PS_Out', 'PS_In', 'Node_In', 'Losses', 'Delay']

ps_inflows.xlsx
  pump_station    daily_avg  regional_daily_avg
0           A1  3806.306262         2285.017243
1           B1  7623.994820         1008.376698
2           C1  5939.263675          885.010311
3           D1  4340.713640         2322.910442
4           D2   298.891211          298.891211
['pump_station', 'daily_avg', 'regional_daily_avg']

Roads loaded:
193874 EPSG:4326
['osm_id', 'code', 'fclass', 'name', 'ref', 'oneway

In [4]:
# ============================================================
# BLOCK 4 -- MUNICIPAL BOUNDARY DOWNLOAD AND OSM SOURCE TRACKING
# ============================================================
# This version follows the approach that worked in the old
# Cyprus Graph Definition notebook:
# - try candidate names in order;
# - accept the first valid polygon returned by OSMnx;
# - do NOT sort by largest polygon, because that caused wrong
#   geometries for Livadia.

if not HAS_OSMNX:
    raise ImportError("osmnx is not installed. Install with: pip install osmnx")

boundary_cache = OUTPUT_DIR / "municipal_boundaries_raw.gpkg"

FORCE_REDOWNLOAD_BOUNDARIES = True

if boundary_cache.exists() and FORCE_REDOWNLOAD_BOUNDARIES:
    boundary_cache.unlink()
    print("Deleted old boundary cache:", boundary_cache)

# ------------------------------------------------------------
# Candidate names copied/adapted from the old working notebook
# ------------------------------------------------------------

boundary_queries = {
    "Larnaca": [
        "Larnaca, Cyprus",
        "Larnaka, Cyprus",
        "Larnaca Municipality, Cyprus",
    ],

    "Aradippou": [
        "Aradippou, Cyprus",
        "Aradippou, Larnaca, Cyprus",
        "Aradhippou, Cyprus",
        "Aradippou Municipality, Cyprus",
    ],

    "Livadia": [
        "Livadia, Larnaca, Cyprus",
        "Livadia, Cyprus",
        "Livadia Municipality, Cyprus",
        "Livadia, Larnaka, Cyprus",
        "Livadhia, Larnaca, Cyprus",
    ],

    "Oroklini": [
        "Oroklini, Cyprus",
        "Voroklini, Cyprus",
        "Oroklini, Larnaca, Cyprus",
        "Voroklini, Larnaca, Cyprus",
    ],

    "Kellia": [
        "Kellia, Cyprus",
        "Kelia, Cyprus",
        "Kellia, Larnaca, Cyprus",
        "Kelia, Larnaca, Cyprus",
    ],

    # In the old notebook this was effectively Dromolaxia first.
    # If you want the combined Dromolaxia-Meneou zone later, we can
    # explicitly union Dromolaxia and Meneou.
    "Dromolaxia-Meneou": [
        "Dromolaxia, Cyprus",
        "Meneou, Cyprus",
        "Dromolaxia-Meneou, Cyprus",
        "Dromolaxia Meneou, Cyprus",
    ],

    "Perivolia": [
        "Perivolia, Larnaca, Cyprus",
        "Pervolia, Larnaca, Cyprus",
        "Perivolia, Cyprus",
        "Pervolia, Cyprus",
    ],

    "Kiti": [
        "Kiti, Cyprus",
        "Kiti, Larnaca, Cyprus",
    ],

    "Pyla": [
        "Pyla, Cyprus",
        "Pyla, Larnaca, Cyprus",
        "Pile, Cyprus",
    ],
}


def geocode_first_polygon_old_style(place_candidates, muni_name):
    """
    Try multiple place-name strings until OSMnx returns a valid polygon.
    This intentionally follows the old notebook logic:
    use the first valid geocoding result, not the largest one.
    """
    last_err = None

    for place in place_candidates:
        try:
            print(f"  Trying: {place}")

            gdf = ox.geocode_to_gdf(place)

            if gdf.empty:
                print("    Empty geocoding result.")
                continue

            gdf = gdf.to_crs(CRS_WGS84)
            gdf = gdf[gdf.geometry.notna()].copy()
            gdf = gdf[~gdf.geometry.is_empty].copy()

            if gdf.empty:
                print("    Geometry empty after cleaning.")
                continue

            # Keep polygonal geometries, but do NOT sort by area.
            poly = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

            if poly.empty:
                print("    Found result, but not polygon/multipolygon.")
                continue

            # Old behaviour: first polygon returned.
            out = poly.iloc[[0]].copy()
            out["municipality"] = muni_name
            out = out[["municipality", "geometry"]]

            area_km2 = float(out.to_crs(CRS_PROJECTED).geometry.area.iloc[0]) / 1e6

            print(f"    OK: {place} | area = {area_km2:.2f} km²")

            # Safety check only: reject absurdly tiny polygons.
            # Livadia should return thousands of buildings, not 4.
            if muni_name == "Livadia" and area_km2 < 5.0:
                print(f"    Rejected Livadia: area too small ({area_km2:.2f} km²)")
                continue

            return out, place, area_km2

        except Exception as e:
            last_err = e
            print(f"    Failed: {repr(e)}")

    raise RuntimeError(
        f"Could not geocode valid polygon for {muni_name}. "
        f"Candidates: {place_candidates}. Last error: {repr(last_err)}"
    )


if boundary_cache.exists():
    print("Loading cached municipal boundaries...")
    muni_boundaries = gpd.read_file(boundary_cache)

else:
    boundaries = []
    failed = []
    used_places = []

    for muni, queries in boundary_queries.items():
        print("\n" + "=" * 70)
        print(f"Downloading boundary for {muni}")
        print("=" * 70)

        try:
            b, used_place, area_km2 = geocode_first_polygon_old_style(queries, muni)
            boundaries.append(b)
            used_places.append({
                "municipality": muni,
                "used_place": used_place,
                "area_km2": area_km2,
            })

        except Exception as e:
            failed.append({
                "municipality": muni,
                "error": repr(e),
            })
            print(f"FAILED {muni}: {repr(e)}")

    if failed:
        print("\nWARNING: Some municipalities failed:")
        display(pd.DataFrame(failed))

    if not boundaries:
        raise RuntimeError("No boundaries downloaded.")

    muni_boundaries = pd.concat(boundaries, ignore_index=True)
    muni_boundaries = gpd.GeoDataFrame(
        muni_boundaries,
        geometry="geometry",
        crs=CRS_WGS84
    )

    muni_boundaries["muni_norm"] = muni_boundaries["municipality"].map(norm_name)

    used_places_df = pd.DataFrame(used_places)
    used_places_df.to_csv(OUTPUT_DIR / "municipal_boundary_osm_sources.csv", index=False)

    muni_boundaries.to_file(boundary_cache, driver="GPKG")

    print("\nSaved boundary cache:")
    print(boundary_cache)

    print("\nBoundary used places:")
    display(used_places_df)

muni_boundaries["muni_norm"] = muni_boundaries["municipality"].map(norm_name)
muni_boundaries_p = muni_boundaries.to_crs(CRS_PROJECTED)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

boundary_diag = muni_boundaries_p.copy()
boundary_diag["area_km2"] = boundary_diag.geometry.area / 1e6

print("\nDownloaded municipal boundaries:")
display(
    boundary_diag[["municipality", "muni_norm", "area_km2"]]
    .sort_values("municipality")
)

missing = sorted(set(ALL_MUNICIPALITIES) - set(muni_boundaries["municipality"]))

if missing:
    print("\nStill missing municipalities:")
    print(missing)
else:
    print("\nAll requested municipalities downloaded successfully.")

# Livadia sanity check
liv = boundary_diag[boundary_diag["municipality"] == "Livadia"]

if len(liv) > 0:
    liv_area = float(liv["area_km2"].iloc[0])
    print(f"\nLivadia area check: {liv_area:.2f} km²")

    if liv_area < 5.0:
        print("WARNING: Livadia boundary still looks too small.")
    else:
        print("Livadia boundary area looks plausible.")


  Trying: Larnaca, Cyprus
    OK: Larnaca, Cyprus | area = 32.88 km²

  Trying: Aradippou, Cyprus
    OK: Aradippou, Cyprus | area = 55.90 km²

  Trying: Livadia, Larnaca, Cyprus
    OK: Livadia, Larnaca, Cyprus | area = 8.63 km²

  Trying: Oroklini, Cyprus
    OK: Oroklini, Cyprus | area = 15.31 km²

  Trying: Kellia, Cyprus
    OK: Kellia, Cyprus | area = 12.54 km²

  Trying: Dromolaxia, Cyprus
    OK: Dromolaxia, Cyprus | area = 21.12 km²

  Trying: Perivolia, Larnaca, Cyprus
    Failed: TypeError("Nominatim did not geocode query 'Perivolia, Larnaca, Cyprus' to a geometry of type (Multi)Polygon.")
  Trying: Pervolia, Larnaca, Cyprus
    OK: Pervolia, Larnaca, Cyprus | area = 8.50 km²

  Trying: Kiti, Cyprus
    OK: Kiti, Cyprus | area = 8.75 km²

  Trying: Pyla, Cyprus
    OK: Pyla, Cyprus | area = 18.34 km²

Saved boundary cache:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\municipal_boundaries_raw.gpkg

Boundary used places:


,municipality,used_place,area_km2
0,Larnaca,"Larnaca, Cyprus",32.883059
1,Aradippou,"Aradippou, Cyprus",55.900128
2,Livadia,"Livadia, Larnaca, Cyprus",8.634687
3,Oroklini,"Oroklini, Cyprus",15.314135
4,Kellia,"Kellia, Cyprus",12.538846
5,Dromolaxia-Meneou,"Dromolaxia, Cyprus",21.121705
6,Perivolia,"Pervolia, Larnaca, Cyprus",8.496351
7,Kiti,"Kiti, Cyprus",8.751847
8,Pyla,"Pyla, Cyprus",18.335664



Downloaded municipal boundaries:


,municipality,muni_norm,area_km2
1,Aradippou,aradippou,55.900128
5,Dromolaxia-Meneou,dromolaxia-meneou,21.121705
4,Kellia,kellia,12.538846
7,Kiti,kiti,8.751847
0,Larnaca,larnaca,32.883059
2,Livadia,livadia,8.634687
3,Oroklini,oroklini,15.314135
6,Perivolia,perivolia,8.496351
8,Pyla,pyla,18.335664



All requested municipalities downloaded successfully.

Livadia area check: 8.63 km²
Livadia boundary area looks plausible.


In [5]:
# ============================================================
# BLOCK 5 -- MUNICIPAL BOUNDARY QUALITY CHECK
# ============================================================

boundary_check = muni_boundaries_p.copy()
boundary_check["area_km2"] = boundary_check.geometry.area / 1e6
boundary_check["centroid_x"] = boundary_check.geometry.centroid.x
boundary_check["centroid_y"] = boundary_check.geometry.centroid.y

print("Municipal boundary areas:")
display(
    boundary_check[["municipality", "area_km2", "centroid_x", "centroid_y"]]
    .sort_values("area_km2")
)

# Save quick boundary check file
boundary_check_path = OUTPUT_DIR / "municipal_boundary_check_layers.gpkg"
if boundary_check_path.exists():
    boundary_check_path.unlink()

boundary_check.to_file(boundary_check_path, layer="municipal_boundaries_check", driver="GPKG")

print("Saved boundary check:")
print(boundary_check_path)


# Optional quick map
if HAS_FOLIUM:
    b_w = boundary_check.to_crs(CRS_WGS84)
    center = b_w.geometry.unary_union.centroid

    m_check = folium.Map(
        location=[center.y, center.x],
        zoom_start=11,
        tiles="CartoDB positron"
    )

    for _, r in b_w.iterrows():
        folium.GeoJson(
            r.geometry,
            tooltip=f"{r['municipality']} | {r['area_km2']:.2f} km²",
            style_function=lambda x: {
                "fillColor": "#0076C2",
                "color": "#003B70",
                "weight": 2,
                "fillOpacity": 0.15,
            }
        ).add_to(m_check)

    folium.LayerControl(collapsed=False).add_to(m_check)

    out_boundary_html = OUTPUT_DIR / "municipal_boundary_check_map.html"
    m_check.save(out_boundary_html)

    print("Saved boundary check map:")
    print(out_boundary_html)

Municipal boundary areas:


,municipality,area_km2,centroid_x,centroid_y
6,Perivolia,8.496351,553942.615174,3.854378e+06
2,Livadia,8.634687,557863.765552,3.868354e+06
7,Kiti,8.751847,552561.660231,3.856229e+06
4,Kellia,12.538846,556804.125242,3.871863e+06
3,Oroklini,15.314135,560008.870449,3.871873e+06
8,Pyla,18.335664,562416.511110,3.873684e+06
5,Dromolaxia-Meneou,21.121705,553816.756250,3.859780e+06
0,Larnaca,32.883059,556634.441812,3.862193e+06
1,Aradippou,55.900128,552518.984969,3.868383e+06


Saved boundary check:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\municipal_boundary_check_layers.gpkg


C:\Users\lucam\AppData\Local\Temp\ipykernel_14652\1415561905.py:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = b_w.geometry.unary_union.centroid


Saved boundary check map:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\municipal_boundary_check_map.html


In [6]:
# ============================================================
# BLOCK 6 -- MICROSOFT BUILDING FOOTPRINTS AND MUNICIPALITY ASSIGNMENT
# ============================================================
# This block:
# 1. reads the Microsoft Global ML Building Footprints manifest;
# 2. computes the quadkeys covering the full study area;
# 3. downloads/parses the required Microsoft building tiles only once;
# 4. clips buildings to the union of the 9 municipal boundaries;
# 5. assigns each building to a municipality using spatial join;
# 6. saves a local cache for faster re-use.

import sys
import subprocess
import importlib
import gzip
import json
import requests
import io
from pathlib import Path
from shapely.geometry import shape as shapely_shape

# ------------------------------------------------------------
# Ensure mercantile is installed
# ------------------------------------------------------------

def ensure_package(package_name):
    try:
        return importlib.import_module(package_name)
    except ImportError:
        print(f"{package_name} not found. Installing it now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"{package_name} installed. Importing again...")
        return importlib.import_module(package_name)

mercantile = ensure_package("mercantile")


# ------------------------------------------------------------
# Microsoft settings
# ------------------------------------------------------------

MS_DATASET_LINKS_URLS = [
    "https://minedbuildings.z5.web.core.windows.net/global-buildings/dataset-links.csv",
    "https://minedbuildings.blob.core.windows.net/global-buildings/dataset-links.csv",
]

MS_BUILDING_CACHE_DIR = OUTPUT_DIR / "ms_buildings_cache"
MS_BUILDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

buildings_cache = OUTPUT_DIR / "microsoft_building_footprints_study_area.gpkg"

# Microsoft manifest uses zoom-level quadkeys.
MS_TILE_ZOOM = 9

# Set True only if you want to delete the cached GPKG and rebuild it.
FORCE_REDOWNLOAD_BUILDINGS = True


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def download_text_with_fallback(urls, cache_path=None):
    """
    Download text from one of multiple URLs.
    If cache exists, use cache first.
    """
    if cache_path is not None and Path(cache_path).exists():
        print("Loading cached manifest:")
        print(cache_path)
        return Path(cache_path).read_text(encoding="utf-8")

    last_error = None

    for url in urls:
        print("Trying Microsoft manifest URL:")
        print(url)

        try:
            r = requests.get(
                url,
                timeout=180,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            r.raise_for_status()

            text = r.text

            if cache_path is not None:
                Path(cache_path).write_text(text, encoding="utf-8")
                print("Saved manifest cache:")
                print(cache_path)

            return text

        except Exception as e:
            print(f"FAILED: {e}")
            last_error = e

    raise RuntimeError("Could not download Microsoft dataset-links.csv.") from last_error


def load_ms_manifest(cache_dir=MS_BUILDING_CACHE_DIR):
    """
    Load Microsoft building-footprint manifest.
    Expected columns: QuadKey, Url.
    """
    manifest_cache = cache_dir / "dataset-links.csv"

    text = download_text_with_fallback(
        MS_DATASET_LINKS_URLS,
        cache_path=manifest_cache
    )

    manifest = pd.read_csv(io.StringIO(text))

    qk_col = next((c for c in manifest.columns if c.lower() == "quadkey"), None)
    url_col = next((c for c in manifest.columns if c.lower() == "url"), None)

    if qk_col is None or url_col is None:
        print("Manifest columns:")
        print(manifest.columns.tolist())
        raise RuntimeError("Microsoft manifest must contain QuadKey and Url columns.")

    print("Microsoft manifest loaded.")
    print(f"Rows: {len(manifest):,}")
    print(f"QuadKey column: {qk_col}")
    print(f"Url column: {url_col}")

    return manifest, qk_col, url_col


def parse_ms_gzip_tile(gzip_path):
    """
    Parse one Microsoft Global ML gzip tile into geometry/property records.
    """
    records = []

    with gzip.open(gzip_path, "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.strip()

            if not line:
                continue

            try:
                feat = json.loads(line)
                geom_obj = feat.get("geometry")
                props = feat.get("properties", {})

                if geom_obj is None:
                    continue

                geom = shapely_shape(geom_obj)

                if geom is None or geom.is_empty:
                    continue

                records.append({
                    "geometry": geom,
                    "height": props.get("height", None),
                    "confidence": props.get("confidence", None),
                })

            except Exception:
                continue

    return records


# ------------------------------------------------------------
# Main execution
# ------------------------------------------------------------

if buildings_cache.exists() and FORCE_REDOWNLOAD_BUILDINGS:
    buildings_cache.unlink()
    print("Deleted old buildings cache:")
    print(buildings_cache)

if buildings_cache.exists():
    print("Loading cached Microsoft buildings:")
    print(buildings_cache)
    buildings = gpd.read_file(buildings_cache)

else:
    # Study area = union of the 9 municipal boundaries
    study_area_wgs = muni_boundaries.to_crs(CRS_WGS84).copy()
    study_union_wgs = study_area_wgs.geometry.unary_union

    minx, miny, maxx, maxy = study_area_wgs.total_bounds

    print("Study-area bbox WGS84:")
    print([minx, miny, maxx, maxy])

    manifest, qk_col, url_col = load_ms_manifest(MS_BUILDING_CACHE_DIR)

    quadkeys = sorted({
        mercantile.quadkey(t)
        for t in mercantile.tiles(minx, miny, maxx, maxy, zooms=MS_TILE_ZOOM)
    })

    print(f"\nQuadkeys covering full study area: {len(quadkeys)}")
    print(quadkeys)

    matched = manifest[manifest[qk_col].astype(str).isin(quadkeys)].copy()

    print(f"\nMatched Microsoft tiles: {len(matched):,} / {len(quadkeys):,}")

    if matched.empty:
        raise RuntimeError("No Microsoft tiles matched the study-area quadkeys.")

    records = []

    for _, tile_row in matched.iterrows():
        qk = str(tile_row[qk_col])
        url = str(tile_row[url_col])
        tile_cache = MS_BUILDING_CACHE_DIR / f"{qk}.gz"

        if not tile_cache.exists():
            print(f"\nDownloading tile {qk}...")
            r = requests.get(
                url,
                timeout=300,
                stream=True,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            r.raise_for_status()

            with open(tile_cache, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        else:
            print(f"\nUsing cached tile {qk}")

        tile_records = parse_ms_gzip_tile(tile_cache)
        print(f"Tile {qk}: {len(tile_records):,} raw features")

        records.extend(tile_records)

    if not records:
        raise RuntimeError("No Microsoft building records parsed.")

    all_b = gpd.GeoDataFrame(records, geometry="geometry", crs=CRS_WGS84)

    print(f"\nRaw buildings from matched tiles: {len(all_b):,}")

    # Coarse bbox filter
    all_b = all_b.cx[minx:maxx, miny:maxy].copy()
    print(f"After bbox filter: {len(all_b):,}")

    # Clip to union of all municipal boundaries
    all_b = all_b[all_b.intersects(study_union_wgs)].copy()
    print(f"After municipal union intersects filter: {len(all_b):,}")

    all_b["geometry"] = all_b.geometry.intersection(study_union_wgs)

    all_b = all_b[
        all_b.geometry.notna() &
        (~all_b.geometry.is_empty)
    ].copy()

    all_b = all_b[
        all_b.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    ].copy()

    print(f"After municipal union clipping: {len(all_b):,}")

    # Project
    all_b = all_b.to_crs(CRS_PROJECTED)
    all_b = all_b.explode(index_parts=False).reset_index(drop=True)

    # Municipality assignment by representative point
    b_cent = all_b.copy()
    b_cent["geometry"] = b_cent.geometry.representative_point()

    muni_p = muni_boundaries.to_crs(CRS_PROJECTED).copy()

    if "muni_norm" not in muni_p.columns:
        muni_p["muni_norm"] = muni_p["municipality"].map(norm_name)

    joined = gpd.sjoin(
        b_cent[["geometry"]],
        muni_p[["municipality", "muni_norm", "geometry"]],
        how="left",
        predicate="within"
    )

    all_b["municipality"] = joined["municipality"].values
    all_b["muni_norm"] = joined["muni_norm"].values

    all_b = all_b[all_b["municipality"].notna()].copy()

    all_b["data_source"] = "microsoft"
    all_b["building"] = "yes"
    all_b["building_area_m2"] = all_b.geometry.area.astype(float)
    all_b["weighted_area"] = all_b["building_area_m2"]
    all_b["building_id"] = range(len(all_b))

    buildings = all_b.copy()

    buildings.to_file(buildings_cache, driver="GPKG")

    print("\nSaved buildings cache:")
    print(buildings_cache)


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print("\nBuildings loaded:")
print(f"Total buildings: {len(buildings):,}")
print("CRS:", buildings.crs)

print("\nBuildings per municipality:")
building_count_summary = (
    buildings
    .groupby("municipality")
    .size()
    .sort_values(ascending=False)
    .rename("n_buildings")
)

display(building_count_summary)

print("\nSuspicious municipalities with fewer than 100 buildings:")
suspicious = building_count_summary[building_count_summary < 100]
display(suspicious)

if "Livadia" in building_count_summary.index:
    print(f"\nLivadia buildings: {building_count_summary.loc['Livadia']:,}")

    if building_count_summary.loc["Livadia"] < 1000:
        print("WARNING: Livadia still looks suspicious. Check the boundary.")
    else:
        print("Livadia building count looks plausible.")
else:
    print("WARNING: Livadia missing from assigned building municipalities.")

Study-area bbox WGS84:
[33.5270625, 34.8155398, 33.7135283, 35.0336159]
Trying Microsoft manifest URL:
https://minedbuildings.z5.web.core.windows.net/global-buildings/dataset-links.csv


C:\Users\lucam\AppData\Local\Temp\ipykernel_14652\1664592226.py:188: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  study_union_wgs = study_area_wgs.geometry.unary_union


Saved manifest cache:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\ms_buildings_cache\dataset-links.csv
Microsoft manifest loaded.
Rows: 30,344
QuadKey column: QuadKey
Url column: Url

Quadkeys covering full study area: 2
['122103131', '122103133']

Matched Microsoft tiles: 2 / 2

Tile 122103131: 344,818 raw features

Tile 122103133: 76,485 raw features

Raw buildings from matched tiles: 421,303
After bbox filter: 55,265
After municipal union intersects filter: 49,718
After municipal union clipping: 49,718

Saved buildings cache:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\microsoft_building_footprints_study_area.gpkg

Buildings loaded:
Total buildings: 49,721
CRS: EPSG:32636

Buildings per municipality:


municipality
Larnaca              14067
Aradippou            11747
Dromolaxia-Meneou     5405
Livadia               4339
Oroklini              4080
Pyla                  3432
Kiti                  3010
Perivolia             2960
Kellia                 681
Name: n_buildings, dtype: int64


Suspicious municipalities with fewer than 100 buildings:


Series([], Name: n_buildings, dtype: int64)


Livadia buildings: 4,339
Livadia building count looks plausible.


In [7]:
# ============================================================
# BLOCK 7 -- URBAN FOOTPRINT DELINEATION
# ============================================================
# This block is compatible with the Microsoft building-footprint block.
# It assumes 'buildings' already contains:
# - geometry
# - municipality
# - muni_norm
# - building_area_m2
# - building_id

from shapely.ops import unary_union

# ------------------------------------------------------------
# Input checks
# ------------------------------------------------------------

if "buildings" not in globals():
    raise RuntimeError("Variable 'buildings' does not exist. Run Block 6 first.")

if len(buildings) == 0:
    raise RuntimeError("Variable 'buildings' is empty. Block 6 did not load any buildings.")

print("Initial buildings from Block 6:")
print(f"Rows: {len(buildings):,}")
print("CRS:", buildings.crs)
print("Columns:", buildings.columns.tolist())

# Ensure CRS
if buildings.crs is None:
    print("WARNING: buildings has no CRS. Assuming WGS84.")
    buildings = buildings.set_crs(CRS_WGS84)

# Project to metric CRS
buildings_p = buildings.to_crs(CRS_PROJECTED).copy()

# Keep valid polygon geometries
buildings_p = buildings_p[buildings_p.geometry.notna()].copy()
buildings_p = buildings_p[~buildings_p.geometry.is_empty].copy()
buildings_p = buildings_p[buildings_p.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

# Ensure building_id
if "building_id" not in buildings_p.columns:
    buildings_p["building_id"] = range(len(buildings_p))

# Ensure municipality field
if "municipality" not in buildings_p.columns:
    raise RuntimeError("Column 'municipality' missing from buildings. Block 6 assignment failed.")

# Ensure muni_norm
if "muni_norm" not in buildings_p.columns:
    buildings_p["muni_norm"] = buildings_p["municipality"].map(norm_name)

# Ensure building area
if "building_area_m2" not in buildings_p.columns:
    buildings_p["building_area_m2"] = buildings_p.geometry.area
else:
    buildings_p["building_area_m2"] = buildings_p["building_area_m2"].fillna(buildings_p.geometry.area)

# Remove invalid/tiny geometries
buildings_p = buildings_p[buildings_p["building_area_m2"] > 1.0].copy()

# Project boundaries
muni_boundaries_p = muni_boundaries.to_crs(CRS_PROJECTED).copy()

if "muni_norm" not in muni_boundaries_p.columns:
    muni_boundaries_p["muni_norm"] = muni_boundaries_p["municipality"].map(norm_name)


# ------------------------------------------------------------
# Diagnostics: buildings by municipality
# ------------------------------------------------------------

print("\nBuildings after cleaning:")
print(f"Rows: {len(buildings_p):,}")

buildings_by_muni = (
    buildings_p
    .groupby("municipality")
    .agg(
        n_buildings=("building_id", "count"),
        total_building_area_m2=("building_area_m2", "sum"),
        mean_building_area_m2=("building_area_m2", "mean"),
    )
    .reset_index()
    .sort_values("n_buildings", ascending=False)
)

display(buildings_by_muni)

missing_muni_buildings = sorted(set(ALL_MUNICIPALITIES) - set(buildings_p["municipality"].unique()))

if missing_muni_buildings:
    print("\nWARNING: No buildings found for these municipalities:")
    print(missing_muni_buildings)
else:
    print("\nBuildings found for all requested municipalities.")


# ------------------------------------------------------------
# Create derived urban footprints from building clusters
# ------------------------------------------------------------

urban_parts = []

for muni in ALL_MUNICIPALITIES:
    print("\n" + "=" * 70)
    print(f"Creating urban footprint for {muni}")
    print("=" * 70)

    b = buildings_p[buildings_p["municipality"] == muni].copy()

    if len(b) == 0:
        print(f"WARNING: no buildings for {muni}")
        continue

    # Buffer buildings to connect nearby built-up fabric.
    # This approximates the grey/built-up areas rather than using the full municipality.
    buffered = b.geometry.buffer(URBAN_BUILDING_BUFFER_M)

    dissolved = unary_union(buffered)

    if dissolved.is_empty:
        print(f"WARNING: empty dissolved geometry for {muni}")
        continue

    if dissolved.geom_type == "Polygon":
        cluster_geoms = [dissolved]
    elif dissolved.geom_type == "MultiPolygon":
        cluster_geoms = list(dissolved.geoms)
    else:
        print(f"WARNING: unexpected dissolved geometry type for {muni}: {dissolved.geom_type}")
        continue

    clusters = gpd.GeoDataFrame(
        {"geometry": cluster_geoms},
        geometry="geometry",
        crs=CRS_PROJECTED
    )

    clusters["cluster_id_local"] = range(len(clusters))
    clusters["cluster_area_m2"] = clusters.geometry.area

    # Count buildings per cluster using representative points
    b_pts = b[["building_id", "geometry"]].copy()
    b_pts["geometry"] = b_pts.geometry.representative_point()

    cluster_join = gpd.sjoin(
        b_pts,
        clusters[["cluster_id_local", "geometry"]],
        how="left",
        predicate="within"
    )

    counts = cluster_join.groupby("cluster_id_local")["building_id"].count()

    clusters["n_buildings"] = (
        clusters["cluster_id_local"]
        .map(counts)
        .fillna(0)
        .astype(int)
    )

    # Filter small isolated rural clusters
    clusters_f = clusters[
        (clusters["n_buildings"] >= MIN_CLUSTER_BUILDINGS) &
        (clusters["cluster_area_m2"] >= MIN_CLUSTER_AREA_M2)
    ].copy()

    if len(clusters_f) == 0:
        print("WARNING: all clusters filtered out. Keeping largest cluster as fallback.")
        clusters_f = clusters.sort_values("cluster_area_m2", ascending=False).head(1).copy()

    if KEEP_ONLY_LARGEST_URBAN_CLUSTER and len(clusters_f) > 0:
        clusters_f = clusters_f.sort_values("cluster_area_m2", ascending=False).head(1).copy()

    clusters_f["municipality"] = muni
    clusters_f["muni_norm"] = norm_name(muni)

    urban_parts.append(clusters_f)

    print(f"Original clusters: {len(clusters):,}")
    print(f"Kept clusters: {len(clusters_f):,}")
    print(f"Buildings in kept clusters approx: {clusters_f['n_buildings'].sum():,}")
    print(f"Urban footprint area: {clusters_f['cluster_area_m2'].sum()/1e6:.2f} km²")

if not urban_parts:
    raise RuntimeError("No urban footprints created. Check building data and parameters.")

urban_footprints_p = pd.concat(urban_parts, ignore_index=True)
urban_footprints_p = gpd.GeoDataFrame(
    urban_footprints_p,
    geometry="geometry",
    crs=CRS_PROJECTED
)

urban_footprints_p["urban_cluster_id"] = range(len(urban_footprints_p))

print("\nUrban footprint summary:")
urban_summary = (
    urban_footprints_p
    .groupby("municipality")
    .agg(
        n_clusters=("urban_cluster_id", "count"),
        n_buildings_in_clusters=("n_buildings", "sum"),
        urban_area_km2=("cluster_area_m2", lambda x: x.sum() / 1e6),
    )
    .reset_index()
    .sort_values("municipality")
)

display(urban_summary)


# ------------------------------------------------------------
# Flag buildings inside derived urban footprint
# ------------------------------------------------------------

build_cent = buildings_p[["building_id", "municipality", "geometry"]].copy()
build_cent["geometry"] = buildings_p.geometry.representative_point()

inside_urban = gpd.sjoin(
    build_cent,
    urban_footprints_p[["municipality", "urban_cluster_id", "geometry"]],
    how="left",
    predicate="within"
)

# Geopandas adds left/right suffixes when both dataframes have same column names.
if "municipality_left" in inside_urban.columns and "municipality_right" in inside_urban.columns:
    inside_urban["same_muni"] = inside_urban["municipality_left"] == inside_urban["municipality_right"]
    inside_ids = set(
        inside_urban.loc[
            inside_urban["index_right"].notna() & inside_urban["same_muni"],
            "building_id"
        ]
    )
else:
    inside_ids = set(
        inside_urban.loc[
            inside_urban["index_right"].notna(),
            "building_id"
        ]
    )

buildings_p["inside_urban_footprint"] = buildings_p["building_id"].isin(inside_ids)

print("\nInside/outside urban footprint by municipality:")
inside_outside_summary = (
    buildings_p
    .groupby(["municipality", "inside_urban_footprint"])
    .size()
    .rename("n_buildings")
    .reset_index()
    .sort_values(["municipality", "inside_urban_footprint"])
)

display(inside_outside_summary)

print("\nUrban-footprint inclusion percentages:")
urban_pct = (
    buildings_p
    .groupby("municipality")
    .agg(
        total_buildings=("building_id", "count"),
        buildings_inside_urban=("inside_urban_footprint", "sum"),
    )
    .reset_index()
)

urban_pct["inside_share_percent"] = (
    urban_pct["buildings_inside_urban"] / urban_pct["total_buildings"] * 100
)

display(urban_pct.sort_values("municipality"))


# ------------------------------------------------------------
# Save quick diagnostic files
# ------------------------------------------------------------

urban_diag_path = OUTPUT_DIR / "urban_footprint_diagnostics.csv"
urban_summary.to_csv(urban_diag_path, index=False)

print("\nSaved:")
print(urban_diag_path)

print("\nCreated variables:")
print("buildings_p:", len(buildings_p), buildings_p.crs)
print("urban_footprints_p:", len(urban_footprints_p), urban_footprints_p.crs)

Initial buildings from Block 6:
Rows: 49,721
CRS: EPSG:32636
Columns: ['height', 'confidence', 'geometry', 'municipality', 'muni_norm', 'data_source', 'building', 'building_area_m2', 'weighted_area', 'building_id']

Buildings after cleaning:
Rows: 49,717


,municipality,n_buildings,total_building_area_m2,mean_building_area_m2
4,Larnaca,14067,3.761500e+06,267.398854
0,Aradippou,11746,2.719768e+06,231.548465
1,Dromolaxia-Meneou,5405,8.288273e+05,153.344558
5,Livadia,4339,7.634087e+05,175.941168
6,Oroklini,4080,6.859587e+05,168.127124
8,Pyla,3432,4.902095e+05,142.834936
3,Kiti,3008,5.258374e+05,174.812961
7,Perivolia,2960,4.461686e+05,150.732641
2,Kellia,680,8.910696e+04,131.039648



Buildings found for all requested municipalities.

Creating urban footprint for Larnaca
Original clusters: 23
Kept clusters: 5
Buildings in kept clusters approx: 13,987
Urban footprint area: 17.93 km²

Creating urban footprint for Aradippou
Original clusters: 113
Kept clusters: 10
Buildings in kept clusters approx: 11,295
Urban footprint area: 27.83 km²

Creating urban footprint for Livadia
Original clusters: 13
Kept clusters: 3
Buildings in kept clusters approx: 4,306
Urban footprint area: 6.54 km²

Creating urban footprint for Oroklini
Original clusters: 26
Kept clusters: 4
Buildings in kept clusters approx: 3,995
Urban footprint area: 7.67 km²

Creating urban footprint for Kellia
Original clusters: 41
Kept clusters: 5
Buildings in kept clusters approx: 542
Urban footprint area: 1.50 km²

Creating urban footprint for Dromolaxia-Meneou
Original clusters: 55
Kept clusters: 6
Buildings in kept clusters approx: 5,235
Urban footprint area: 8.78 km²

Creating urban footprint for Perivolia

,municipality,n_clusters,n_buildings_in_clusters,urban_area_km2
0,Aradippou,10,11295,27.826953
1,Dromolaxia-Meneou,6,5235,8.783621
2,Kellia,5,542,1.498881
3,Kiti,2,2975,6.547053
4,Larnaca,5,13987,17.926462
5,Livadia,3,4306,6.543583
6,Oroklini,4,3995,7.665345
7,Perivolia,3,2946,5.889754
8,Pyla,4,3350,7.531355



Inside/outside urban footprint by municipality:


,municipality,inside_urban_footprint,n_buildings
0,Aradippou,False,451
1,Aradippou,True,11295
2,Dromolaxia-Meneou,False,170
3,Dromolaxia-Meneou,True,5235
4,Kellia,False,138
5,Kellia,True,542
6,Kiti,False,33
7,Kiti,True,2975
8,Larnaca,False,80
9,Larnaca,True,13987



Urban-footprint inclusion percentages:


,municipality,total_buildings,buildings_inside_urban,inside_share_percent
0,Aradippou,11746,11295,96.160395
1,Dromolaxia-Meneou,5405,5235,96.854764
2,Kellia,680,542,79.705882
3,Kiti,3008,2975,98.902926
4,Larnaca,14067,13987,99.431293
5,Livadia,4339,4306,99.239456
6,Oroklini,4080,3995,97.916667
7,Perivolia,2960,2946,99.527027
8,Pyla,3432,3350,97.610723



Saved:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\urban_footprint_diagnostics.csv

Created variables:
buildings_p: 49717 EPSG:32636
urban_footprints_p: 42 EPSG:32636


In [8]:
# ============================================================
# BLOCK 8 -- URBAN FOOTPRINT INTERACTIVE CHECK MAP
# ============================================================
# This map uses OpenStreetMap XYZ tiles as background.
# It shows:
# - municipal boundaries;
# - derived urban footprints;
# - buildings inside the urban footprint;
# - buildings outside the urban footprint;
# - municipal labels with building counts.
#
# Use this before population allocation to check whether the urban footprints
# correctly represent the built-up / grey areas.

if not HAS_FOLIUM:
    raise ImportError("folium is not installed. Install with: pip install folium")

if "buildings_p" not in globals():
    raise RuntimeError("buildings_p not found. Run Block 7 first.")

if "urban_footprints_p" not in globals():
    raise RuntimeError("urban_footprints_p not found. Run Block 7 first.")

if "muni_boundaries_p" not in globals():
    raise RuntimeError("muni_boundaries_p not found. Run Block 4 first.")


# ------------------------------------------------------------
# Convert layers to WGS84 for Folium
# ------------------------------------------------------------

muni_w = muni_boundaries_p.to_crs(CRS_WGS84)
urban_w = urban_footprints_p.to_crs(CRS_WGS84)
build_w = buildings_p.to_crs(CRS_WGS84)

center = muni_w.geometry.unary_union.centroid


# ------------------------------------------------------------
# Create map with OpenStreetMap XYZ tiles
# ------------------------------------------------------------

m_urban_osm = folium.Map(
    location=[center.y, center.x],
    zoom_start=12,
    tiles=None
)

folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap",
    overlay=False,
    control=True
).add_to(m_urban_osm)

folium.TileLayer(
    tiles="CartoDB positron",
    name="CartoDB Positron",
    overlay=False,
    control=True
).add_to(m_urban_osm)

Fullscreen().add_to(m_urban_osm)
MiniMap(toggle_display=True).add_to(m_urban_osm)


# ------------------------------------------------------------
# Municipal boundaries
# ------------------------------------------------------------

fg_bound = folium.FeatureGroup(name="Municipal boundaries", show=True)

for _, r in muni_w.iterrows():
    is_north = r["municipality"] in NORTH_MUNICIPALITIES
    color = "#0076C2" if is_north else "#777777"

    folium.GeoJson(
        r.geometry,
        tooltip=f"{r['municipality']} boundary",
        style_function=lambda x, color=color: {
            "fillColor": "#ffffff",
            "color": color,
            "weight": 2,
            "fillOpacity": 0.00,
        }
    ).add_to(fg_bound)

fg_bound.add_to(m_urban_osm)


# ------------------------------------------------------------
# Derived urban footprints
# ------------------------------------------------------------

fg_urban = folium.FeatureGroup(name="Derived urban footprints", show=True)

for _, r in urban_w.iterrows():
    is_north = r["municipality"] in NORTH_MUNICIPALITIES
    fill = "#0076C2" if is_north else "#B0B0B0"
    line = "#003B70" if is_north else "#555555"

    popup = (
        f"<b>{r['municipality']}</b><br>"
        f"Urban cluster ID: {r['urban_cluster_id']}<br>"
        f"Buildings in cluster: {int(r['n_buildings']):,}<br>"
        f"Cluster area: {r['cluster_area_m2'] / 1e6:.3f} km²"
    )

    folium.GeoJson(
        r.geometry,
        tooltip=f"{r['municipality']} urban footprint",
        popup=popup,
        style_function=lambda x, fill=fill, line=line: {
            "fillColor": fill,
            "color": line,
            "weight": 2,
            "fillOpacity": 0.25,
        }
    ).add_to(fg_urban)

fg_urban.add_to(m_urban_osm)


# ------------------------------------------------------------
# Buildings inside urban footprint
# ------------------------------------------------------------

fg_build_in = folium.FeatureGroup(name="Buildings inside urban footprint", show=False)

inside_buildings = build_w[build_w["inside_urban_footprint"]].copy()

for _, r in inside_buildings.iterrows():
    pt = r.geometry.representative_point()

    popup = (
        f"<b>Building inside urban footprint</b><br>"
        f"Building ID: {r['building_id']}<br>"
        f"Municipality: {r['municipality']}<br>"
        f"Area: {r['building_area_m2']:.1f} m²"
    )

    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=1.2,
        color="#0076C2",
        fill=True,
        fill_color="#0076C2",
        fill_opacity=0.65,
        weight=0.4,
        popup=popup
    ).add_to(fg_build_in)

fg_build_in.add_to(m_urban_osm)


# ------------------------------------------------------------
# Buildings outside urban footprint
# ------------------------------------------------------------

fg_build_out = folium.FeatureGroup(name="Buildings outside urban footprint", show=False)

outside_buildings = build_w[~build_w["inside_urban_footprint"]].copy()

for _, r in outside_buildings.iterrows():
    pt = r.geometry.representative_point()

    popup = (
        f"<b>Building outside urban footprint</b><br>"
        f"Building ID: {r['building_id']}<br>"
        f"Municipality: {r['municipality']}<br>"
        f"Area: {r['building_area_m2']:.1f} m²"
    )

    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=1.0,
        color="#999999",
        fill=True,
        fill_color="#999999",
        fill_opacity=0.45,
        weight=0.3,
        popup=popup
    ).add_to(fg_build_out)

fg_build_out.add_to(m_urban_osm)


# ------------------------------------------------------------
# Building density / summary labels
# ------------------------------------------------------------

fg_labels = folium.FeatureGroup(name="Municipality summary labels", show=True)

urban_pct_map = (
    buildings_p
    .groupby("municipality")
    .agg(
        total_buildings=("building_id", "count"),
        buildings_inside_urban=("inside_urban_footprint", "sum"),
    )
    .reset_index()
)

urban_pct_map["inside_share_percent"] = (
    urban_pct_map["buildings_inside_urban"] /
    urban_pct_map["total_buildings"] * 100
)

for _, r in muni_w.iterrows():
    muni = r["municipality"]
    centroid = r.geometry.centroid

    row = urban_pct_map[urban_pct_map["municipality"] == muni]

    if len(row) == 0:
        label_text = muni
        popup = f"{muni}<br>No buildings found."
    else:
        row = row.iloc[0]

        label_text = (
            f"{muni}<br>"
            f"{int(row['buildings_inside_urban']):,}/"
            f"{int(row['total_buildings']):,}"
        )

        popup = (
            f"<b>{muni}</b><br>"
            f"Total buildings: {int(row['total_buildings']):,}<br>"
            f"Buildings inside urban footprint: {int(row['buildings_inside_urban']):,}<br>"
            f"Inside share: {row['inside_share_percent']:.1f}%"
        )

    folium.Marker(
        location=[centroid.y, centroid.x],
        popup=popup,
        tooltip=muni,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 9pt;
                font-weight: bold;
                color: #003B70;
                background-color: rgba(255,255,255,0.85);
                border: 1px solid #003B70;
                border-radius: 4px;
                padding: 3px 5px;
                text-align: center;
                white-space: nowrap;">
                {label_text}
            </div>
            """
        )
    ).add_to(fg_labels)

fg_labels.add_to(m_urban_osm)


# ------------------------------------------------------------
# Layer control and save
# ------------------------------------------------------------

folium.LayerControl(collapsed=False).add_to(m_urban_osm)

out_html = OUTPUT_DIR / "urban_footprint_buildings_check_map.html"
m_urban_osm.save(out_html)

print("Saved OSM urban footprint check map:")
print(out_html)

C:\Users\lucam\AppData\Local\Temp\ipykernel_14652\2433960483.py:36: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = muni_w.geometry.unary_union.centroid


Saved OSM urban footprint check map:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\urban_footprint_buildings_check_map.html


In [9]:
# ============================================================
# BLOCK 9 -- POPULATION AND WASTEWATER DEMAND ALLOCATION
# ============================================================
# This block:
# - assigns municipal population to ALL buildings inside each administrative boundary;
# - keeps the inside/outside urban-footprint flag for diagnostics;
# - estimates wastewater demand for every building;
# - assigns each municipality to WWTP_1_SOUTH or WWTP_2_NORTH.
#
# IMPORTANT:
# Urban footprint is NOT used here to exclude buildings from population allocation.
# It is only used as a classification variable.
# Network inclusion can be decided later in Block 11.

# ------------------------------------------------------------
# Input checks
# ------------------------------------------------------------

required_vars = [
    "buildings_p",
    "MUNICIPALITY_POPULATION",
    "WWTP_ASSIGNMENT",
]

for v in required_vars:
    if v not in globals():
        raise RuntimeError(f"Missing variable '{v}'. Run previous blocks first.")

if len(buildings_p) == 0:
    raise RuntimeError("buildings_p is empty. Cannot allocate population.")

print("Buildings available for demand allocation:")
print(f"Rows: {len(buildings_p):,}")
print("CRS:", buildings_p.crs)

missing_pop = sorted(set(ALL_MUNICIPALITIES) - set(MUNICIPALITY_POPULATION.keys()))
if missing_pop:
    print("WARNING: Missing population values for:")
    print(missing_pop)

missing_wwtp = sorted(set(ALL_MUNICIPALITIES) - set(WWTP_ASSIGNMENT.keys()))
if missing_wwtp:
    print("WARNING: Missing WWTP assignment for:")
    print(missing_wwtp)

if "inside_urban_footprint" not in buildings_p.columns:
    print("WARNING: 'inside_urban_footprint' not found. Setting all buildings as inside.")
    buildings_p["inside_urban_footprint"] = True


# ------------------------------------------------------------
# Reset demand fields
# ------------------------------------------------------------

buildings_p = buildings_p.copy()

buildings_p["assigned_population"] = 0.0
buildings_p["wastewater_l_day"] = 0.0
buildings_p["wastewater_m3_day"] = 0.0

if "building_area_m2" not in buildings_p.columns:
    buildings_p["building_area_m2"] = buildings_p.geometry.area
else:
    buildings_p["building_area_m2"] = buildings_p["building_area_m2"].fillna(buildings_p.geometry.area)

buildings_p["building_area_m2"] = buildings_p["building_area_m2"].clip(lower=0.0)


# ------------------------------------------------------------
# Allocate population to ALL buildings in each municipality
# ------------------------------------------------------------

allocation_rows = []

for muni in ALL_MUNICIPALITIES:
    pop = float(MUNICIPALITY_POPULATION.get(muni, 0.0))

    mask_muni = buildings_p["municipality"] == muni

    # IMPORTANT:
    # Population is assigned to all buildings in the administrative boundary,
    # not only to buildings inside the urban footprint.
    mask_demand = mask_muni

    b = buildings_p.loc[mask_demand].copy()

    total_buildings_muni = int(mask_muni.sum())
    buildings_inside_urban = int((mask_muni & buildings_p["inside_urban_footprint"]).sum())
    buildings_outside_urban = int((mask_muni & ~buildings_p["inside_urban_footprint"]).sum())

    if len(b) == 0:
        print(f"WARNING: no buildings for {muni}. Population not assigned.")

        allocation_rows.append({
            "municipality": muni,
            "population_input": pop,
            "total_buildings": total_buildings_muni,
            "buildings_inside_urban": buildings_inside_urban,
            "buildings_outside_urban": buildings_outside_urban,
            "total_weighted_area_m2": 0.0,
            "assigned_population": 0.0,
            "allocation_status": "no_buildings",
        })

        continue

    area_sum = float(b["building_area_m2"].sum())

    if area_sum > 0:
        weights = buildings_p.loc[mask_demand, "building_area_m2"] / area_sum
    else:
        weights = pd.Series(
            1.0 / len(b),
            index=buildings_p.loc[mask_demand].index
        )

    buildings_p.loc[mask_demand, "assigned_population"] = weights * pop

    assigned_pop = float(buildings_p.loc[mask_demand, "assigned_population"].sum())

    allocation_rows.append({
        "municipality": muni,
        "population_input": pop,
        "total_buildings": total_buildings_muni,
        "buildings_inside_urban": buildings_inside_urban,
        "buildings_outside_urban": buildings_outside_urban,
        "total_weighted_area_m2": area_sum,
        "assigned_population": assigned_pop,
        "allocation_status": "ok",
    })

    print(
        f"{muni:20s} | pop input: {pop:10.0f} | "
        f"buildings: {total_buildings_muni:6d} | "
        f"inside urban: {buildings_inside_urban:6d} | "
        f"outside urban: {buildings_outside_urban:6d} | "
        f"assigned: {assigned_pop:10.1f}"
    )

allocation_df = pd.DataFrame(allocation_rows)


# ------------------------------------------------------------
# Estimate wastewater demand
# ------------------------------------------------------------

buildings_p["wastewater_l_day"] = (
    buildings_p["assigned_population"] * float(WASTEWATER_L_PER_CAPITA_DAY)
)

buildings_p["wastewater_m3_day"] = buildings_p["wastewater_l_day"] / 1000.0

buildings_p["wwtp_assignment"] = buildings_p["municipality"].map(WWTP_ASSIGNMENT)
buildings_p["wwtp_assignment"] = buildings_p["wwtp_assignment"].fillna("UNASSIGNED")


# ------------------------------------------------------------
# Municipality summary
# ------------------------------------------------------------

summary = (
    buildings_p
    .groupby("municipality")
    .agg(
        n_buildings_total=("building_id", "count"),
        n_buildings_inside_urban=("inside_urban_footprint", "sum"),
        assigned_population=("assigned_population", "sum"),
        wastewater_m3_day=("wastewater_m3_day", "sum"),
        total_building_area_m2=("building_area_m2", "sum"),
    )
    .reset_index()
)

summary["n_buildings_outside_urban"] = (
    summary["n_buildings_total"] - summary["n_buildings_inside_urban"]
)

summary = summary.merge(
    allocation_df[
        [
            "municipality",
            "population_input",
            "total_weighted_area_m2",
            "allocation_status",
        ]
    ],
    on="municipality",
    how="left"
)

summary["wwtp_assignment"] = summary["municipality"].map(WWTP_ASSIGNMENT).fillna("UNASSIGNED")
summary["population_error"] = summary["assigned_population"] - summary["population_input"]

summary["inside_urban_building_share_percent"] = (
    summary["n_buildings_inside_urban"] / summary["n_buildings_total"] * 100
)

print("\n=== Municipality population and wastewater summary ===")
display(summary.sort_values("municipality"))


# ------------------------------------------------------------
# Inside/outside urban demand summary
# ------------------------------------------------------------

urban_demand_summary = (
    buildings_p
    .groupby(["municipality", "inside_urban_footprint"])
    .agg(
        n_buildings=("building_id", "count"),
        assigned_population=("assigned_population", "sum"),
        wastewater_m3_day=("wastewater_m3_day", "sum"),
        building_area_m2=("building_area_m2", "sum"),
    )
    .reset_index()
)

urban_demand_summary["zone_class"] = urban_demand_summary["inside_urban_footprint"].map(
    {True: "inside_urban_footprint", False: "outside_urban_footprint"}
)

print("\n=== Inside/outside urban demand summary ===")
display(
    urban_demand_summary[
        [
            "municipality",
            "zone_class",
            "n_buildings",
            "assigned_population",
            "wastewater_m3_day",
            "building_area_m2",
        ]
    ].sort_values(["municipality", "zone_class"])
)


# ------------------------------------------------------------
# WWTP summary
# ------------------------------------------------------------

wwtp_load_initial = (
    summary
    .groupby("wwtp_assignment")
    .agg(
        municipalities=("municipality", lambda x: ", ".join(sorted(x))),
        assigned_population=("assigned_population", "sum"),
        wastewater_m3_day=("wastewater_m3_day", "sum"),
        n_buildings_total=("n_buildings_total", "sum"),
        n_buildings_inside_urban=("n_buildings_inside_urban", "sum"),
        n_buildings_outside_urban=("n_buildings_outside_urban", "sum"),
    )
    .reset_index()
)

total_q = float(wwtp_load_initial["wastewater_m3_day"].sum())

if total_q > 0:
    wwtp_load_initial["wastewater_share_percent"] = (
        wwtp_load_initial["wastewater_m3_day"] / total_q * 100
    )
else:
    wwtp_load_initial["wastewater_share_percent"] = 0.0

print("\n=== Initial WWTP load assignment ===")
display(wwtp_load_initial)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

summary_path = OUTPUT_DIR / "population_wastewater_summary_by_municipality.csv"
wwtp_path = OUTPUT_DIR / "initial_wwtp_load_summary_by_treatment_area.csv"
allocation_path = OUTPUT_DIR / "population_allocation_diagnostics.csv"
urban_demand_path = OUTPUT_DIR / "urban_vs_nonurban_demand_summary.csv"

summary.to_csv(summary_path, index=False)
wwtp_load_initial.to_csv(wwtp_path, index=False)
allocation_df.to_csv(allocation_path, index=False)
urban_demand_summary.to_csv(urban_demand_path, index=False)

print("\nSaved:")
print(summary_path)
print(wwtp_path)
print(allocation_path)
print(urban_demand_path)


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print("\n=== Diagnostics ===")
print(f"Total assigned population: {buildings_p['assigned_population'].sum():,.1f}")
print(f"Total wastewater demand: {buildings_p['wastewater_m3_day'].sum():,.1f} m³/day")
print(f"Assumed wastewater generation: {WASTEWATER_L_PER_CAPITA_DAY:.1f} L/cap/day")

max_abs_error = summary["population_error"].abs().max()
print(f"Maximum municipality population assignment error: {max_abs_error:.6f}")

if max_abs_error > 1e-6:
    print("WARNING: population assignment does not exactly match input for at least one municipality.")

print("\nNOTE:")
print("Population has been assigned to ALL buildings inside each administrative boundary.")
print("The urban-footprint flag is preserved only as a classification variable.")
print("Later blocks can still choose whether the network connects all buildings or only urban-footprint buildings.")

Buildings available for demand allocation:
Rows: 49,717
CRS: EPSG:32636
Larnaca              | pop input:      52046 | buildings:  14067 | inside urban:  13987 | outside urban:     80 | assigned:    52046.0
Aradippou            | pop input:      22932 | buildings:  11746 | inside urban:  11295 | outside urban:    451 | assigned:    22932.0
Livadia              | pop input:       8581 | buildings:   4339 | inside urban:   4306 | outside urban:     33 | assigned:     8581.0
Oroklini             | pop input:       7575 | buildings:   4080 | inside urban:   3995 | outside urban:     85 | assigned:     7575.0
Kellia               | pop input:        387 | buildings:    680 | inside urban:    542 | outside urban:    138 | assigned:      387.0
Dromolaxia-Meneou    | pop input:       6838 | buildings:   5405 | inside urban:   5235 | outside urban:    170 | assigned:     6838.0
Perivolia            | pop input:       3920 | buildings:   2960 | inside urban:   2946 | outside urban:     14 | assi

,municipality,n_buildings_total,n_buildings_inside_urban,assigned_population,wastewater_m3_day,total_building_area_m2,n_buildings_outside_urban,population_input,total_weighted_area_m2,allocation_status,wwtp_assignment,population_error,inside_urban_building_share_percent
0,Aradippou,11746,11295,22932.0,2751.84,2.719768e+06,451,22932.0,2.719768e+06,ok,WWTP_2_NORTH,0.000000e+00,96.160395
1,Dromolaxia-Meneou,5405,5235,6838.0,820.56,8.288273e+05,170,6838.0,8.288273e+05,ok,WWTP_1_SOUTH,9.094947e-13,96.854764
2,Kellia,680,542,387.0,46.44,8.910696e+04,138,387.0,8.910696e+04,ok,WWTP_2_NORTH,5.684342e-14,79.705882
3,Kiti,3008,2975,5136.0,616.32,5.258374e+05,33,5136.0,5.258374e+05,ok,WWTP_1_SOUTH,0.000000e+00,98.902926
4,Larnaca,14067,13987,52046.0,6245.52,3.761500e+06,80,52046.0,3.761500e+06,ok,WWTP_1_SOUTH,0.000000e+00,99.431293
5,Livadia,4339,4306,8581.0,1029.72,7.634087e+05,33,8581.0,7.634087e+05,ok,WWTP_2_NORTH,0.000000e+00,99.239456
6,Oroklini,4080,3995,7575.0,909.00,6.859587e+05,85,7575.0,6.859587e+05,ok,WWTP_2_NORTH,0.000000e+00,97.916667
7,Perivolia,2960,2946,3920.0,470.40,4.461686e+05,14,3920.0,4.461686e+05,ok,WWTP_1_SOUTH,0.000000e+00,99.527027
8,Pyla,3432,3350,2845.0,341.40,4.902095e+05,82,2845.0,4.902095e+05,ok,WWTP_2_NORTH,-4.547474e-13,97.610723



=== Inside/outside urban demand summary ===


,municipality,zone_class,n_buildings,assigned_population,wastewater_m3_day,building_area_m2
1,Aradippou,inside_urban_footprint,11295,22472.895477,2696.747457,2.665318e+06
0,Aradippou,outside_urban_footprint,451,459.104523,55.092543,5.445046e+04
3,Dromolaxia-Meneou,inside_urban_footprint,5235,6654.437239,798.532469,8.065779e+05
2,Dromolaxia-Meneou,outside_urban_footprint,170,183.562761,22.027531,2.224946e+04
5,Kellia,inside_urban_footprint,542,345.424468,41.450936,7.953417e+04
4,Kellia,outside_urban_footprint,138,41.575532,4.989064,9.572789e+03
7,Kiti,inside_urban_footprint,2975,5099.405477,611.928657,5.220907e+05
6,Kiti,outside_urban_footprint,33,36.594523,4.391343,3.746645e+03
9,Larnaca,inside_urban_footprint,13987,51874.345312,6224.921437,3.749094e+06
8,Larnaca,outside_urban_footprint,80,171.654688,20.598563,1.240593e+04



=== Initial WWTP load assignment ===


,wwtp_assignment,municipalities,assigned_population,wastewater_m3_day,n_buildings_total,n_buildings_inside_urban,n_buildings_outside_urban,wastewater_share_percent
0,WWTP_1_SOUTH,"Dromolaxia-Meneou, Kiti, Larnaca, Perivolia",67940.0,8152.8,25440,25143,297,61.617994
1,WWTP_2_NORTH,"Aradippou, Kellia, Livadia, Oroklini, Pyla",42320.0,5078.4,24277,23488,789,38.382006



Saved:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\population_wastewater_summary_by_municipality.csv
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\initial_wwtp_load_summary_by_treatment_area.csv
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\population_allocation_diagnostics.csv
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\urban_vs_nonurban_demand_summary.csv

=== Diagnostics ===
Total assigned population: 110,260.0
Total wastewater demand: 13,231.2 m³/day
Assumed wastewater generation: 120.0 L/cap/day
Maximum municipality population assignment error: 0.000000

NOTE:
Population has been assigned to ALL buildings inside each administrative boundary.
The urban-footprint flag is preserved only as a classification variable.
Later blocks can still choose whether the network connects all buildings or only urban-footprint buildings.


In [10]:
# ============================================================
# BLOCK 10 -- ROAD GRAPH CONSTRUCTION
# ============================================================
# This block builds a NetworkX road graph from the local road shapefile.
# The graph is clipped around the municipal/urban study area to reduce size.

if "roads_p" not in globals():
    raise RuntimeError("roads_p not found. Run Block 3 first.")

if "muni_boundaries_p" not in globals():
    raise RuntimeError("muni_boundaries_p not found. Run Block 4 first.")

if "urban_footprints_p" not in globals():
    raise RuntimeError("urban_footprints_p not found. Run Block 7 first.")

print("Roads before clipping:")
print(f"Rows: {len(roads_p):,}")
print("CRS:", roads_p.crs)


# ------------------------------------------------------------
# Clip roads around both urban footprints and municipal boundaries
# ------------------------------------------------------------
# Since population is now assigned to all buildings, roads must cover also
# buildings outside the derived urban footprints. Therefore we use the union
# of the municipal boundaries, not only the urban footprints.

USE_MUNICIPAL_BOUNDARY_FOR_ROAD_CLIP = True

if USE_MUNICIPAL_BOUNDARY_FOR_ROAD_CLIP:
    road_clip_area = muni_boundaries_p.geometry.unary_union.buffer(ROAD_BUFFER_AROUND_URBAN_M)
else:
    road_clip_area = urban_footprints_p.geometry.unary_union.buffer(ROAD_BUFFER_AROUND_URBAN_M)

roads_clip = roads_p[roads_p.intersects(road_clip_area)].copy()

print("\nRoads after clipping:")
print(f"Rows: {len(roads_clip):,} / {len(roads_p):,}")

if len(roads_clip) == 0:
    raise RuntimeError("No roads after clipping. Increase buffer or check CRS.")


# ------------------------------------------------------------
# Convert road geometries into graph nodes and edges
# ------------------------------------------------------------

node_coord_to_id = {}
node_records = []
edge_records = []


def get_node_id(coord):
    """
    Create a unique node ID from rounded projected coordinates.
    Rounding avoids duplicate nodes caused by tiny coordinate differences.
    """
    key = (round(coord[0], 3), round(coord[1], 3))

    if key not in node_coord_to_id:
        node_id = len(node_coord_to_id)
        node_coord_to_id[key] = node_id

        node_records.append({
            "node_id": node_id,
            "x": key[0],
            "y": key[1],
            "geometry": Point(key)
        })

    return node_coord_to_id[key]


edge_id = 0

for idx, row in roads_clip.iterrows():
    segments = explode_lines_to_segments(row.geometry)

    for seg in segments:
        coords = list(seg.coords)

        if len(coords) < 2:
            continue

        u = get_node_id(coords[0])
        v = get_node_id(coords[1])
        length_m = float(seg.length)

        if length_m <= 0:
            continue

        edge_records.append({
            "edge_id": edge_id,
            "u": int(u),
            "v": int(v),
            "length_m": length_m,
            "geometry": seg
        })

        edge_id += 1


road_nodes_p = gpd.GeoDataFrame(node_records, geometry="geometry", crs=CRS_PROJECTED)
road_edges_p = gpd.GeoDataFrame(edge_records, geometry="geometry", crs=CRS_PROJECTED)

print("\nRoad graph raw layers:")
print(f"Road nodes: {len(road_nodes_p):,}")
print(f"Road edges: {len(road_edges_p):,}")


# ------------------------------------------------------------
# Build NetworkX graph
# ------------------------------------------------------------

G = nx.Graph()

for _, r in road_nodes_p.iterrows():
    G.add_node(
        int(r["node_id"]),
        x=float(r["x"]),
        y=float(r["y"])
    )

for _, r in road_edges_p.iterrows():
    u = int(r["u"])
    v = int(r["v"])
    length = float(r["length_m"])
    eid = int(r["edge_id"])

    if G.has_edge(u, v):
        if length < G[u][v]["length_m"]:
            G[u][v]["length_m"] = length
            G[u][v]["edge_id"] = eid
    else:
        G.add_edge(u, v, length_m=length, edge_id=eid)


print("\nNetworkX road graph:")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


# ------------------------------------------------------------
# Connected components
# ------------------------------------------------------------

components = list(nx.connected_components(G))

node_component = {}
for cid, comp in enumerate(components):
    for n in comp:
        node_component[n] = cid

road_nodes_p["component"] = road_nodes_p["node_id"].map(node_component)
road_edges_p["component"] = road_edges_p["u"].map(node_component)

print("\nLargest road components by number of nodes:")
display(
    road_nodes_p["component"]
    .value_counts()
    .head(10)
    .rename_axis("component")
    .reset_index(name="n_nodes")
)

Roads before clipping:
Rows: 193,874
CRS: EPSG:32636


C:\Users\lucam\AppData\Local\Temp\ipykernel_14652\261093868.py:31: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  road_clip_area = muni_boundaries_p.geometry.unary_union.buffer(ROAD_BUFFER_AROUND_URBAN_M)



Roads after clipping:
Rows: 14,774 / 193,874

Road graph raw layers:
Road nodes: 66,849
Road edges: 74,012

NetworkX road graph:
Nodes: 66849
Edges: 73996

Largest road components by number of nodes:


,component,n_nodes
0,0,66613
1,10,72
2,3,64
3,2,23
4,8,13
5,6,6
6,12,6
7,15,5
8,16,4
9,21,3


In [11]:
# ============================================================
# BLOCK 11 -- SNAP DEMAND TO ROAD NODES
# ============================================================
# This block aggregates building-level demand onto road graph nodes.
#
# If CONNECT_ONLY_URBAN_FOOTPRINT_BUILDINGS = False:
#   all buildings with assigned population are connected.
#
# If CONNECT_ONLY_URBAN_FOOTPRINT_BUILDINGS = True:
#   only buildings inside the derived urban footprint are connected.

if "buildings_p" not in globals():
    raise RuntimeError("buildings_p not found. Run Block 9 first.")

if "road_nodes_p" not in globals():
    raise RuntimeError("road_nodes_p not found. Run Block 10 first.")

if "assigned_population" not in buildings_p.columns:
    raise RuntimeError("assigned_population missing from buildings_p. Run Block 9 first.")


# ------------------------------------------------------------
# Select buildings to connect
# ------------------------------------------------------------

CONNECT_ONLY_URBAN_FOOTPRINT_BUILDINGS = False

if CONNECT_ONLY_URBAN_FOOTPRINT_BUILDINGS:
    demand_buildings_p = buildings_p[
        buildings_p["inside_urban_footprint"] &
        (buildings_p["assigned_population"] > 0)
    ].copy()

    print("Connection mode: only buildings inside urban footprint.")

else:
    demand_buildings_p = buildings_p[
        buildings_p["assigned_population"] > 0
    ].copy()

    print("Connection mode: all buildings with assigned population.")

print("Demand buildings selected:")
print(f"Rows: {len(demand_buildings_p):,}")

if len(demand_buildings_p) == 0:
    raise RuntimeError("No demand buildings selected.")


# ------------------------------------------------------------
# Use representative points for snapping
# ------------------------------------------------------------

demand_points_p = demand_buildings_p.copy()
demand_points_p["geometry"] = demand_points_p.geometry.representative_point()

snap = nearest_node_id(
    points_gdf=demand_points_p,
    nodes_gdf=road_nodes_p,
    point_id_col="building_id"
)

demand_buildings_p = demand_buildings_p.merge(
    snap,
    on="building_id",
    how="left"
)

print("\nSnap distance statistics [m]:")
display(demand_buildings_p["snap_dist_m"].describe())

print("\nWorst 20 snapping distances:")
display(
    demand_buildings_p[
        [
            "building_id",
            "municipality",
            "inside_urban_footprint",
            "assigned_population",
            "snap_dist_m",
        ]
    ]
    .sort_values("snap_dist_m", ascending=False)
    .head(20)
)


# ------------------------------------------------------------
# Aggregate demand at road-node level
# ------------------------------------------------------------

demand_nodes_p = (
    demand_buildings_p
    .groupby(["municipality", "muni_norm", "wwtp_assignment", "node_id"], dropna=False)
    .agg(
        n_buildings=("building_id", "count"),
        n_buildings_inside_urban=("inside_urban_footprint", "sum"),
        assigned_population=("assigned_population", "sum"),
        wastewater_m3_day=("wastewater_m3_day", "sum"),
        total_building_area_m2=("building_area_m2", "sum"),
        mean_snap_dist_m=("snap_dist_m", "mean"),
        max_snap_dist_m=("snap_dist_m", "max"),
    )
    .reset_index()
)

demand_nodes_p["n_buildings_outside_urban"] = (
    demand_nodes_p["n_buildings"] - demand_nodes_p["n_buildings_inside_urban"]
)

demand_nodes_p = demand_nodes_p.merge(
    road_nodes_p[["node_id", "component", "geometry"]],
    on="node_id",
    how="left"
)

demand_nodes_p = gpd.GeoDataFrame(
    demand_nodes_p,
    geometry="geometry",
    crs=CRS_PROJECTED
)

print("\nDemand road nodes per municipality:")
display(
    demand_nodes_p
    .groupby("municipality")
    .agg(
        n_demand_nodes=("node_id", "count"),
        n_buildings=("n_buildings", "sum"),
        n_buildings_inside_urban=("n_buildings_inside_urban", "sum"),
        n_buildings_outside_urban=("n_buildings_outside_urban", "sum"),
        population=("assigned_population", "sum"),
        wastewater_m3_day=("wastewater_m3_day", "sum"),
        mean_snap_dist_m=("mean_snap_dist_m", "mean"),
        max_snap_dist_m=("max_snap_dist_m", "max"),
    )
    .reset_index()
    .sort_values("municipality")
)

print("\nDemand road nodes by WWTP:")
display(
    demand_nodes_p
    .groupby("wwtp_assignment")
    .agg(
        n_demand_nodes=("node_id", "count"),
        n_buildings=("n_buildings", "sum"),
        population=("assigned_population", "sum"),
        wastewater_m3_day=("wastewater_m3_day", "sum"),
    )
    .reset_index()
)

Connection mode: all buildings with assigned population.
Demand buildings selected:
Rows: 49,717

Snap distance statistics [m]:


count    49717.000000
mean        28.771582
std         20.819663
min          0.393060
25%         17.583538
50%         23.880628
75%         33.336640
max        382.892659
Name: snap_dist_m, dtype: float64


Worst 20 snapping distances:


,building_id,municipality,inside_urban_footprint,assigned_population,snap_dist_m
28119,28121,Kellia,False,0.163083,382.892659
34450,34452,Aradippou,False,1.193715,345.756073
16882,16884,Aradippou,False,0.078283,332.350134
9076,9076,Aradippou,False,0.300291,319.966511
36427,36429,Dromolaxia-Meneou,False,0.181167,317.847689
37947,37949,Dromolaxia-Meneou,False,0.302559,310.573968
20561,20563,Aradippou,False,0.168704,310.545043
41401,41403,Dromolaxia-Meneou,False,0.242461,302.393573
29775,29777,Aradippou,True,0.273804,301.245543
20757,20759,Aradippou,True,0.390521,295.158755



Demand road nodes per municipality:


,municipality,n_demand_nodes,n_buildings,n_buildings_inside_urban,n_buildings_outside_urban,population,wastewater_m3_day,mean_snap_dist_m,max_snap_dist_m
0,Aradippou,4974,11746,11295,451,22932.0,2751.84,31.680975,345.756073
1,Dromolaxia-Meneou,1671,5405,5235,170,6838.0,820.56,29.352012,317.847689
2,Kellia,271,680,542,138,387.0,46.44,40.134887,382.892659
3,Kiti,1142,3008,2975,33,5136.0,616.32,30.261306,230.422956
4,Larnaca,6255,14067,13987,80,52046.0,6245.52,22.637410,170.327207
5,Livadia,1831,4339,4306,33,8581.0,1029.72,24.150860,190.977922
6,Oroklini,1829,4080,3995,85,7575.0,909.00,25.058411,180.811019
7,Perivolia,1053,2960,2946,14,3920.0,470.40,26.737608,169.149886
8,Pyla,1438,3432,3350,82,2845.0,341.40,25.807938,213.683946



Demand road nodes by WWTP:


,wwtp_assignment,n_demand_nodes,n_buildings,population,wastewater_m3_day
0,WWTP_1_SOUTH,10121,25440,67940.0,8152.8
1,WWTP_2_NORTH,10343,24277,42320.0,5078.4


In [12]:
# ============================================================
# BLOCK 12 -- DEMAND AND ROAD NODE CHECK MAP
# ============================================================

if not HAS_FOLIUM:
    raise ImportError("folium is not installed. Install with: pip install folium")

if "demand_nodes_p" not in globals():
    raise RuntimeError("demand_nodes_p not found. Run Block 11 first.")

if "demand_buildings_p" not in globals():
    raise RuntimeError("demand_buildings_p not found. Run Block 11 first.")

muni_w = muni_boundaries_p.to_crs(CRS_WGS84)
urban_w = urban_footprints_p.to_crs(CRS_WGS84)
demand_build_w = demand_buildings_p.to_crs(CRS_WGS84)
demand_nodes_w = demand_nodes_p.to_crs(CRS_WGS84)

center = muni_w.geometry.unary_union.centroid

m_demand = folium.Map(
    location=[center.y, center.x],
    zoom_start=11,
    tiles="OpenStreetMap"
)

Fullscreen().add_to(m_demand)
MiniMap(toggle_display=True).add_to(m_demand)


# ------------------------------------------------------------
# Boundaries
# ------------------------------------------------------------

fg_bound = folium.FeatureGroup(name="Municipal boundaries", show=True)

for _, r in muni_w.iterrows():
    is_north = r["municipality"] in NORTH_MUNICIPALITIES
    color = "#0076C2" if is_north else "#777777"

    folium.GeoJson(
        r.geometry,
        tooltip=r["municipality"],
        style_function=lambda x, color=color: {
            "fillColor": "#ffffff",
            "color": color,
            "weight": 2,
            "fillOpacity": 0.00,
        }
    ).add_to(fg_bound)

fg_bound.add_to(m_demand)


# ------------------------------------------------------------
# Urban footprint
# ------------------------------------------------------------

fg_urban = folium.FeatureGroup(name="Derived urban footprints", show=True)

for _, r in urban_w.iterrows():
    folium.GeoJson(
        r.geometry,
        tooltip=f"{r['municipality']} urban footprint",
        style_function=lambda x: {
            "fillColor": "#B7DDF5",
            "color": "#0076C2",
            "weight": 1,
            "fillOpacity": 0.25,
        }
    ).add_to(fg_urban)

fg_urban.add_to(m_demand)


# ------------------------------------------------------------
# Demand buildings
# ------------------------------------------------------------

fg_build = folium.FeatureGroup(name="Demand buildings", show=False)

for _, r in demand_build_w.iterrows():
    pt = r.geometry.representative_point()

    inside = bool(r["inside_urban_footprint"])
    color = "#0076C2" if inside else "#999999"

    popup = (
        f"Building ID: {r['building_id']}<br>"
        f"Municipality: {r['municipality']}<br>"
        f"Inside urban footprint: {inside}<br>"
        f"Population: {r['assigned_population']:.2f}<br>"
        f"Wastewater: {r['wastewater_m3_day']:.3f} m³/day<br>"
        f"Road node: {r['node_id']}<br>"
        f"Snap distance: {r['snap_dist_m']:.1f} m"
    )

    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=1.2,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.50,
        weight=0.3,
        popup=popup
    ).add_to(fg_build)

fg_build.add_to(m_demand)


# ------------------------------------------------------------
# Demand road nodes
# ------------------------------------------------------------

fg_nodes = folium.FeatureGroup(name="Demand road nodes", show=True)

for _, r in demand_nodes_w.iterrows():
    pt = r.geometry
    pop = max(float(r["assigned_population"]), 0.0)
    radius = max(2.5, min(12, math.sqrt(pop) / 3.0))

    is_north = r["municipality"] in NORTH_MUNICIPALITIES
    color = "#0076C2" if is_north else "#777777"

    popup = (
        f"<b>{r['municipality']}</b><br>"
        f"Node ID: {r['node_id']}<br>"
        f"Buildings: {r['n_buildings']}<br>"
        f"Inside urban buildings: {r['n_buildings_inside_urban']}<br>"
        f"Outside urban buildings: {r['n_buildings_outside_urban']}<br>"
        f"Population: {r['assigned_population']:.1f}<br>"
        f"Wastewater: {r['wastewater_m3_day']:.2f} m³/day<br>"
        f"WWTP: {r['wwtp_assignment']}<br>"
        f"Component: {r['component']}"
    )

    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        weight=1,
        popup=popup
    ).add_to(fg_nodes)

fg_nodes.add_to(m_demand)


folium.LayerControl(collapsed=False).add_to(m_demand)

out_html = OUTPUT_DIR / "demand_road_nodes_check_map.html"
m_demand.save(out_html)

print("Saved demand road nodes check map:")
print(out_html)

C:\Users\lucam\AppData\Local\Temp\ipykernel_14652\3209762765.py:19: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = muni_w.geometry.unary_union.centroid


Saved demand road nodes check map:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\demand_road_nodes_check_map.html


In [13]:
# ============================================================
# BLOCK 13 -- EXISTING SEWER COVERAGE AND QGIS REVIEW EXPORT
# ============================================================
# Purpose:
# Identify urban demand nodes that are likely already covered by the existing
# sewer network, then export the remaining uncovered urban demand nodes for
# manual QGIS review.
#
# Manual workflow:
# 1. Open qgis_review_uncovered_urban_nodes_auto.gpkg in QGIS.
# 2. Delete the points that are false positives, i.e. nodes that the automatic
#    model marks as requiring new network but that are actually already covered.
# 3. Save/export the edited layer as:
#    qgis_review_uncovered_urban_nodes_manual.gpkg
# 4. Run Block 14 to import the manual file and update the coverage classification.

if "local_gdfs" not in globals() or "conduits" not in local_gdfs:
    raise RuntimeError("Existing conduits not found in local_gdfs. Run Block 3 first.")

if "demand_nodes_p" not in globals():
    raise RuntimeError("demand_nodes_p not found. Run Block 11 first.")

if "urban_footprints_p" not in globals():
    raise RuntimeError("urban_footprints_p not found. Run Block 7 first.")

if "muni_boundaries_p" not in globals():
    raise RuntimeError("muni_boundaries_p not found. Run Block 4 first.")


# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

# Direct service distance from existing pipes.
# Start conservative. If too many already-served nodes remain red, increase slightly.
EXISTING_SEWER_SERVICE_BUFFER_M = 80.0

# Limit the automatically covered area to the derived urban footprints.
LIMIT_EXISTING_SERVICE_TO_URBAN_FOOTPRINT = True

# Output names for QGIS manual review.
QGIS_REVIEW_UNCOVERED_NODES_NAME = "qgis_review_uncovered_urban_nodes_auto.gpkg"
QGIS_REVIEW_UNCOVERED_NODES_MANUAL_NAME = "qgis_review_uncovered_urban_nodes_manual.gpkg"


# ------------------------------------------------------------
# Load and prepare existing conduits
# ------------------------------------------------------------

existing_conduits_p = local_gdfs["conduits"].copy()

if existing_conduits_p.crs is None:
    raise RuntimeError("Existing conduits layer has no CRS.")

existing_conduits_p = existing_conduits_p.to_crs(CRS_PROJECTED)
existing_conduits_p = existing_conduits_p[existing_conduits_p.geometry.notna()].copy()
existing_conduits_p = existing_conduits_p[~existing_conduits_p.geometry.is_empty].copy()

existing_conduits_p = existing_conduits_p[
    existing_conduits_p.geometry.geom_type.isin(["LineString", "MultiLineString"])
].copy()

print("Existing conduits loaded:")
print(f"Rows: {len(existing_conduits_p):,}")
print("CRS:", existing_conduits_p.crs)


# ------------------------------------------------------------
# Clip existing conduits to northern municipalities
# ------------------------------------------------------------

north_boundaries_p = muni_boundaries_p[
    muni_boundaries_p["municipality"].isin(NORTH_MUNICIPALITIES)
].copy()

north_union = north_boundaries_p.geometry.union_all()

existing_conduits_north_p = existing_conduits_p[
    existing_conduits_p.intersects(north_union)
].copy()

existing_conduits_north_p["geometry"] = (
    existing_conduits_north_p.geometry.intersection(north_union)
)

existing_conduits_north_p = existing_conduits_north_p[
    existing_conduits_north_p.geometry.notna()
    & (~existing_conduits_north_p.geometry.is_empty)
].copy()

print("\nExisting conduits inside northern municipalities:")
print(f"Rows: {len(existing_conduits_north_p):,}")


# ------------------------------------------------------------
# Create automatic existing service area
# ------------------------------------------------------------

raw_service_geom = existing_conduits_north_p.geometry.buffer(
    EXISTING_SEWER_SERVICE_BUFFER_M
).union_all()

raw_service_area_p = gpd.GeoDataFrame(
    [{
        "name": "raw_existing_service_buffer",
        "service_buffer_m": EXISTING_SEWER_SERVICE_BUFFER_M,
        "geometry": raw_service_geom,
    }],
    geometry="geometry",
    crs=CRS_PROJECTED
)

north_urban_p = urban_footprints_p[
    urban_footprints_p["municipality"].isin(NORTH_MUNICIPALITIES)
].copy()

north_urban_union = north_urban_p.geometry.union_all()

if LIMIT_EXISTING_SERVICE_TO_URBAN_FOOTPRINT:
    existing_service_geom = raw_service_geom.intersection(north_urban_union)
else:
    existing_service_geom = raw_service_geom

existing_service_area_urban_p = gpd.GeoDataFrame(
    [{
        "name": "existing_service_area_automatic",
        "service_buffer_m": EXISTING_SEWER_SERVICE_BUFFER_M,
        "geometry": existing_service_geom,
    }],
    geometry="geometry",
    crs=CRS_PROJECTED
)

existing_service_area_urban_p = existing_service_area_urban_p[
    existing_service_area_urban_p.geometry.notna()
    & (~existing_service_area_urban_p.geometry.is_empty)
].copy()

print("\nAutomatic existing service area created.")
print(f"Raw buffer area: {raw_service_area_p.geometry.area.sum()/1e6:.3f} km²")
if len(existing_service_area_urban_p) > 0:
    print(f"Service area inside urban footprints: {existing_service_area_urban_p.geometry.area.sum()/1e6:.3f} km²")
else:
    print("WARNING: existing service area inside urban footprints is empty.")


# ------------------------------------------------------------
# Mark automatically covered demand nodes
# ------------------------------------------------------------

demand_nodes_p = demand_nodes_p.copy()

demand_nodes_p["is_urban_demand_node"] = demand_nodes_p["n_buildings_inside_urban"] > 0
demand_nodes_p["already_covered_by_existing_auto"] = False

north_demand_mask = demand_nodes_p["municipality"].isin(NORTH_MUNICIPALITIES)

if len(existing_service_area_urban_p) > 0:
    service_union = existing_service_area_urban_p.geometry.union_all()

    demand_nodes_p.loc[north_demand_mask, "already_covered_by_existing_auto"] = (
        demand_nodes_p.loc[north_demand_mask].geometry.intersects(service_union)
        | demand_nodes_p.loc[north_demand_mask].geometry.within(service_union)
    )

# At this stage, before manual correction:
demand_nodes_p["already_covered_by_existing"] = demand_nodes_p["already_covered_by_existing_auto"]

demand_nodes_p["requires_new_network_auto"] = (
    demand_nodes_p["municipality"].isin(NORTH_MUNICIPALITIES)
    & demand_nodes_p["is_urban_demand_node"]
    & (~demand_nodes_p["already_covered_by_existing_auto"])
)

demand_nodes_p["requires_new_network"] = demand_nodes_p["requires_new_network_auto"]


# ------------------------------------------------------------
# Export uncovered urban nodes for QGIS manual review
# ------------------------------------------------------------

qgis_review_nodes_p = demand_nodes_p[
    demand_nodes_p["requires_new_network_auto"]
].copy()

# Add stable manual-review ID.
# This must be preserved in QGIS.
qgis_review_nodes_p["manual_review_id"] = qgis_review_nodes_p["node_id"].astype(int)

# Add explicit action field. You can ignore it in QGIS if you prefer deleting points.
qgis_review_nodes_p["manual_status"] = "KEEP_REQUIRES_NEW_NETWORK"

qgis_review_nodes_p = qgis_review_nodes_p[
    [
        "manual_review_id",
        "node_id",
        "municipality",
        "component",
        "n_buildings",
        "n_buildings_inside_urban",
        "n_buildings_outside_urban",
        "assigned_population",
        "wastewater_m3_day",
        "already_covered_by_existing_auto",
        "requires_new_network_auto",
        "manual_status",
        "geometry",
    ]
].copy()

qgis_review_path = OUTPUT_DIR / QGIS_REVIEW_UNCOVERED_NODES_NAME

if qgis_review_path.exists():
    qgis_review_path.unlink()

qgis_review_nodes_p.to_file(
    qgis_review_path,
    layer="uncovered_urban_nodes_review",
    driver="GPKG"
)

print("\nExported QGIS review file:")
print(qgis_review_path)
print(f"Nodes exported for manual review: {len(qgis_review_nodes_p):,}")


# ------------------------------------------------------------
# Export supporting layers for QGIS
# ------------------------------------------------------------

qgis_support_path = OUTPUT_DIR / "qgis_review_support_layers.gpkg"

if qgis_support_path.exists():
    qgis_support_path.unlink()

north_boundaries_p.to_file(
    qgis_support_path,
    layer="north_municipal_boundaries",
    driver="GPKG"
)

north_urban_p.to_file(
    qgis_support_path,
    layer="north_urban_footprints",
    driver="GPKG"
)

existing_conduits_north_p.to_file(
    qgis_support_path,
    layer="existing_conduits_north",
    driver="GPKG"
)

raw_service_area_p.to_file(
    qgis_support_path,
    layer="raw_existing_service_buffer",
    driver="GPKG"
)

existing_service_area_urban_p.to_file(
    qgis_support_path,
    layer="existing_service_area_auto",
    driver="GPKG"
)

print("\nExported QGIS support layers:")
print(qgis_support_path)


# ------------------------------------------------------------
# Automatic coverage summary
# ------------------------------------------------------------

existing_coverage_summary_auto = (
    demand_nodes_p[demand_nodes_p["municipality"].isin(NORTH_MUNICIPALITIES)]
    .groupby("municipality")
    .agg(
        total_demand_nodes=("node_id", "count"),
        urban_demand_nodes=("is_urban_demand_node", "sum"),
        already_covered_nodes_auto=("already_covered_by_existing_auto", "sum"),
        new_required_nodes_auto=("requires_new_network_auto", "sum"),
        total_population=("assigned_population", "sum"),
        population_already_covered_auto=(
            "assigned_population",
            lambda x: x[demand_nodes_p.loc[x.index, "already_covered_by_existing_auto"]].sum()
        ),
        population_requiring_new_network_auto=(
            "assigned_population",
            lambda x: x[demand_nodes_p.loc[x.index, "requires_new_network_auto"]].sum()
        ),
        wastewater_already_covered_auto_m3_day=(
            "wastewater_m3_day",
            lambda x: x[demand_nodes_p.loc[x.index, "already_covered_by_existing_auto"]].sum()
        ),
        wastewater_requiring_new_network_auto_m3_day=(
            "wastewater_m3_day",
            lambda x: x[demand_nodes_p.loc[x.index, "requires_new_network_auto"]].sum()
        ),
    )
    .reset_index()
)

existing_coverage_summary_auto["already_covered_node_share_auto_percent"] = (
    existing_coverage_summary_auto["already_covered_nodes_auto"]
    / existing_coverage_summary_auto["urban_demand_nodes"].replace(0, pd.NA)
    * 100
).fillna(0.0)

print("\nAutomatic existing coverage summary:")
display(existing_coverage_summary_auto)

existing_coverage_summary_auto.to_csv(
    OUTPUT_DIR / "existing_sewer_coverage_summary_auto_north.csv",
    index=False
)

print("\nSaved automatic coverage summary:")
print(OUTPUT_DIR / "existing_sewer_coverage_summary_auto_north.csv")

Existing conduits loaded:
Rows: 6,380
CRS: EPSG:32636

Existing conduits inside northern municipalities:
Rows: 1,797

Automatic existing service area created.
Raw buffer area: 8.478 km²
Service area inside urban footprints: 7.671 km²

Exported QGIS review file:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\qgis_review_uncovered_urban_nodes_auto.gpkg
Nodes exported for manual review: 7,830

Exported QGIS support layers:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\qgis_review_support_layers.gpkg

Automatic existing coverage summary:


,municipality,total_demand_nodes,urban_demand_nodes,already_covered_nodes_auto,new_required_nodes_auto,total_population,population_already_covered_auto,population_requiring_new_network_auto,wastewater_already_covered_auto_m3_day,wastewater_requiring_new_network_auto_m3_day,already_covered_node_share_auto_percent
0,Aradippou,4974,4699,892,3807,22932.0,3679.430254,18795.037513,441.531630,2255.404502,18.982762
1,Kellia,271,197,0,197,387.0,0.000000,345.424468,0.000000,41.450936,0.000000
2,Livadia,1831,1811,266,1545,8581.0,1290.832284,7168.231874,154.899874,860.187825,14.688018
3,Oroklini,1829,1778,498,1280,7575.0,2034.846976,5474.158686,244.181637,656.899042,28.008999
4,Pyla,1438,1386,385,1001,2845.0,795.369691,2009.676993,95.444363,241.161239,27.777778



Saved automatic coverage summary:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\existing_sewer_coverage_summary_auto_north.csv


In [14]:
# ============================================================
# BLOCK 14 -- MANUAL QGIS REVIEW IMPORT AND COVERAGE UPDATE
# ============================================================
# Purpose:
# Read the manually edited QGIS file.
#
# Logic:
# - Original automatic file contains all urban nodes requiring new network.
# - Manual file contains only the nodes that still require new network after
#   manual deletion.
# - Therefore, deleted nodes are interpreted as manually reclassified as
#   already covered by the existing sewer network.
#
# Output:
# - demand_nodes_p updated with:
#   already_covered_by_existing_manual
#   excluded_by_manual_review
#   already_covered_by_existing
#   requires_new_network
# - manual correction summaries.

manual_path = OUTPUT_DIR / QGIS_REVIEW_UNCOVERED_NODES_MANUAL_NAME
auto_path = OUTPUT_DIR / QGIS_REVIEW_UNCOVERED_NODES_NAME

if not auto_path.exists():
    raise FileNotFoundError(f"Automatic QGIS review file not found: {auto_path}")

if not manual_path.exists():
    raise FileNotFoundError(
        f"Manual QGIS review file not found:\n{manual_path}\n\n"
        "Create it in QGIS by deleting false-positive nodes and saving the edited layer "
        "with suffix '_manual'."
    )

print("Reading automatic review file:")
print(auto_path)

qgis_auto = gpd.read_file(auto_path)

print("Reading manual review file:")
print(manual_path)

qgis_manual = gpd.read_file(manual_path)

if "manual_review_id" not in qgis_auto.columns:
    raise RuntimeError("manual_review_id missing in automatic review file.")

if "manual_review_id" not in qgis_manual.columns:
    raise RuntimeError("manual_review_id missing in manual review file.")

auto_ids = set(qgis_auto["manual_review_id"].astype(int).tolist())
manual_keep_ids = set(qgis_manual["manual_review_id"].astype(int).tolist())

manual_excluded_ids = sorted(auto_ids - manual_keep_ids)

print("\nManual review results:")
print(f"Automatic nodes requiring new network: {len(auto_ids):,}")
print(f"Manual nodes kept as requiring new network: {len(manual_keep_ids):,}")
print(f"Nodes manually excluded/reclassified as already covered: {len(manual_excluded_ids):,}")


# ------------------------------------------------------------
# Update demand_nodes_p
# ------------------------------------------------------------

demand_nodes_p = demand_nodes_p.copy()

demand_nodes_p["manual_review_id"] = demand_nodes_p["node_id"].astype(int)

demand_nodes_p["excluded_by_manual_review"] = (
    demand_nodes_p["manual_review_id"].isin(manual_excluded_ids)
)

# These are nodes that were originally considered uncovered, but you manually
# removed from the QGIS review layer because they are actually already covered.
demand_nodes_p["already_covered_by_existing_manual"] = (
    demand_nodes_p["excluded_by_manual_review"]
)

# Final already-covered classification:
# automatic covered OR manually reclassified as covered.
demand_nodes_p["already_covered_by_existing"] = (
    demand_nodes_p["already_covered_by_existing_auto"]
    | demand_nodes_p["already_covered_by_existing_manual"]
)

# Final new-network requirement:
# urban northern demand node AND not covered by existing automatic/manual.
demand_nodes_p["requires_new_network"] = (
    demand_nodes_p["municipality"].isin(NORTH_MUNICIPALITIES)
    & demand_nodes_p["is_urban_demand_node"]
    & (~demand_nodes_p["already_covered_by_existing"])
)

# Final source label
demand_nodes_p["coverage_class"] = "not_northern_or_nonurban"

demand_nodes_p.loc[
    demand_nodes_p["already_covered_by_existing_auto"],
    "coverage_class"
] = "covered_existing_auto"

demand_nodes_p.loc[
    demand_nodes_p["already_covered_by_existing_manual"],
    "coverage_class"
] = "covered_existing_manual"

demand_nodes_p.loc[
    demand_nodes_p["requires_new_network"],
    "coverage_class"
] = "requires_new_network"


# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

manual_excluded_nodes_p = demand_nodes_p[
    demand_nodes_p["excluded_by_manual_review"]
].copy()

manual_kept_nodes_p = demand_nodes_p[
    demand_nodes_p["requires_new_network"]
].copy()

manual_exclusion_summary = (
    manual_excluded_nodes_p
    .groupby("municipality")
    .agg(
        manually_excluded_nodes=("node_id", "count"),
        manually_excluded_population=("assigned_population", "sum"),
        manually_excluded_wastewater_m3_day=("wastewater_m3_day", "sum"),
        manually_excluded_buildings=("n_buildings", "sum"),
    )
    .reset_index()
)

final_existing_coverage_summary = (
    demand_nodes_p[demand_nodes_p["municipality"].isin(NORTH_MUNICIPALITIES)]
    .groupby("municipality")
    .agg(
        total_demand_nodes=("node_id", "count"),
        urban_demand_nodes=("is_urban_demand_node", "sum"),
        already_covered_nodes_auto=("already_covered_by_existing_auto", "sum"),
        already_covered_nodes_manual=("already_covered_by_existing_manual", "sum"),
        already_covered_nodes_final=("already_covered_by_existing", "sum"),
        new_required_nodes_final=("requires_new_network", "sum"),
        total_population=("assigned_population", "sum"),
        population_covered_auto=(
            "assigned_population",
            lambda x: x[demand_nodes_p.loc[x.index, "already_covered_by_existing_auto"]].sum()
        ),
        population_covered_manual=(
            "assigned_population",
            lambda x: x[demand_nodes_p.loc[x.index, "already_covered_by_existing_manual"]].sum()
        ),
        population_covered_existing_final=(
            "assigned_population",
            lambda x: x[demand_nodes_p.loc[x.index, "already_covered_by_existing"]].sum()
        ),
        population_requiring_new_network_final=(
            "assigned_population",
            lambda x: x[demand_nodes_p.loc[x.index, "requires_new_network"]].sum()
        ),
        wastewater_covered_existing_final_m3_day=(
            "wastewater_m3_day",
            lambda x: x[demand_nodes_p.loc[x.index, "already_covered_by_existing"]].sum()
        ),
        wastewater_requiring_new_network_final_m3_day=(
            "wastewater_m3_day",
            lambda x: x[demand_nodes_p.loc[x.index, "requires_new_network"]].sum()
        ),
    )
    .reset_index()
)

final_existing_coverage_summary["already_covered_node_share_final_percent"] = (
    final_existing_coverage_summary["already_covered_nodes_final"]
    / final_existing_coverage_summary["urban_demand_nodes"].replace(0, pd.NA)
    * 100
).fillna(0.0)

print("\n=== Manual exclusion summary ===")
display(manual_exclusion_summary)

print("\n=== Final existing coverage summary after manual correction ===")
display(final_existing_coverage_summary)

manual_exclusion_summary.to_csv(
    OUTPUT_DIR / "manual_review_exclusion_summary.csv",
    index=False
)

final_existing_coverage_summary.to_csv(
    OUTPUT_DIR / "existing_sewer_coverage_summary_after_manual_review.csv",
    index=False
)

manual_excluded_nodes_p.to_file(
    OUTPUT_DIR / "manual_review_excluded_nodes.gpkg",
    layer="manual_excluded_nodes",
    driver="GPKG"
)

manual_kept_nodes_p.to_file(
    OUTPUT_DIR / "final_nodes_requiring_new_network.gpkg",
    layer="nodes_requiring_new_network",
    driver="GPKG"
)

print("\nSaved:")
print(OUTPUT_DIR / "manual_review_exclusion_summary.csv")
print(OUTPUT_DIR / "existing_sewer_coverage_summary_after_manual_review.csv")
print(OUTPUT_DIR / "manual_review_excluded_nodes.gpkg")
print(OUTPUT_DIR / "final_nodes_requiring_new_network.gpkg")

Reading automatic review file:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\qgis_review_uncovered_urban_nodes_auto.gpkg
Reading manual review file:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\qgis_review_uncovered_urban_nodes_manual.gpkg


C:\Users\lucam\anaconda3\envs\Project_Goosee\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\qgis_review_uncovered_urban_nodes_manual.gpkg' may only be partially supported
  return ogr_read(



Manual review results:
Automatic nodes requiring new network: 7,817
Manual nodes kept as requiring new network: 7,483
Nodes manually excluded/reclassified as already covered: 334

=== Manual exclusion summary ===


,municipality,manually_excluded_nodes,manually_excluded_population,manually_excluded_wastewater_m3_day,manually_excluded_buildings
0,Aradippou,94,422.954326,50.754519,209
1,Larnaca,19,75.913875,9.109665,31
2,Livadia,71,481.360225,57.763227,157
3,Oroklini,76,230.749679,27.689962,131
4,Pyla,93,194.285946,23.314314,199



=== Final existing coverage summary after manual correction ===


,municipality,total_demand_nodes,urban_demand_nodes,already_covered_nodes_auto,already_covered_nodes_manual,already_covered_nodes_final,new_required_nodes_final,total_population,population_covered_auto,population_covered_manual,population_covered_existing_final,population_requiring_new_network_final,wastewater_covered_existing_final_m3_day,wastewater_requiring_new_network_final_m3_day,already_covered_node_share_final_percent
0,Aradippou,4974,4699,892,94,986,3713,22932.0,3679.430254,422.954326,4102.384580,18372.083187,492.286150,2204.649982,20.983188
1,Kellia,271,197,0,0,0,197,387.0,0.000000,0.000000,0.000000,345.424468,0.000000,41.450936,0.000000
2,Livadia,1831,1811,266,71,337,1474,8581.0,1290.832284,481.360225,1772.192509,6686.871649,212.663101,802.424598,18.608504
3,Oroklini,1829,1778,498,76,574,1204,7575.0,2034.846976,230.749679,2265.596655,5243.409007,271.871599,629.209081,32.283465
4,Pyla,1438,1386,385,93,478,908,2845.0,795.369691,194.285946,989.655637,1815.391047,118.758676,217.846926,34.487734



Saved:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\manual_review_exclusion_summary.csv
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\existing_sewer_coverage_summary_after_manual_review.csv
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\manual_review_excluded_nodes.gpkg
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\final_nodes_requiring_new_network.gpkg


In [15]:
# ============================================================
# BLOCK 15 -- MANUAL REVIEW CONSISTENCY CHECK
# ============================================================

manual_path = OUTPUT_DIR / "qgis_review_uncovered_urban_nodes_manual.gpkg"
auto_path = OUTPUT_DIR / "qgis_review_uncovered_urban_nodes_auto.gpkg"

print("Auto file:")
print(auto_path)
print("Exists:", auto_path.exists())

print("\nManual file:")
print(manual_path)
print("Exists:", manual_path.exists())

qgis_auto = gpd.read_file(auto_path)
qgis_manual = gpd.read_file(manual_path)

print("\nRows in automatic review file:", len(qgis_auto))
print("Rows in manual review file:", len(qgis_manual))

auto_ids = set(qgis_auto["manual_review_id"].astype(int))
manual_ids = set(qgis_manual["manual_review_id"].astype(int))

deleted_ids = auto_ids - manual_ids

print("\nNodes deleted manually:", len(deleted_ids))

print("\nCoverage class counts after manual review import:")
display(demand_nodes_p["coverage_class"].value_counts(dropna=False))

print("\nRequires new network count:")
print(demand_nodes_p["requires_new_network"].sum())

print("\nManual covered count:")
print(demand_nodes_p["already_covered_by_existing_manual"].sum())

print("\nNodes manually excluded but still requiring new network:")
bad = demand_nodes_p[
    demand_nodes_p["excluded_by_manual_review"] &
    demand_nodes_p["requires_new_network"]
]

display(bad[[
    "node_id",
    "municipality",
    "coverage_class",
    "excluded_by_manual_review",
    "already_covered_by_existing_manual",
    "requires_new_network",
    "assigned_population",
]])

Auto file:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\qgis_review_uncovered_urban_nodes_auto.gpkg
Exists: True

Manual file:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\qgis_review_uncovered_urban_nodes_manual.gpkg
Exists: True


C:\Users\lucam\anaconda3\envs\Project_Goosee\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\qgis_review_uncovered_urban_nodes_manual.gpkg' may only be partially supported
  return ogr_read(



Rows in automatic review file: 7830
Rows in manual review file: 7496

Nodes deleted manually: 334

Coverage class counts after manual review import:


coverage_class
not_northern_or_nonurban    10574
requires_new_network         7496
covered_existing_auto        2041
covered_existing_manual       353
Name: count, dtype: int64


Requires new network count:
7496

Manual covered count:
353

Nodes manually excluded but still requiring new network:


,node_id,municipality,coverage_class,excluded_by_manual_review,already_covered_by_existing_manual,requires_new_network,assigned_population


---
## BLOCK 16 — Final Coverage Map After Manual Review


In [16]:
# ============================================================
# BLOCK 16 -- FINAL COVERAGE MAP AFTER MANUAL REVIEW
# ============================================================

if not HAS_FOLIUM:
    raise ImportError("folium is not installed. Install with: pip install folium")

if "coverage_class" not in demand_nodes_p.columns:
    raise RuntimeError("coverage_class missing. Run Block 14 first.")

muni_w = muni_boundaries_p.to_crs(CRS_WGS84)
urban_w = urban_footprints_p.to_crs(CRS_WGS84)
existing_w = existing_conduits_north_p.to_crs(CRS_WGS84)
service_w = existing_service_area_urban_p.to_crs(CRS_WGS84)
demand_w = demand_nodes_p.to_crs(CRS_WGS84)

north_muni_w = muni_w[muni_w["municipality"].isin(NORTH_MUNICIPALITIES)].copy()
north_urban_w = urban_w[urban_w["municipality"].isin(NORTH_MUNICIPALITIES)].copy()
north_demand_w = demand_w[demand_w["municipality"].isin(NORTH_MUNICIPALITIES)].copy()

center = north_muni_w.geometry.union_all().centroid

m_manual = folium.Map(
    location=[center.y, center.x],
    zoom_start=12,
    tiles="OpenStreetMap"
)

Fullscreen().add_to(m_manual)
MiniMap(toggle_display=True).add_to(m_manual)


# ------------------------------------------------------------
# Boundaries
# ------------------------------------------------------------

fg_bound = folium.FeatureGroup(name="Municipal boundaries", show=True)

for _, r in muni_w.iterrows():
    is_north = r["municipality"] in NORTH_MUNICIPALITIES
    color = "#0076C2" if is_north else "#777777"

    folium.GeoJson(
        r.geometry,
        tooltip=f"{r['municipality']} boundary",
        style_function=lambda x, color=color: {
            "fillColor": "#ffffff",
            "color": color,
            "weight": 2,
            "fillOpacity": 0.00,
        }
    ).add_to(fg_bound)

fg_bound.add_to(m_manual)


# ------------------------------------------------------------
# Urban footprints
# ------------------------------------------------------------

fg_urban = folium.FeatureGroup(name="Northern urban footprints", show=True)

for _, r in north_urban_w.iterrows():
    folium.GeoJson(
        r.geometry,
        tooltip=f"{r['municipality']} urban footprint",
        style_function=lambda x: {
            "fillColor": "#B7DDF5",
            "color": "#0076C2",
            "weight": 1.5,
            "fillOpacity": 0.20,
        }
    ).add_to(fg_urban)

fg_urban.add_to(m_manual)


# ------------------------------------------------------------
# Existing service area
# ------------------------------------------------------------

fg_service = folium.FeatureGroup(name="Automatic existing service area", show=True)

for _, r in service_w.iterrows():
    folium.GeoJson(
        r.geometry,
        tooltip="Automatic existing service area",
        style_function=lambda x: {
            "fillColor": "#66BB6A",
            "color": "#2E7D32",
            "weight": 1,
            "fillOpacity": 0.25,
        }
    ).add_to(fg_service)

fg_service.add_to(m_manual)


# ------------------------------------------------------------
# Existing conduits
# ------------------------------------------------------------

fg_existing = folium.FeatureGroup(name="Existing conduits", show=True)

for _, r in existing_w.iterrows():
    folium.GeoJson(
        r.geometry,
        tooltip="Existing conduit",
        style_function=lambda x: {
            "color": "#004D40",
            "weight": 3,
            "opacity": 0.80,
        }
    ).add_to(fg_existing)

fg_existing.add_to(m_manual)


# ------------------------------------------------------------
# Nodes by final coverage class
# ------------------------------------------------------------

layer_specs = {
    "covered_existing_auto": {
        "name": "Covered by existing automatically",
        "color": "#2E7D32",
        "show": False,
        "radius": 3,
    },
    "covered_existing_manual": {
        "name": "Covered by existing manually corrected",
        "color": "#FF9800",
        "show": True,
        "radius": 4,
    },
    "requires_new_network": {
        "name": "Final nodes requiring new network",
        "color": "#E03C31",
        "show": True,
        "radius": 4,
    },
    "not_northern_or_nonurban": {
        "name": "Non-urban / not in northern optimisation",
        "color": "#999999",
        "show": False,
        "radius": 2,
    },
}

for class_name, spec in layer_specs.items():
    fg = folium.FeatureGroup(name=spec["name"], show=spec["show"])

    subset = north_demand_w[north_demand_w["coverage_class"] == class_name].copy()

    for _, r in subset.iterrows():
        pt = r.geometry

        popup = (
            f"<b>{r['municipality']}</b><br>"
            f"Node ID: {r['node_id']}<br>"
            f"Coverage class: {r['coverage_class']}<br>"
            f"Auto covered: {r['already_covered_by_existing_auto']}<br>"
            f"Manual covered: {r['already_covered_by_existing_manual']}<br>"
            f"Requires new network: {r['requires_new_network']}<br>"
            f"Buildings: {r['n_buildings']}<br>"
            f"Population: {r['assigned_population']:.2f}<br>"
            f"Wastewater: {r['wastewater_m3_day']:.3f} m³/day"
        )

        folium.CircleMarker(
            location=[pt.y, pt.x],
            radius=spec["radius"],
            color=spec["color"],
            fill=True,
            fill_color=spec["color"],
            fill_opacity=0.80,
            weight=1,
            popup=popup
        ).add_to(fg)

    fg.add_to(m_manual)


# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

fg_labels = folium.FeatureGroup(name="Municipality final labels", show=True)

final_label_df = (
    north_demand_w
    .groupby("municipality")
    .agg(
        urban_demand_nodes=("is_urban_demand_node", "sum"),
        final_required_nodes=("requires_new_network", "sum"),
        auto_covered_nodes=("already_covered_by_existing_auto", "sum"),
        manual_covered_nodes=("already_covered_by_existing_manual", "sum"),
        population=("assigned_population", "sum"),
    )
    .reset_index()
)

for _, r in north_muni_w.iterrows():
    muni = r["municipality"]
    centroid = r.geometry.centroid

    row = final_label_df[final_label_df["municipality"] == muni]

    if len(row) == 0:
        label_text = muni
        popup = f"{muni}<br>No demand nodes."
    else:
        row = row.iloc[0]
        label_text = (
            f"{muni}<br>"
            f"new {int(row['final_required_nodes'])} / "
            f"urban {int(row['urban_demand_nodes'])}"
        )

        popup = (
            f"<b>{muni}</b><br>"
            f"Urban demand nodes: {int(row['urban_demand_nodes'])}<br>"
            f"Auto covered nodes: {int(row['auto_covered_nodes'])}<br>"
            f"Manual covered nodes: {int(row['manual_covered_nodes'])}<br>"
            f"Final nodes requiring new network: {int(row['final_required_nodes'])}<br>"
            f"Population: {row['population']:.1f}"
        )

    folium.Marker(
        location=[centroid.y, centroid.x],
        popup=popup,
        tooltip=muni,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 9pt;
                font-weight: bold;
                color: #003B70;
                background-color: rgba(255,255,255,0.88);
                border: 1px solid #003B70;
                border-radius: 4px;
                padding: 3px 5px;
                text-align: center;
                white-space: nowrap;">
                {label_text}
            </div>
            """
        )
    ).add_to(fg_labels)

fg_labels.add_to(m_manual)

folium.LayerControl(collapsed=False).add_to(m_manual)

out_html = OUTPUT_DIR / "final_coverage_after_manual_review_map.html"
m_manual.save(out_html)

print("Saved final manual coverage map:")
print(out_html)

Saved final manual coverage map:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\final_coverage_after_manual_review_map.html


---
## BLOCK 17 — WWTP2 Site Selection

Goal: find a single candidate location for the new WWTP2 that

1. minimises the population-weighted distance to demand in the three admissible
   municipalities: Kellia, Oroklini and Aradippou;
2. falls outside the urban footprint by at least `WWTP2_MIN_DIST_FROM_URBAN_M`;
3. falls inside the union of the admissible municipal boundaries;
4. is snapped to the nearest road graph node, which becomes the common root used by
   the routing-heuristic blocks.

Important caveat: this centroid is computed in projected Euclidean space (EPSG:32636),
not directly on the road network. The Euclidean-optimal point and the
network-shortest-path optimal point are not guaranteed to coincide, so the method is a
practical approximation.


In [17]:
# ============================================================
# BLOCK 17 -- WWTP2 SITE SELECTION
# GEOGRAPHIC FILTER + LOW-ELEVATION P10 FILTER + SITE SCORE
# ============================================================

from pathlib import Path

required_vars_17 = [
    "demand_nodes_p", "muni_boundaries_p", "urban_footprints_p",
    "road_nodes_p", "G", "CRS_PROJECTED", "CRS_WGS84",
    "WWTP2_ADMISSIBLE_MUNICIPALITIES",
]
for v in required_vars_17:
    if v not in globals():
        raise RuntimeError(f"Block 17: '{v}' not found. Run previous blocks first.")

print("=" * 80)
print("BLOCK 17 -- WWTP2 SITE SELECTION: GEOGRAPHIC + LOW-ELEVATION P10 + SCORE")
print("=" * 80)


# ------------------------------------------------------------
# 17.0 Parameters
# ------------------------------------------------------------

WWTP2_MIN_DIST_FROM_URBAN_M = globals().get("WWTP2_MIN_DIST_FROM_URBAN_M", 150.0)
WWTP2_MAX_SNAP_DIST_M = globals().get("WWTP2_MAX_SNAP_DIST_M", 400.0)

WWTP2_HIGHWAY_REFS_NORTH_LIMIT = ["S5", "S6", "S7"]
WWTP2_NORTH_OF_HIGHWAY_MARGIN_M = 250.0
WWTP2_FALLBACK_NORTH_QUANTILE = 0.82
WWTP2_MIN_NORTH_OFFSET_FROM_CENTROID_M = 1200.0
WWTP2_PREFERRED_EXTRA_NORTH_M = 600.0

# Hard DEM-based low-elevation constraint.
# The final WWTP2 candidate must be inside the lowest 10% elevation range
# among the geographically feasible candidate set.
WWTP2_USE_LOW_ELEVATION_CONSTRAINT = globals().get("WWTP2_USE_LOW_ELEVATION_CONSTRAINT", True)
WWTP2_LOW_ELEVATION_PERCENTILE = globals().get("WWTP2_LOW_ELEVATION_PERCENTILE", 0.10)

# Same DEM folder used in the pumping-station notebook.
WWTP2_ELEVATION_DIR = Path(r"C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas")

# In the pumping-station notebook the file is called elevation1.tif.
# Try that first, then fall back to elevation.tif if needed.
_possible_elevation_paths = [
    WWTP2_ELEVATION_DIR / "elevation1.tif",
    WWTP2_ELEVATION_DIR / "Elevation1.tif",
    WWTP2_ELEVATION_DIR / "elevation.tif",
    WWTP2_ELEVATION_DIR / "Elevation.tif",
]

if "ELEVATION_TIF_PATH" in globals():
    _possible_elevation_paths.insert(0, Path(ELEVATION_TIF_PATH))

WWTP2_ELEVATION_RASTER_PATH = None
for _p in _possible_elevation_paths:
    if Path(_p).exists():
        WWTP2_ELEVATION_RASTER_PATH = Path(_p)
        break


def _geom_union(geoms):
    try:
        return geoms.union_all()
    except Exception:
        return geoms.unary_union


def _sample_dem_like_pumping_code(points_gdf, dem_tif, target_crs, output_col="elevation_m"):
    """
    Sample DEM at point geometries using the same logic used in the pumping-station notebook.

    Key point:
    - points are reprojected to the DEM CRS using GeoPandas .to_crs(dem_crs)
    - raster values are read with src.index() and a 1x1 window

    This avoids the previous rasterio RioCRS.from_user_input() issue.
    """
    try:
        import rasterio
        from rasterio.windows import Window
    except Exception as exc:
        raise RuntimeError(
            "Block 17: rasterio is required for DEM sampling, "
            f"but it could not be imported. Details: {exc}"
        )

    dem_tif = Path(dem_tif)

    if not dem_tif.exists():
        raise FileNotFoundError(
            "Block 17: elevation raster required but not found.\n"
            f"Expected path: {dem_tif}"
        )

    points_gdf = points_gdf.copy()

    if points_gdf.crs is None:
        points_gdf = points_gdf.set_crs(target_crs)
    else:
        points_gdf = points_gdf.to_crs(target_crs)

    with rasterio.open(str(dem_tif)) as src:
        dem_crs = src.crs
        dem_nodata = src.nodata

        if dem_crs is None:
            raise RuntimeError(
                "Block 17: the DEM raster has no CRS. "
                "Cannot safely sample elevations for WWTP2 selection."
            )

        print(f"DEM CRS                   : {dem_crs}")
        print(f"Road-node CRS before DEM  : {points_gdf.crs}")

        # This is the key line copied from the pumping-station logic.
        pts_proj = points_gdf.to_crs(dem_crs)

        elevs = []

        for geom in pts_proj.geometry:
            try:
                row, col = src.index(geom.x, geom.y)

                if row < 0 or col < 0 or row >= src.height or col >= src.width:
                    val = np.nan
                else:
                    arr = src.read(1, window=Window(col, row, 1, 1))
                    val = float(arr[0, 0])

                    if dem_nodata is not None and np.isclose(val, dem_nodata):
                        val = np.nan

            except Exception:
                val = np.nan

            elevs.append(val)

    points_gdf[output_col] = elevs

    missing_pct = 100 * sum(np.isnan(v) for v in elevs) / max(len(elevs), 1)
    print(f"DEM sampled points        : {len(elevs):,}")
    print(f"DEM missing percentage    : {missing_pct:.1f}%")

    return points_gdf


# ------------------------------------------------------------
# 17.1 Demand basis
# ------------------------------------------------------------

demand_basis = demand_nodes_p.copy()
demand_basis = demand_basis.set_crs(CRS_PROJECTED) if demand_basis.crs is None else demand_basis.to_crs(CRS_PROJECTED)

if "is_urban_demand_node" in demand_basis.columns:
    urban_mask = demand_basis["is_urban_demand_node"].fillna(False).astype(bool)
else:
    urban_mask = pd.Series(True, index=demand_basis.index)

demand_basis = demand_basis[
    demand_basis["municipality"].isin(WWTP2_ADMISSIBLE_MUNICIPALITIES)
    & urban_mask
    & (pd.to_numeric(demand_basis["assigned_population"], errors="coerce").fillna(0) > 0)
    & demand_basis.geometry.notna()
    & (~demand_basis.geometry.is_empty)
].copy()

if len(demand_basis) == 0:
    raise RuntimeError("Block 17: no admissible urban demand nodes found for WWTP2 site selection.")

demand_basis["assigned_population"] = pd.to_numeric(
    demand_basis["assigned_population"], errors="coerce"
).fillna(0.0)

xs = demand_basis.geometry.x.to_numpy()
ys = demand_basis.geometry.y.to_numpy()
weights = demand_basis["assigned_population"].to_numpy()

centroid_x = float(np.average(xs, weights=weights))
centroid_y = float(np.average(ys, weights=weights))
weighted_centroid_p = Point(centroid_x, centroid_y)

print(f"Admissible municipalities : {WWTP2_ADMISSIBLE_MUNICIPALITIES}")
print(f"Demand nodes considered   : {len(demand_basis):,}")
print(f"Population considered     : {weights.sum():,.0f}")
print(f"Weighted centroid         : x={centroid_x:.1f}, y={centroid_y:.1f}")


# ------------------------------------------------------------
# 17.2 Feasible territory and urban exclusion
# ------------------------------------------------------------

admissible_muni_p = muni_boundaries_p[
    muni_boundaries_p["municipality"].isin(WWTP2_ADMISSIBLE_MUNICIPALITIES)
].copy()
admissible_muni_p = admissible_muni_p.set_crs(CRS_PROJECTED) if admissible_muni_p.crs is None else admissible_muni_p.to_crs(CRS_PROJECTED)

urban_admissible_p = urban_footprints_p[
    urban_footprints_p["municipality"].isin(WWTP2_ADMISSIBLE_MUNICIPALITIES)
].copy()
urban_admissible_p = urban_admissible_p.set_crs(CRS_PROJECTED) if urban_admissible_p.crs is None else urban_admissible_p.to_crs(CRS_PROJECTED)

admissible_union = _geom_union(admissible_muni_p.geometry)
urban_union = _geom_union(urban_admissible_p.geometry)


# ------------------------------------------------------------
# 17.3 Try to detect S5/S6/S7 corridor
# ------------------------------------------------------------

highway_threshold_y = None
highway_source = None
highway_candidates_p = None

if "roads_p" in globals():
    roads_for_search = roads_p.copy()
    roads_for_search = roads_for_search.set_crs(CRS_PROJECTED) if roads_for_search.crs is None else roads_for_search.to_crs(CRS_PROJECTED)

    text_cols = [
        c for c in roads_for_search.columns
        if c != roads_for_search.geometry.name and roads_for_search[c].dtype == "object"
    ]

    if text_cols:
        mask_ref = pd.Series(False, index=roads_for_search.index)
        ref_pattern = "|".join(WWTP2_HIGHWAY_REFS_NORTH_LIMIT)

        for c in text_cols:
            mask_ref = mask_ref | roads_for_search[c].astype(str).str.contains(
                ref_pattern, case=False, na=False
            )

        highway_candidates_p = roads_for_search[
            mask_ref
            & roads_for_search.geometry.notna()
            & (~roads_for_search.geometry.is_empty)
            & roads_for_search.intersects(admissible_union.buffer(2500.0))
        ].copy()

        if len(highway_candidates_p) > 0:
            highway_threshold_y = float(highway_candidates_p.geometry.total_bounds[3])
            highway_threshold_y += WWTP2_NORTH_OF_HIGHWAY_MARGIN_M
            highway_source = "detected S5/S6/S7 road reference"

fallback_threshold_y = max(
    float(np.quantile(ys, WWTP2_FALLBACK_NORTH_QUANTILE)),
    centroid_y + WWTP2_MIN_NORTH_OFFSET_FROM_CENTROID_M,
)

if highway_threshold_y is None:
    north_threshold_y = fallback_threshold_y
    north_source = "fallback: demand north-quantile plus centroid offset"
else:
    north_threshold_y = max(highway_threshold_y, fallback_threshold_y)
    north_source = highway_source + " + fallback check"

preferred_y = north_threshold_y + WWTP2_PREFERRED_EXTRA_NORTH_M

print(f"North-threshold source    : {north_source}")
print(f"North-threshold y         : {north_threshold_y:.1f}")
print(f"Preferred y               : {preferred_y:.1f}")


# ------------------------------------------------------------
# 17.4 Candidate road-node preparation
# ------------------------------------------------------------

road_candidates = road_nodes_p.copy()
road_candidates = road_candidates.set_crs(CRS_PROJECTED) if road_candidates.crs is None else road_candidates.to_crs(CRS_PROJECTED)
road_candidates = road_candidates[
    road_candidates.geometry.notna()
    & (~road_candidates.geometry.is_empty)
].copy()

road_candidates["node_id"] = road_candidates["node_id"].astype(int)
road_candidates["inside_admissible"] = road_candidates.geometry.within(admissible_union)
road_candidates["dist_to_urban_m"] = road_candidates.geometry.distance(urban_union)
road_candidates["y_coord"] = road_candidates.geometry.y
road_candidates["north_ok"] = road_candidates["y_coord"] >= north_threshold_y
road_candidates["dist_to_demand_centroid_m"] = road_candidates.geometry.distance(weighted_centroid_p)
road_candidates["preferred_y_gap_m"] = (road_candidates["y_coord"] - preferred_y).abs()

dominant_component = None
if (
    "component" in demand_basis.columns
    and "component" in road_candidates.columns
    and demand_basis["component"].notna().any()
):
    dominant_component = int(
        demand_basis
        .dropna(subset=["component"])
        .groupby("component")["assigned_population"]
        .sum()
        .sort_values(ascending=False)
        .index[0]
    )
    road_candidates["component_ok"] = road_candidates["component"].astype("Int64") == dominant_component
else:
    road_candidates["component_ok"] = True


# ------------------------------------------------------------
# 17.5 Sample DEM elevation for all road-node candidates
# ------------------------------------------------------------

road_candidates["elevation_m"] = np.nan
road_candidates["low_elevation_ok"] = False

if WWTP2_USE_LOW_ELEVATION_CONSTRAINT:
    if WWTP2_ELEVATION_RASTER_PATH is None:
        raise FileNotFoundError(
            "Block 17: elevation raster not found.\n\n"
            "Expected one of these files:\n"
            + "\n".join([str(p) for p in _possible_elevation_paths])
            + "\n\nCheck that the file is located in:\n"
            + r"C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas"
        )

    print(f"Elevation raster          : {WWTP2_ELEVATION_RASTER_PATH}")

    road_candidates = _sample_dem_like_pumping_code(
        road_candidates,
        WWTP2_ELEVATION_RASTER_PATH,
        CRS_PROJECTED,
        output_col="elevation_m",
    )

    n_valid_elev_all = int(np.isfinite(road_candidates["elevation_m"].to_numpy()).sum())

    if n_valid_elev_all == 0:
        raise RuntimeError(
            "Block 17: elevation raster was found, but no valid elevation values "
            "could be sampled at the candidate road nodes.\n\n"
            "This means the DEM and the road-node geometries still do not overlap "
            "after CRS conversion. Check whether the DEM file is the same one used "
            "successfully in the pumping-station notebook."
        )

    print(f"Valid sampled elevations  : {n_valid_elev_all:,} / {len(road_candidates):,}")
    print(f"Low-elevation percentile  : p{WWTP2_LOW_ELEVATION_PERCENTILE * 100:.0f}")
else:
    road_candidates["low_elevation_ok"] = True
    print("WARNING: WWTP2 low-elevation constraint disabled.")


# ------------------------------------------------------------
# 17.6 Planning score
# ------------------------------------------------------------

# This is the final score used only after:
# 1. geographic filtering;
# 2. hard low-elevation p10 filtering.
#
# Lower score is better.
road_candidates["site_score"] = (
    road_candidates["dist_to_demand_centroid_m"]
    + 0.35 * road_candidates["preferred_y_gap_m"]
    - 0.05 * np.maximum(road_candidates["y_coord"] - north_threshold_y, 0)
)


# ------------------------------------------------------------
# 17.7 Geographic candidate sets
# ------------------------------------------------------------

candidate_sets_geo = [
    (
        "strict: admissible + north + urban clearance + dominant component",
        road_candidates[
            road_candidates["inside_admissible"]
            & road_candidates["north_ok"]
            & road_candidates["component_ok"]
            & (road_candidates["dist_to_urban_m"] >= WWTP2_MIN_DIST_FROM_URBAN_M)
        ],
    ),
    (
        "relaxed component: admissible + north + urban clearance",
        road_candidates[
            road_candidates["inside_admissible"]
            & road_candidates["north_ok"]
            & (road_candidates["dist_to_urban_m"] >= WWTP2_MIN_DIST_FROM_URBAN_M)
        ],
    ),
    (
        "relaxed urban clearance: admissible + north + dominant component",
        road_candidates[
            road_candidates["inside_admissible"]
            & road_candidates["north_ok"]
            & road_candidates["component_ok"]
        ],
    ),
    (
        "fallback: admissible + urban clearance + dominant component",
        road_candidates[
            road_candidates["inside_admissible"]
            & road_candidates["component_ok"]
            & (road_candidates["dist_to_urban_m"] >= WWTP2_MIN_DIST_FROM_URBAN_M)
        ],
    ),
]

selected_geo_rule = None
selected_geo_candidates = None

for rule, cand in candidate_sets_geo:
    if len(cand) > 0:
        selected_geo_rule = rule
        selected_geo_candidates = cand.copy()
        break

if selected_geo_candidates is None or len(selected_geo_candidates) == 0:
    raise RuntimeError("Block 17: no feasible WWTP2 road-node candidate found after geographic filtering.")

print("\nGeographic filtering:")
print(f"  Rule used             : {selected_geo_rule}")
print(f"  Geographic candidates : {len(selected_geo_candidates):,}")


# ------------------------------------------------------------
# 17.8 Hard low-elevation p10 filter
# ------------------------------------------------------------

elevation_threshold_m = np.nan
elevation_filter_status = "disabled"

if WWTP2_USE_LOW_ELEVATION_CONSTRAINT:
    selected_geo_candidates = selected_geo_candidates.copy()

    finite_geo_elev = np.isfinite(selected_geo_candidates["elevation_m"].to_numpy())
    geo_elev_values = selected_geo_candidates.loc[finite_geo_elev, "elevation_m"].astype(float)

    if len(geo_elev_values) == 0:
        raise RuntimeError(
            "Block 17: geographic candidates were found, but none of them has a valid DEM elevation value."
        )

    elevation_threshold_m = float(geo_elev_values.quantile(WWTP2_LOW_ELEVATION_PERCENTILE))

    selected_geo_candidates["low_elevation_ok"] = (
        np.isfinite(selected_geo_candidates["elevation_m"])
        & (selected_geo_candidates["elevation_m"] <= elevation_threshold_m)
    )

    selected_candidates = selected_geo_candidates[
        selected_geo_candidates["low_elevation_ok"]
    ].copy()

    if len(selected_candidates) == 0:
        raise RuntimeError(
            "Block 17: no candidate remained after the low-elevation p10 filter. "
            "Consider relaxing WWTP2_LOW_ELEVATION_PERCENTILE to 0.15 or 0.20."
        )

    elevation_filter_status = (
        f"active: candidate must be within lowest "
        f"{WWTP2_LOW_ELEVATION_PERCENTILE * 100:.0f}% elevation of geographic candidates"
    )

    print("\nLow-elevation filtering:")
    print(f"  Elevation threshold   : {elevation_threshold_m:.2f} m")
    print(f"  Candidates after p10  : {len(selected_candidates):,}")
    print(f"  Geo elevation min     : {geo_elev_values.min():.2f} m")
    print(f"  Geo elevation median  : {geo_elev_values.median():.2f} m")
    print(f"  Geo elevation max     : {geo_elev_values.max():.2f} m")
else:
    selected_candidates = selected_geo_candidates.copy()
    elevation_filter_status = "disabled"


# ------------------------------------------------------------
# 17.9 Final selection by site score
# ------------------------------------------------------------

selected_site = selected_candidates.sort_values("site_score").iloc[0]

WWTP2_ROOT_NODE = int(selected_site["node_id"])
wwtp2_site_point_p = selected_site.geometry
WWTP2_SNAP_DIST_M = 0.0

dist_to_urban_m = float(wwtp2_site_point_p.distance(urban_union))
north_clearance_m = float(wwtp2_site_point_p.y - north_threshold_y)
dist_to_centroid_m = float(wwtp2_site_point_p.distance(weighted_centroid_p))

selected_elevation_m = (
    float(selected_site["elevation_m"])
    if "elevation_m" in selected_site.index and pd.notna(selected_site["elevation_m"])
    else np.nan
)

selected_site_score = (
    float(selected_site["site_score"])
    if "site_score" in selected_site.index and pd.notna(selected_site["site_score"])
    else np.nan
)

selected_low_elevation_ok = (
    bool(selected_site["low_elevation_ok"])
    if "low_elevation_ok" in selected_site.index and pd.notna(selected_site["low_elevation_ok"])
    else False
)

site_wgs = gpd.GeoSeries([wwtp2_site_point_p], crs=CRS_PROJECTED).to_crs(CRS_WGS84).iloc[0]
centroid_wgs = gpd.GeoSeries([weighted_centroid_p], crs=CRS_PROJECTED).to_crs(CRS_WGS84).iloc[0]

WWTP2_LONLAT = (float(site_wgs.x), float(site_wgs.y))
WWTP2_LATLON = (float(site_wgs.y), float(site_wgs.x))

print("\nSelected WWTP2 candidate:")
print(f"  Selection sequence     : geographic filter -> low-elevation p10 -> site score")
print(f"  Geographic rule used   : {selected_geo_rule}")
print(f"  Root node              : {WWTP2_ROOT_NODE}")
print(f"  Dominant component     : {dominant_component}")
print(f"  Distance to urban      : {dist_to_urban_m:.1f} m")
print(f"  North clearance        : {north_clearance_m:.1f} m")
print(f"  Distance to centroid   : {dist_to_centroid_m:.1f} m")
print(f"  Site score             : {selected_site_score:.1f}")

if np.isfinite(selected_elevation_m):
    print(f"  Elevation              : {selected_elevation_m:.2f} m")
    print(f"  Elevation p10 limit    : {elevation_threshold_m:.2f} m")
    print(f"  Low-elevation OK       : {selected_low_elevation_ok}")
else:
    print("  Elevation              : not available")

print(f"  Elevation filter       : {elevation_filter_status}")
print(f"  WWTP2_LONLAT           : {WWTP2_LONLAT}")
print(f"  WWTP2_LATLON           : {WWTP2_LATLON}")

if dist_to_urban_m < WWTP2_MIN_DIST_FROM_URBAN_M:
    print(
        f"WARNING: selected site is only {dist_to_urban_m:.1f} m from urban footprint; "
        f"target was {WWTP2_MIN_DIST_FROM_URBAN_M:.1f} m."
    )

if north_clearance_m < 0:
    print("WARNING: selected site is not north of the intended threshold.")

if WWTP2_USE_LOW_ELEVATION_CONSTRAINT and not selected_low_elevation_ok:
    raise RuntimeError(
        "Block 17: selected WWTP2 candidate failed the low-elevation p10 constraint. "
        "This should not happen; check the filtering logic."
    )


# ------------------------------------------------------------
# 17.10 Save WWTP2 site layer
# ------------------------------------------------------------

wwtp2_site_p = gpd.GeoDataFrame(
    [{
        "site_type": "WWTP2_proposed_geo_low_elevation_p10",
        "method": "geographic_filter_low_elevation_p10_then_site_score",
        "root_node_id": WWTP2_ROOT_NODE,
        "dominant_component": dominant_component,
        "admissible_municipalities": ", ".join(WWTP2_ADMISSIBLE_MUNICIPALITIES),
        "population_basis": float(weights.sum()),
        "weighted_centroid_x": centroid_x,
        "weighted_centroid_y": centroid_y,
        "north_threshold_source": north_source,
        "north_threshold_y": float(north_threshold_y),
        "north_clearance_m": north_clearance_m,
        "dist_to_urban_m": dist_to_urban_m,
        "dist_to_centroid_m": dist_to_centroid_m,
        "elevation_m": selected_elevation_m,
        "elevation_raster": str(WWTP2_ELEVATION_RASTER_PATH),
        "elevation_filter_status": elevation_filter_status,
        "elevation_percentile_filter": float(WWTP2_LOW_ELEVATION_PERCENTILE),
        "elevation_threshold_m": elevation_threshold_m,
        "low_elevation_ok": selected_low_elevation_ok,
        "site_score": selected_site_score,
        "geographic_selection_rule": selected_geo_rule,
        "selection_sequence": "geographic_filter -> low_elevation_p10 -> site_score",
        "geometry": wwtp2_site_point_p,
    }],
    geometry="geometry",
    crs=CRS_PROJECTED,
)

wwtp2_site_path = shared_path("wwtp2_site_selection.gpkg")
if wwtp2_site_path.exists():
    wwtp2_site_path.unlink()

wwtp2_site_p.to_file(
    wwtp2_site_path,
    layer="wwtp2_proposed_site",
    driver="GPKG",
)
print(f"\nSaved: {wwtp2_site_path}")


# ------------------------------------------------------------
# 17.11 Sanity-check map
# ------------------------------------------------------------

if HAS_FOLIUM:
    m_site = folium.Map(location=[WWTP2_LATLON[0], WWTP2_LATLON[1]], zoom_start=13, tiles="OpenStreetMap")
    Fullscreen().add_to(m_site)
    MiniMap(toggle_display=True).add_to(m_site)

    admissible_muni_w = admissible_muni_p.to_crs(CRS_WGS84)
    urban_admissible_w = urban_admissible_p.to_crs(CRS_WGS84)

    fg_muni = folium.FeatureGroup(name="Admissible municipalities", show=True)
    for _, r in admissible_muni_w.iterrows():
        folium.GeoJson(
            r.geometry,
            tooltip=r["municipality"],
            style_function=lambda x: {
                "fillColor": "#ffffff",
                "fillOpacity": 0.00,
                "color": "#0076C2",
                "weight": 2,
            },
        ).add_to(fg_muni)
    fg_muni.add_to(m_site)

    fg_urban = folium.FeatureGroup(name="Urban footprints", show=True)
    for _, r in urban_admissible_w.iterrows():
        folium.GeoJson(
            r.geometry,
            tooltip=f"{r['municipality']} urban footprint",
            style_function=lambda x: {
                "fillColor": "#B7DDF5",
                "fillOpacity": 0.25,
                "color": "#0076C2",
                "weight": 1,
            },
        ).add_to(fg_urban)
    fg_urban.add_to(m_site)

    if highway_candidates_p is not None and len(highway_candidates_p) > 0:
        fg_highway = folium.FeatureGroup(name="Detected S5/S6/S7 corridor", show=True)
        for _, r in highway_candidates_p.to_crs(CRS_WGS84).iterrows():
            folium.GeoJson(
                r.geometry,
                tooltip="Detected S5/S6/S7 reference",
                style_function=lambda x: {
                    "color": "#222222",
                    "weight": 4,
                    "opacity": 0.8,
                },
            ).add_to(fg_highway)
        fg_highway.add_to(m_site)

    folium.CircleMarker(
        location=[centroid_wgs.y, centroid_wgs.x],
        radius=6,
        color="gray",
        fill=True,
        fill_color="gray",
        fill_opacity=0.8,
        tooltip="Population-weighted centroid",
    ).add_to(m_site)

    popup_elev = (
        f"{selected_elevation_m:.2f} m"
        if np.isfinite(selected_elevation_m)
        else "not available"
    )

    popup_threshold = (
        f"{elevation_threshold_m:.2f} m"
        if np.isfinite(elevation_threshold_m)
        else "not available"
    )

    folium.Marker(
        location=[WWTP2_LATLON[0], WWTP2_LATLON[1]],
        popup=(
            f"WWTP2 selected site<br>"
            f"Root node: {WWTP2_ROOT_NODE}<br>"
            f"Selection: geographic filter -> low-elevation p10 -> score<br>"
            f"Urban distance: {dist_to_urban_m:.0f} m<br>"
            f"North clearance: {north_clearance_m:.0f} m<br>"
            f"Elevation: {popup_elev}<br>"
            f"P10 elevation threshold: {popup_threshold}<br>"
            f"Site score: {selected_site_score:.0f}"
        ),
        icon=folium.Icon(color="red", icon="tint"),
    ).add_to(m_site)

    folium.LayerControl(collapsed=False).add_to(m_site)

    wwtp2_map_path = shared_path("wwtp2_site_selection_map.html")
    m_site.save(str(wwtp2_map_path))
    print(f"Saved: {wwtp2_map_path}")
else:
    print("Folium not installed: WWTP2 sanity-check map skipped.")

BLOCK 17 -- WWTP2 SITE SELECTION: GEOGRAPHIC + LOW-ELEVATION P10 + SCORE
Admissible municipalities : ['Kellia', 'Oroklini', 'Aradippou']
Demand nodes considered   : 6,674
Population considered     : 30,329
Weighted centroid         : x=555271.0, y=3867532.2
North-threshold source    : fallback: demand north-quantile plus centroid offset
North-threshold y         : 3870614.1
Preferred y               : 3871214.1
Elevation raster          : C:\Users\lucam\OneDrive\Desktop\Cyprus Project\areas\elevation1.tif
DEM CRS                   : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Road-node CRS before DEM  : EPSG:32636
DEM sampled points        : 66,849
DEM missing percentage    : 0.1%
Valid sampled elevations  : 66,806 / 66,849
Low-elevation percentile  : p10

Geographic filtering:
  Rule used             : strict: 

---
## BLOCK 18 — Shared Routing-Heuristic Setup

All routing heuristics connect the same set of required terminals
(`demand_nodes_p[requires_new_network] == True`) using the same root node
(`WWTP2_ROOT_NODE` from Block 17) on the same road graph `G`.

Only the connection algorithm differs. Therefore, differences in length, cost and
coverage are attributable to the routing heuristic, not to a different WWTP2 location.

The three methods are:

- **Block 19 — rooted_dijkstra:** baseline union of shortest paths from WWTP2 to all required terminals.
- **Block 20 — mst_steiner:** MST-based approximation to reduce total network length.
- **Block 21 — greedy_scored:** demand-weighted progressive expansion based on population served per incremental length.


In [18]:
# ============================================================
# BLOCK 18 -- SHARED ROUTING-HEURISTIC HELPERS
# ============================================================

required_vars_12 = [
    "G", "demand_nodes_p", "road_edges_p", "road_nodes_p",
    "existing_conduits_north_p", "existing_service_area_urban_p",
    "WWTP2_ROOT_NODE", "OUTPUT_DIR", "HEURISTIC_DIRS",
]
for v in required_vars_12:
    if v not in globals():
        raise RuntimeError(f"Block 18: '{v}' not found. Run previous blocks first.")


def get_edge_geometry_from_edge_id(edge_id, u, v):
    if edge_id is not None:
        match = road_edges_p[road_edges_p["edge_id"] == edge_id]
        if len(match) > 0:
            return match.iloc[0].geometry
    gu = road_nodes_p.loc[road_nodes_p["node_id"] == u, "geometry"].iloc[0]
    gv = road_nodes_p.loc[road_nodes_p["node_id"] == v, "geometry"].iloc[0]
    return LineString([gu, gv])


def compute_new_network_coverage(demand_nodes, selected_nodes):
    """Same accounting logic as the original routing block."""
    selected_nodes = set(int(n) for n in selected_nodes)
    dn = demand_nodes.copy()
    dn["covered_by_new_network"] = dn["node_id"].astype(int).isin(selected_nodes)

    mandatory = dn[dn["requires_new_network"]].copy()
    mandatory_covered = dn[dn["requires_new_network"] & dn["covered_by_new_network"]].copy()

    existing_auto = dn[dn.get("already_covered_by_existing_auto", False) == True].copy()
    existing_manual = dn[dn.get("already_covered_by_existing_manual", False) == True].copy()
    existing_final = dn[dn["already_covered_by_existing"] == True].copy()

    opportunistic = dn[
        dn["covered_by_new_network"]
        & (~dn["requires_new_network"])
        & (~dn["already_covered_by_existing"])
        & (dn["assigned_population"] > 0)
    ].copy()

    system_covered = dn[dn["already_covered_by_existing"] | dn["covered_by_new_network"]].copy()

    result = {
        "n_required_nodes": int(mandatory["node_id"].nunique()),
        "n_required_nodes_covered": int(mandatory_covered["node_id"].nunique()),
        "required_population": float(mandatory["assigned_population"].sum()),
        "required_population_covered": float(mandatory_covered["assigned_population"].sum()),
        "required_wastewater_m3_day": float(mandatory["wastewater_m3_day"].sum()),
        "required_wastewater_m3_day_covered": float(mandatory_covered["wastewater_m3_day"].sum()),
        "existing_auto_population": float(existing_auto["assigned_population"].sum()),
        "existing_manual_population": float(existing_manual["assigned_population"].sum()),
        "existing_total_population": float(existing_final["assigned_population"].sum()),
        "opportunistic_population": float(opportunistic["assigned_population"].sum()),
        "opportunistic_nodes": int(opportunistic["node_id"].nunique()),
        "system_covered_population": float(system_covered["assigned_population"].sum()),
        "system_covered_wastewater_m3_day": float(system_covered["wastewater_m3_day"].sum()),
        "system_covered_buildings": int(system_covered["n_buildings"].sum()),
    }
    result["required_node_coverage_percent"] = (
        100.0 * result["n_required_nodes_covered"] / result["n_required_nodes"]
        if result["n_required_nodes"] > 0 else 100.0
    )
    return result, dn, mandatory_covered, opportunistic


def edges_and_nodes_from_paths(paths_dict, terminals):
    """Given {target_node: [path nodes]}, return union of edges and nodes."""
    selected_edges = set()
    selected_nodes = set()
    unreachable = []
    for t in terminals:
        if t not in paths_dict:
            unreachable.append(t)
            continue
        path = paths_dict[t]
        for n in path:
            selected_nodes.add(int(n))
        for u, v in zip(path[:-1], path[1:]):
            selected_edges.add(tuple(sorted((int(u), int(v)))))
    return selected_edges, selected_nodes, unreachable


print("Block 18 helpers loaded.")


Block 18 helpers loaded.


---
### BLOCK 19 — Routing Heuristic: Rooted Dijkstra Baseline


In [19]:
# ============================================================
# BLOCK 19 -- ROUTING HEURISTIC: ROOTED DIJKSTRA BASELINE
# ============================================================
# Single Dijkstra from WWTP2_ROOT_NODE. Union of shortest paths from
# root to every required terminal. Identical logic to the original
# the original routing logic, only the root is now WWTP2_ROOT_NODE for every municipality
# instead of a per-municipality urban centroid.

HEURISTIC_NAME = "rooted_dijkstra"
print("=" * 80)
print(f"HEURISTIC: {HEURISTIC_NAME}")
print("=" * 80)

new_pipe_records_A = []
new_network_node_records_A = []
new_network_summary_rows_A = []
coverage_tables_A = []

for muni in NORTH_MUNICIPALITIES:
    print(f"\n--- {muni} ---")
    dn_all = demand_nodes_p[demand_nodes_p["municipality"] == muni].copy()
    if len(dn_all) == 0:
        continue
    dn_required = dn_all[dn_all["requires_new_network"]].copy()
    print(f"Required terminals: {dn_required['node_id'].nunique():,}")

    if len(dn_required) == 0:
        coverage_result, dn_cov, _, _ = compute_new_network_coverage(dn_all, set())
        dn_cov["municipality"] = muni
        dn_cov["component"] = -1
        coverage_tables_A.append(dn_cov)
        new_network_summary_rows_A.append({
            "municipality": muni, "component": -1, "root_node": WWTP2_ROOT_NODE,
            "n_selected_edges": 0, "n_selected_nodes": 0,
            "new_pipe_length_m": 0.0, "new_pipe_length_km": 0.0,
            "new_pipe_cost_eur": 0.0, **coverage_result,
        })
        continue

    for comp_id, dn_comp_required in dn_required.groupby("component"):
        dn_comp_all = dn_all[dn_all["component"] == comp_id].copy()
        terminals = sorted(dn_comp_required["node_id"].dropna().astype(int).unique().tolist())
        if len(terminals) == 0:
            continue

        comp_nodes = set(road_nodes_p.loc[road_nodes_p["component"] == comp_id, "node_id"]
                          .dropna().astype(int).tolist())
        comp_nodes.add(WWTP2_ROOT_NODE)
        G_sub = G.subgraph(comp_nodes).copy()

        if WWTP2_ROOT_NODE not in G_sub:
            print(f"  Component {comp_id}: root not reachable in this component, skipping.")
            continue

        lengths, paths = nx.single_source_dijkstra(G_sub, source=WWTP2_ROOT_NODE, weight="length_m")
        selected_edges, selected_nodes, unreachable = edges_and_nodes_from_paths(paths, terminals)

        length_m = 0.0
        for u, v in selected_edges:
            if not G_sub.has_edge(u, v):
                continue
            edge_data = G_sub[u][v]
            length = float(edge_data["length_m"])
            edge_id = edge_data.get("edge_id", None)
            length_m += length
            new_pipe_records_A.append({
                "municipality": muni, "component": int(comp_id), "u": int(u), "v": int(v),
                "edge_id": int(edge_id) if edge_id is not None else None,
                "length_m": length, "cost_eur": length * PIPE_COST_EUR_PER_M,
                "pipe_status": "new_proposed", "heuristic": HEURISTIC_NAME,
                "geometry": get_edge_geometry_from_edge_id(edge_id, u, v),
            })

        for n in selected_nodes:
            geom = road_nodes_p.loc[road_nodes_p["node_id"] == n, "geometry"].iloc[0]
            new_network_node_records_A.append({
                "municipality": muni, "component": int(comp_id), "node_id": int(n),
                "is_required_terminal": int(n in set(terminals)), "geometry": geom,
            })

        coverage_result, dn_cov, _, _ = compute_new_network_coverage(dn_comp_all, selected_nodes)
        dn_cov["municipality"] = muni
        dn_cov["component"] = int(comp_id)
        coverage_tables_A.append(dn_cov)

        new_network_summary_rows_A.append({
            "municipality": muni, "component": int(comp_id), "root_node": WWTP2_ROOT_NODE,
            "n_selected_edges": int(len(selected_edges)), "n_selected_nodes": int(len(selected_nodes)),
            "new_pipe_length_m": float(length_m), "new_pipe_length_km": float(length_m / 1000.0),
            "new_pipe_cost_eur": float(length_m * PIPE_COST_EUR_PER_M),
            "n_unreachable": int(len(unreachable)), **coverage_result,
        })
        print(f"  Component {comp_id}: {length_m/1000:.2f} km, "
              f"coverage {coverage_result['required_node_coverage_percent']:.1f}%")

new_pipes_A_p = gpd.GeoDataFrame(new_pipe_records_A, geometry="geometry", crs=CRS_PROJECTED) \
    if new_pipe_records_A else gpd.GeoDataFrame(columns=["municipality","component","u","v",
        "edge_id","length_m","cost_eur","pipe_status","heuristic","geometry"],
        geometry="geometry", crs=CRS_PROJECTED)

new_network_nodes_A_p = gpd.GeoDataFrame(new_network_node_records_A, geometry="geometry",
    crs=CRS_PROJECTED) if new_network_node_records_A else gpd.GeoDataFrame(
    columns=["municipality","component","node_id","is_required_terminal","geometry"],
    geometry="geometry", crs=CRS_PROJECTED)

new_network_summary_A = pd.DataFrame(new_network_summary_rows_A)
demand_coverage_A_p = gpd.GeoDataFrame(pd.concat(coverage_tables_A, ignore_index=True),
    geometry="geometry", crs=CRS_PROJECTED) if coverage_tables_A else gpd.GeoDataFrame(
    columns=list(demand_nodes_p.columns) + ["covered_by_new_network"],
    geometry="geometry", crs=CRS_PROJECTED)

print(f"\n=== {HEURISTIC_NAME} TOTAL ===")
print(f"Length: {new_pipes_A_p['length_m'].sum()/1000:.2f} km")
print(f"Cost:   EUR {new_pipes_A_p['cost_eur'].sum():,.0f}")


HEURISTIC: rooted_dijkstra

--- Oroklini ---
Required terminals: 1,204
  Component 0: 77.32 km, coverage 100.0%

--- Kellia ---
Required terminals: 197
  Component 0: 15.47 km, coverage 100.0%

--- Livadia ---
Required terminals: 1,474
  Component 0: 70.46 km, coverage 100.0%

--- Aradippou ---
Required terminals: 3,713
  Component 0: 275.75 km, coverage 100.0%

--- Pyla ---
Required terminals: 908
  Component 0: 86.77 km, coverage 100.0%
  Component 16: 0.00 km, coverage 0.0%

=== rooted_dijkstra TOTAL ===
Length: 525.76 km
Cost:   EUR 210,305,835


### BLOCK 20 — Routing Heuristic: Prim-Steiner (Nearest Unconnected Terminal)

Replaces MST-Steiner which caused `MemoryError` on large graphs.
Grows the tree one terminal at a time, always picking the **nearest**
unconnected terminal from the current tree frontier via a single
`multi_source_dijkstra` call per iteration. Memory cost: O(|V|) per
iteration, vs O(T × |V|) all at once for MST-Steiner.

**Key difference from Block 19 (rooted_dijkstra):**
Block 19 routes each terminal *independently* to the root — parallel
branches that overlap near the root. Block 20 always extends the
*current tree frontier*, so terminals near each other share a common
branch before reaching the root. Typically 10–20% less total pipe length.


In [20]:
# ============================================================
# BLOCK 20 -- ROUTING HEURISTIC: PRIM-STEINER
#             (NEAREST UNCONNECTED TERMINAL EXPANSION)
# ============================================================
# Computationally feasible alternative to MST-Steiner.
# MST-Steiner stored one full Dijkstra result per terminal simultaneously
# → MemoryError when terminals > ~100. This version does one
# multi_source_dijkstra per iteration, reuses it, and discards it.
#
# Algorithm:
#   1. Start tree = {WWTP2_ROOT_NODE}
#   2. While unconnected terminals remain:
#      a. Multi-source Dijkstra from all current tree nodes.
#      b. Pick unconnected terminal with MINIMUM distance to tree.
#      c. Add its path to the tree.
#   3. Repeat.
#
# Why results differ from Block 19 (rooted_dijkstra):
#   Block 19: each terminal routes to ROOT independently.
#             → Many long parallel paths converging at root.
#   Block 20: each terminal routes to nearest TREE NODE.
#             → Terminals cluster onto shared branches first,
#               then the branch reaches root.
#             → Fewer total km of pipe.

HEURISTIC_NAME = "prim_steiner"
print("=" * 80)
print(f"HEURISTIC: {HEURISTIC_NAME}")
print("Algorithm : Prim nearest-terminal expansion")
print("=" * 80)

new_pipe_records_B         = []
new_network_node_records_B = []
new_network_summary_rows_B = []
coverage_tables_B          = []

for muni in NORTH_MUNICIPALITIES:
    print(f"\n--- {muni} ---")
    dn_all      = demand_nodes_p[demand_nodes_p["municipality"] == muni].copy()
    if len(dn_all) == 0:
        continue
    dn_required = dn_all[dn_all["requires_new_network"]].copy()

    if len(dn_required) == 0:
        coverage_result, dn_cov, _, _ = compute_new_network_coverage(dn_all, set())
        dn_cov["municipality"] = muni
        dn_cov["component"]    = -1
        coverage_tables_B.append(dn_cov)
        new_network_summary_rows_B.append({
            "municipality": muni, "component": -1, "root_node": WWTP2_ROOT_NODE,
            "n_selected_edges": 0, "n_selected_nodes": 0,
            "new_pipe_length_m": 0.0, "new_pipe_length_km": 0.0,
            "new_pipe_cost_eur": 0.0, **coverage_result,
        })
        continue

    for comp_id, dn_comp_required in dn_required.groupby("component"):
        dn_comp_all = dn_all[dn_all["component"] == comp_id].copy()
        terminals   = sorted(
            dn_comp_required["node_id"].dropna().astype(int).unique().tolist()
        )
        if len(terminals) == 0:
            continue

        comp_nodes = set(
            road_nodes_p.loc[road_nodes_p["component"] == comp_id, "node_id"]
            .dropna().astype(int).tolist()
        )
        comp_nodes.add(WWTP2_ROOT_NODE)
        G_sub = G.subgraph(comp_nodes).copy()

        if WWTP2_ROOT_NODE not in G_sub:
            print(f"  Component {comp_id}: root not in subgraph, skipping.")
            continue

        # ── Prim-Steiner expansion ────────────────────────────────
        tree_nodes     = {WWTP2_ROOT_NODE}
        selected_edges = set()
        remaining      = set(t for t in terminals if t in G_sub)
        unreachable    = [t for t in terminals if t not in G_sub]
        n_iters        = 0

        while remaining:
            n_iters += 1
            # One multi-source Dijkstra from ALL current tree nodes.
            # Memory: one lengths dict + one paths dict (both O(|V|)).
            lengths, paths = nx.multi_source_dijkstra(
                G_sub, sources=list(tree_nodes), weight="length_m"
            )
            # Nearest unconnected terminal
            best_t, best_dist = None, float("inf")
            for t in remaining:
                d = lengths.get(t, float("inf"))
                if d < best_dist:
                    best_dist, best_t = d, t

            if best_t is None or best_dist == float("inf"):
                unreachable.extend(list(remaining))
                break

            path = paths[best_t]
            for n in path:
                tree_nodes.add(int(n))
            for u, v in zip(path[:-1], path[1:]):
                selected_edges.add(tuple(sorted((int(u), int(v)))))
            remaining.discard(best_t)

        selected_nodes = tree_nodes
        print(f"  Component {comp_id}: {n_iters} iters, "
              f"{len(selected_edges)} edges, "
              f"{len(unreachable)} unreachable")

        # ── Pipe records ──────────────────────────────────────────
        length_m = 0.0
        for u, v in selected_edges:
            if not G_sub.has_edge(u, v):
                continue
            ed      = G_sub[u][v]
            length  = float(ed["length_m"])
            edge_id = ed.get("edge_id", None)
            length_m += length
            new_pipe_records_B.append({
                "municipality": muni,
                "component":    int(comp_id),
                "u":            int(u),
                "v":            int(v),
                "edge_id":      int(edge_id) if edge_id is not None else None,
                "length_m":     length,
                "cost_eur":     length * PIPE_COST_EUR_PER_M,
                "pipe_status":  "new_proposed",
                "heuristic":    HEURISTIC_NAME,
                "geometry":     get_edge_geometry_from_edge_id(edge_id, u, v),
            })

        for n in selected_nodes:
            geom = road_nodes_p.loc[road_nodes_p["node_id"] == n, "geometry"].iloc[0]
            new_network_node_records_B.append({
                "municipality":         muni,
                "component":            int(comp_id),
                "node_id":              int(n),
                "is_required_terminal": int(n in set(terminals)),
                "geometry":             geom,
            })

        coverage_result, dn_cov, _, _ = compute_new_network_coverage(
            dn_comp_all, selected_nodes
        )
        dn_cov["municipality"] = muni
        dn_cov["component"]    = int(comp_id)
        coverage_tables_B.append(dn_cov)

        new_network_summary_rows_B.append({
            "municipality":       muni,
            "component":          int(comp_id),
            "root_node":          WWTP2_ROOT_NODE,
            "n_selected_edges":   int(len(selected_edges)),
            "n_selected_nodes":   int(len(selected_nodes)),
            "new_pipe_length_m":  float(length_m),
            "new_pipe_length_km": float(length_m / 1000.0),
            "new_pipe_cost_eur":  float(length_m * PIPE_COST_EUR_PER_M),
            "n_unreachable":      int(len(unreachable)),
            **coverage_result,
        })
        print(f"  Component {comp_id}: {length_m/1000:.2f} km  "
              f"coverage {coverage_result['required_node_coverage_percent']:.1f}%")

# ── Assemble output GeoDataFrames ─────────────────────────────────
_ecols_p = ["municipality","component","u","v","edge_id",
             "length_m","cost_eur","pipe_status","heuristic","geometry"]
new_pipes_B_p = (
    gpd.GeoDataFrame(new_pipe_records_B, geometry="geometry", crs=CRS_PROJECTED)
    if new_pipe_records_B
    else gpd.GeoDataFrame(columns=_ecols_p, geometry="geometry", crs=CRS_PROJECTED)
)

_ecols_n = ["municipality","component","node_id","is_required_terminal","geometry"]
new_network_nodes_B_p = (
    gpd.GeoDataFrame(new_network_node_records_B, geometry="geometry", crs=CRS_PROJECTED)
    if new_network_node_records_B
    else gpd.GeoDataFrame(columns=_ecols_n, geometry="geometry", crs=CRS_PROJECTED)
)

new_network_summary_B = pd.DataFrame(new_network_summary_rows_B)

demand_coverage_B_p = (
    gpd.GeoDataFrame(
        pd.concat(coverage_tables_B, ignore_index=True),
        geometry="geometry", crs=CRS_PROJECTED,
    )
    if coverage_tables_B
    else gpd.GeoDataFrame(
        columns=list(demand_nodes_p.columns) + ["covered_by_new_network"],
        geometry="geometry", crs=CRS_PROJECTED,
    )
)

print(f"\n=== {HEURISTIC_NAME} TOTAL ===")
print(f"Total pipe length : {new_pipes_B_p['length_m'].sum()/1000:.2f} km")
print(f"Total cost        : EUR {new_pipes_B_p['cost_eur'].sum():,.0f}")


HEURISTIC: prim_steiner
Algorithm : Prim nearest-terminal expansion

--- Oroklini ---
  Component 0: 1204 iters, 2188 edges, 0 unreachable
  Component 0: 60.44 km  coverage 100.0%

--- Kellia ---
  Component 0: 197 iters, 334 edges, 0 unreachable
  Component 0: 14.66 km  coverage 100.0%

--- Livadia ---
  Component 0: 1474 iters, 2646 edges, 0 unreachable
  Component 0: 56.83 km  coverage 100.0%

--- Aradippou ---
  Component 0: 3713 iters, 7039 edges, 0 unreachable
  Component 0: 220.06 km  coverage 100.0%

--- Pyla ---
  Component 0: 907 iters, 1922 edges, 0 unreachable
  Component 0: 62.71 km  coverage 100.0%
  Component 16: 1 iters, 0 edges, 1 unreachable
  Component 16: 0.00 km  coverage 0.0%

=== prim_steiner TOTAL ===
Total pipe length : 414.70 km
Total cost        : EUR 165,879,694


---
## BLOCK 22 — Routing-Heuristic Comparison Table


In [21]:
# ============================================================
# BLOCK 22 -- ROUTING-HEURISTIC COMPARISON TABLE
# ============================================================

comparison_rows = []
for name, pipes_gdf, summary_df in [
    ("rooted_dijkstra", new_pipes_A_p, new_network_summary_A),
    ("prim_steiner",    new_pipes_B_p, new_network_summary_B),
]:
    total_len_km = pipes_gdf["length_m"].sum() / 1000.0
    total_cost   = pipes_gdf["cost_eur"].sum()
    cov_pct      = summary_df["required_node_coverage_percent"].mean() if len(summary_df) else 0.0
    pop_covered  = summary_df["required_population_covered"].sum() if len(summary_df) else 0.0

    comparison_rows.append({
        "heuristic": name,
        "total_length_km": round(total_len_km, 2),
        "total_cost_eur": round(total_cost, 0),
        "mean_coverage_percent": round(cov_pct, 1),
        "required_population_covered": round(pop_covered, 0),
        "cost_per_capita_eur": round(total_cost / pop_covered, 1) if pop_covered > 0 else None,
    })

heuristic_comparison_df = pd.DataFrame(comparison_rows)
print("=" * 80)
print("HEURISTIC COMPARISON -- Block 22")
print("=" * 80)
print(heuristic_comparison_df.to_string(index=False))

baseline_len = heuristic_comparison_df.loc[
    heuristic_comparison_df["heuristic"] == "rooted_dijkstra", "total_length_km"
].iloc[0]
heuristic_comparison_df["length_vs_baseline_percent"] = (
    (heuristic_comparison_df["total_length_km"] - baseline_len) / baseline_len * 100
).round(1)

print("\nLength relative to rooted_dijkstra baseline:")
for _, r in heuristic_comparison_df.iterrows():
    sign = "+" if r["length_vs_baseline_percent"] >= 0 else ""
    print(f"  {r['heuristic']:<18} {sign}{r['length_vs_baseline_percent']:.1f}%")

heuristic_comparison_df.to_csv(shared_path("routing_heuristic_comparison_summary.csv"), index=False)
print(f"\nSaved: {shared_path('routing_heuristic_comparison_summary.csv')}")


HEURISTIC COMPARISON -- Block 22
      heuristic  total_length_km  total_cost_eur  mean_coverage_percent  required_population_covered  cost_per_capita_eur
rooted_dijkstra           525.76     210305835.0                   83.3                      32463.0               6478.3
   prim_steiner           414.70     165879694.0                   83.3                      32463.0               5109.8

Length relative to rooted_dijkstra baseline:
  rooted_dijkstra    +0.0%
  prim_steiner       -21.1%

Saved: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\routing_heuristic_comparison_summary.csv


---
## BLOCK 23 — Sewer-to-Road Helper Network

Runs **once** (heuristic-independent). Links existing conduits to road nodes.
Produces `conduits_road_helper_p` required by Block 25 (pipe balancing).


In [22]:
# ============================================================
# BLOCK 23 -- SEWER-TO-ROAD HELPER NETWORK (runs once, shared)
# ============================================================

# ============================================================
# BLOCK 13 -- SEWER-TO-ROAD HELPER NETWORK
# ============================================================
# Purpose:
# Build a helper network linking the existing sewer system to the road graph.
#
# NOTE: FromNode/ToNode are reconstructed geometrically from conduit endpoints
# (first/last point of each LineString) snapped directly to the nearest road node
# via cKDTree. No attribute columns required on the conduits layer.
#
# Outputs:
# - sewer_nodes_road_p:         sewer nodes / manholes linked to nearest road node
# - sewer_to_road_connectors_p: connector lines between sewer nodes and road nodes
# - conduits_road_helper_p:     existing conduits with FromNode/ToNode and road-node endpoints

from scipy.spatial import cKDTree
import numpy as np

# ------------------------------------------------------------
# 13.0 Input checks
# ------------------------------------------------------------

# (var checks omitted — guaranteed by heuristic loop context)
if "conduits" not in local_gdfs:
    raise RuntimeError("local_gdfs['conduits'] not found. Run Block 3 first.")

if "manholes" not in local_gdfs:
    raise RuntimeError("local_gdfs['manholes'] not found. Run Block 3 first.")

if "node_id" not in road_nodes_p.columns:
    raise RuntimeError("road_nodes_p must contain a 'node_id' column.")


# ------------------------------------------------------------
# 13.1 Parameters
# ------------------------------------------------------------

MAX_SEWER_TO_ROAD_CONNECTOR_M = 150.0
STUDY_AREA_BUFFER_M = 1000.0


# ------------------------------------------------------------
# 13.2 Prepare study area and road nodes
# ------------------------------------------------------------

study_area_p = muni_boundaries_p.geometry.union_all().buffer(STUDY_AREA_BUFFER_M)

road_nodes_helper_p = road_nodes_p.copy()
road_nodes_helper_p = road_nodes_helper_p.to_crs(CRS_PROJECTED)
road_nodes_helper_p = road_nodes_helper_p[
    road_nodes_helper_p.geometry.notna()
    & (~road_nodes_helper_p.geometry.is_empty)
].copy()

road_nodes_helper_p = road_nodes_helper_p[
    road_nodes_helper_p.intersects(study_area_p)
].copy()

road_nodes_lookup_p = road_nodes_helper_p[["node_id", "geometry"]].copy()
road_nodes_lookup_p = road_nodes_lookup_p.rename(columns={"node_id": "nearest_road_node_id"})

print("Road nodes available for sewer-road matching:")
print(f"Rows: {len(road_nodes_lookup_p):,}")


# ------------------------------------------------------------
# 13.3 Prepare existing sewer nodes / manholes
# ------------------------------------------------------------

manholes_p = local_gdfs["manholes"].copy()

if manholes_p.crs is None:
    raise RuntimeError("manholes layer has no CRS.")

manholes_p = manholes_p.to_crs(CRS_PROJECTED)
manholes_p = manholes_p[
    manholes_p.geometry.notna()
    & (~manholes_p.geometry.is_empty)
].copy()

manholes_p = manholes_p[manholes_p.intersects(study_area_p)].copy()

if "Name" not in manholes_p.columns:
    manholes_p = manholes_p.reset_index(drop=True)
    manholes_p["Name"] = ["MH_" + str(i) for i in manholes_p.index]
    print("WARNING: manholes layer has no 'Name' column — synthetic IDs assigned.")

sewer_nodes_p = manholes_p.copy()
sewer_nodes_p["sewer_node_name"] = sewer_nodes_p["Name"].astype(str)

sewer_nodes_p = (
    sewer_nodes_p[["sewer_node_name", "geometry"]]
    .drop_duplicates("sewer_node_name")
    .copy()
)

sewer_nodes_p = gpd.GeoDataFrame(sewer_nodes_p, geometry="geometry", crs=CRS_PROJECTED)

print("\nSewer nodes / manholes available:")
print(f"Rows: {len(sewer_nodes_p):,}")


# ------------------------------------------------------------
# 13.4 Match sewer nodes to nearest road node
# ------------------------------------------------------------

sewer_nodes_road_p = gpd.sjoin_nearest(
    sewer_nodes_p,
    road_nodes_lookup_p,
    how="left",
    distance_col="sewer_to_road_distance_m"
)

sewer_nodes_road_p = sewer_nodes_road_p.drop(columns=["index_right"], errors="ignore")
sewer_nodes_road_p = sewer_nodes_road_p.drop_duplicates("sewer_node_name").copy()
sewer_nodes_road_p["nearest_road_node_id"] = sewer_nodes_road_p["nearest_road_node_id"].astype("Int64")

print("\nSewer nodes matched to road nodes:")
print(f"Rows: {len(sewer_nodes_road_p):,}")
print(f"Mean distance: {sewer_nodes_road_p['sewer_to_road_distance_m'].mean():.2f} m")
print(f"Max distance: {sewer_nodes_road_p['sewer_to_road_distance_m'].max():.2f} m")

print("\nDistance diagnostics:")
display(sewer_nodes_road_p["sewer_to_road_distance_m"].describe().to_frame().T)


# ------------------------------------------------------------
# 13.5 Create sewer-to-road connector geometries
# ------------------------------------------------------------

road_geom_by_id = dict(
    zip(
        road_nodes_lookup_p["nearest_road_node_id"].astype(int),
        road_nodes_lookup_p.geometry
    )
)

connector_rows = []

for _, r in sewer_nodes_road_p.iterrows():
    if pd.isna(r["nearest_road_node_id"]):
        continue

    road_node_id = int(r["nearest_road_node_id"])
    road_geom = road_geom_by_id.get(road_node_id)

    if road_geom is None:
        continue

    connector_rows.append({
        "sewer_node_name": r["sewer_node_name"],
        "nearest_road_node_id": road_node_id,
        "sewer_to_road_distance_m": float(r["sewer_to_road_distance_m"]),
        "within_distance_threshold": float(r["sewer_to_road_distance_m"]) <= MAX_SEWER_TO_ROAD_CONNECTOR_M,
        "geometry": LineString([r.geometry, road_geom]),
    })

sewer_to_road_connectors_p = gpd.GeoDataFrame(
    connector_rows, geometry="geometry", crs=CRS_PROJECTED
)

print("\nSewer-to-road connectors created:")
print(f"Rows: {len(sewer_to_road_connectors_p):,}")
print(
    f"Within {MAX_SEWER_TO_ROAD_CONNECTOR_M:.0f} m: "
    f"{sewer_to_road_connectors_p['within_distance_threshold'].sum():,}"
)


# ------------------------------------------------------------
# 13.6 Prepare conduits
# ------------------------------------------------------------

conduits_p = local_gdfs["conduits"].copy()

if conduits_p.crs is None:
    raise RuntimeError("conduits layer has no CRS.")

conduits_p = conduits_p.to_crs(CRS_PROJECTED)
conduits_p = conduits_p[
    conduits_p.geometry.notna()
    & (~conduits_p.geometry.is_empty)
].copy()

conduits_p = conduits_p[
    conduits_p.geometry.geom_type.isin(["LineString", "MultiLineString"])
].copy()

conduits_p = conduits_p[conduits_p.intersects(study_area_p)].copy()
conduits_p = conduits_p.reset_index(drop=True)
conduits_p["existing_conduit_id"] = conduits_p.index.astype(int)

if "Name" in conduits_p.columns:
    conduits_p["existing_conduit_name"] = conduits_p["Name"].astype(str)
else:
    conduits_p["existing_conduit_name"] = conduits_p["existing_conduit_id"].astype(str)

print("\nExisting conduits available:")
print(f"Rows: {len(conduits_p):,}")


# ------------------------------------------------------------
# 13.6b Reconstruct FromNode / ToNode from conduit endpoints
# snapped directly to nearest road node via cKDTree
# ------------------------------------------------------------

road_coords = np.array([(g.x, g.y) for g in road_nodes_lookup_p.geometry])
road_ids    = road_nodes_lookup_p["nearest_road_node_id"].values
road_tree   = cKDTree(road_coords)

from_node_list = []
to_node_list   = []

for geom in conduits_p.geometry:
    if geom.geom_type == "MultiLineString":
        pt_from = Point(list(geom.geoms[0].coords)[0])
        pt_to   = Point(list(geom.geoms[-1].coords)[-1])
    else:
        coords  = list(geom.coords)
        pt_from = Point(coords[0])
        pt_to   = Point(coords[-1])

    _, idx_from = road_tree.query([pt_from.x, pt_from.y])
    _, idx_to   = road_tree.query([pt_to.x,   pt_to.y])

    from_node_list.append(str(road_ids[idx_from]))
    to_node_list.append(  str(road_ids[idx_to]))

conduits_p["FromNode"] = from_node_list
conduits_p["ToNode"]   = to_node_list

print(f"\nFromNode assigned to road node: {len(from_node_list):,} / {len(conduits_p):,}")
print(f"ToNode   assigned to road node: {len(to_node_list):,} / {len(conduits_p):,}")


# ------------------------------------------------------------
# 13.7 Attach FromNode/ToNode road-node geometry to conduits
# ------------------------------------------------------------

road_node_geom_lookup = road_nodes_lookup_p.copy()
road_node_geom_lookup["nearest_road_node_id"] = road_node_geom_lookup["nearest_road_node_id"].astype(str)

from_lookup = road_node_geom_lookup.rename(columns={
    "nearest_road_node_id": "FromNode",
    "geometry":             "from_road_geometry",
})
from_lookup["road_node_from"] = from_lookup["FromNode"].astype("int64")

to_lookup = road_node_geom_lookup.rename(columns={
    "nearest_road_node_id": "ToNode",
    "geometry":             "to_road_geometry",
})
to_lookup["road_node_to"] = to_lookup["ToNode"].astype("int64")

conduits_road_helper_p = conduits_p.merge(from_lookup, on="FromNode", how="left")
conduits_road_helper_p = conduits_road_helper_p.merge(to_lookup, on="ToNode", how="left")

conduits_road_helper_p["road_node_from"] = conduits_road_helper_p["road_node_from"].astype("Int64")
conduits_road_helper_p["road_node_to"]   = conduits_road_helper_p["road_node_to"].astype("Int64")

conduits_road_helper_p["has_from_road_node"] = conduits_road_helper_p["road_node_from"].notna()
conduits_road_helper_p["has_to_road_node"]   = conduits_road_helper_p["road_node_to"].notna()
conduits_road_helper_p["has_both_road_nodes"] = (
    conduits_road_helper_p["has_from_road_node"]
    & conduits_road_helper_p["has_to_road_node"]
)

print("\nConduits linked to road-node endpoints:")
print(f"Rows: {len(conduits_road_helper_p):,}")
print(f"With From road node: {conduits_road_helper_p['has_from_road_node'].sum():,}")
print(f"With To road node:   {conduits_road_helper_p['has_to_road_node'].sum():,}")
print(f"With both road nodes: {conduits_road_helper_p['has_both_road_nodes'].sum():,}")


# ------------------------------------------------------------
# 13.8 Diagnostics by Area if available
# ------------------------------------------------------------

if "Area" in conduits_road_helper_p.columns:
    print("\nConduit road-endpoint matching by Area:")
    display(
        conduits_road_helper_p
        .groupby("Area")
        .agg(
            n_conduits=("existing_conduit_id", "count"),
            matched_from=("has_from_road_node", "sum"),
            matched_to=("has_to_road_node", "sum"),
            matched_both=("has_both_road_nodes", "sum"),
        )
        .reset_index()
        .sort_values("n_conduits", ascending=False)
    )


# ------------------------------------------------------------
# 13.9 Clean multiple geometry columns before saving
# ------------------------------------------------------------

conduits_road_helper_export_p = conduits_road_helper_p.copy()
active_geometry_col = conduits_road_helper_export_p.geometry.name

extra_geometry_cols = [
    col for col in conduits_road_helper_export_p.columns
    if col != active_geometry_col
    and str(conduits_road_helper_export_p[col].dtype) == "geometry"
]

print("\nExtra geometry columns found in conduits_road_helper_p:")
print(extra_geometry_cols)

for col in extra_geometry_cols:
    wkt_col = f"{col}_wkt"
    conduits_road_helper_export_p[wkt_col] = conduits_road_helper_export_p[col].apply(
        lambda g: g.wkt if g is not None and not pd.isna(g) else None
    )
    conduits_road_helper_export_p = conduits_road_helper_export_p.drop(columns=[col])

conduits_road_helper_export_p = gpd.GeoDataFrame(
    conduits_road_helper_export_p,
    geometry=active_geometry_col,
    crs=CRS_PROJECTED
)


# ------------------------------------------------------------
# 13.10 Save outputs
# ------------------------------------------------------------

helper_gpkg = shared_path("Step1_Block13_sewer_to_road_helper_network.gpkg")

if helper_gpkg.exists():
    helper_gpkg.unlink()

sewer_nodes_road_p.to_file(
    helper_gpkg, layer="sewer_nodes_matched_to_road_nodes", driver="GPKG"
)
sewer_to_road_connectors_p.to_file(
    helper_gpkg, layer="sewer_to_road_connectors", driver="GPKG"
)
conduits_road_helper_export_p.to_file(
    helper_gpkg, layer="conduits_with_road_node_endpoints", driver="GPKG"
)

sewer_nodes_road_p.drop(columns="geometry").to_csv(
    shared_path("Step1_Block13_sewer_nodes_matched_to_road_nodes.csv"), index=False
)
conduits_road_helper_export_p.drop(columns="geometry").to_csv(
    shared_path("Step1_Block13_conduits_with_road_node_endpoints.csv"), index=False
)
sewer_to_road_connectors_p.drop(columns="geometry").to_csv(
    shared_path("Step1_Block13_sewer_to_road_connectors.csv"), index=False
)

print("\nSaved helper network:")
print(helper_gpkg)


# ------------------------------------------------------------
# 13.11 Keep clean helper variable for Block 14
# ------------------------------------------------------------

conduits_road_helper_p = conduits_road_helper_export_p.copy()

geom_cols_after = [
    col for col in conduits_road_helper_p.columns
    if str(conduits_road_helper_p[col].dtype) == "geometry"
]

print("\nClean conduits_road_helper_p prepared for Block 14.")
print(f"Rows: {len(conduits_road_helper_p):,}")
print("Geometry columns after cleaning:")
print(geom_cols_after)

if geom_cols_after != ["geometry"]:
    print("WARNING: more than one geometry column may still be present.")
else:
    print("OK: conduits_road_helper_p has only one geometry column.")

Road nodes available for sewer-road matching:
Rows: 66,183

Sewer nodes / manholes available:
Rows: 5,668

Sewer nodes matched to road nodes:
Rows: 5,668
Mean distance: 11.64 m
Max distance: 120.90 m

Distance diagnostics:


,count,mean,std,min,25%,50%,75%,max
sewer_to_road_distance_m,5668.0,11.644709,12.164827,0.066379,4.353129,6.986344,14.565015,120.897879



Sewer-to-road connectors created:
Rows: 5,668
Within 150 m: 5,668

Existing conduits available:
Rows: 6,379

FromNode assigned to road node: 6,379 / 6,379
ToNode   assigned to road node: 6,379 / 6,379

Conduits linked to road-node endpoints:
Rows: 6,379
With From road node: 6,379
With To road node:   6,379
With both road nodes: 6,379

Extra geometry columns found in conduits_road_helper_p:
['from_road_geometry', 'to_road_geometry']

Saved helper network:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\Step1_Block13_sewer_to_road_helper_network.gpkg

Clean conduits_road_helper_p prepared for Block 14.
Rows: 6,379
Geometry columns after cleaning:
['geometry']
OK: conduits_road_helper_p has only one geometry column.


---
## BLOCKS 24–26 — Per-Heuristic Downstream Pipeline

For each heuristic, in sequence:

- **Block 24** — Final two-network map and WWTP load summary (pre-balancing, equivalent to original Block 12.5)
- **Block 25** — Pipe-based WWTP reassignment and load balancing to 50/50 (original Block 14). Produces `balanced_load_nodes_p_h`, `remaining_existing_pipes_WWTP1_p_h`, `selected_existing_pipes_to_WWTP2_p_h`.
- **Block 25.5** — Final WWTP pipe assignment map (original Block 14.5)
- **Block 26** — Full statistics: §1 cost, §2 population pre and post balancing, §3 KPIs, §3b coverage by municipality, §4 map, §5 QGIS export, §6 CSV (original Block 15 format).


In [23]:
# ============================================================
# BLOCK 24-26 -- PER-HEURISTIC DOWNSTREAM PIPELINE
# ============================================================
# For each heuristic, in order:
#   Block 24: final two-network map and load summary (Block 12.5 equivalent)
#   Block 25: pipe-based WWTP reassignment and load balancing (Block 14)
#   Block 26: statistics, cost summary, map and QGIS export (Block 15)
# ============================================================

HEURISTIC_OUTPUTS = {
    "rooted_dijkstra": {
        "new_pipes_p":        new_pipes_A_p,
        "new_network_nodes_p": new_network_nodes_A_p,
        "demand_coverage_p":  demand_coverage_A_p,
        "summary":            new_network_summary_A,
    },
    "prim_steiner": {
        "new_pipes_p":        new_pipes_B_p,
        "new_network_nodes_p": new_network_nodes_B_p,
        "demand_coverage_p":  demand_coverage_B_p,
        "summary":            new_network_summary_B,
    },
}

for heuristic_name, outputs in HEURISTIC_OUTPUTS.items():
    new_pipes_p_h               = outputs["new_pipes_p"]
    heuristic_dir               = HEURISTIC_DIRS[heuristic_name]
    heuristic_dir.mkdir(parents=True, exist_ok=True)
    # Aliases so original block code uses correct per-heuristic variables
    municipal_demand_coverage_p = outputs["demand_coverage_p"]
    new_pipes_p                 = new_pipes_p_h

    print("\n" + "#" * 80)
    print(f"# HEURISTIC: {heuristic_name}")
    print("#" * 80)

    # ── BLOCK 24: Final two-network map and load summary ──────────
    print("\n" + "=" * 80)
    print(f"BLOCK 24 [{heuristic_name}] -- FINAL TWO-NETWORK MAP AND WWTP LOAD SUMMARY")
    print("=" * 80)

    # ============================================================
    # BLOCK 12.5 -- FINAL TWO-NETWORK MAP AND WWTP LOAD SUMMARY
    # ============================================================
    # Purpose:
    # Create final assignment and load summary distinguishing:
    #
    # Existing / WWTP1 network:
    # - urban demand nodes in southern municipalities;
    # - northern urban nodes automatically classified as already covered by existing sewer;
    # - northern urban nodes manually reclassified as already covered after QGIS review.
    #
    # New / WWTP2 network:
    # - northern urban demand nodes covered by the new proposed pipe network;
    # - excluding nodes already assigned to existing network to avoid double counting.
    #
    # IMPORTANT:
    # Population and wastewater loads are counted ONLY for urban demand nodes:
    # is_urban_demand_node == True
    #
    # IMPORTANT FIX:
    # covered_by_new_network is joined using:
    # node_id + municipality + component
    # because node_id alone is not unique across municipal demand aggregation.

    # ------------------------------------------------------------
    # 10.7.0 Input checks
    # ------------------------------------------------------------




    if "conduits" not in local_gdfs:
        raise RuntimeError("Existing conduits not found in local_gdfs. Run Block 3 first.")

    required_cols_demand = [
        "node_id",
        "municipality",
        "component",
        "assigned_population",
        "wastewater_m3_day",
        "n_buildings",
        "is_urban_demand_node",
    ]

    for c in required_cols_demand:
        if c not in demand_nodes_p.columns:
            raise RuntimeError(f"Column '{c}' missing from demand_nodes_p.")

    required_cols_cov = [
        "node_id",
        "municipality",
        "component",
        "covered_by_new_network",
    ]

    for c in required_cols_cov:
        if c not in municipal_demand_coverage_p.columns:
            raise RuntimeError(f"Column '{c}' missing from municipal_demand_coverage_p.")


    # ------------------------------------------------------------
    # 10.7.1 Prepare full existing conduits layer
    # ------------------------------------------------------------

    existing_conduits_all_p = local_gdfs["conduits"].copy()

    if existing_conduits_all_p.crs is None:
        raise RuntimeError("Existing conduits layer has no CRS.")

    existing_conduits_all_p = existing_conduits_all_p.to_crs(CRS_PROJECTED)
    existing_conduits_all_p = existing_conduits_all_p[
        existing_conduits_all_p.geometry.notna()
        & (~existing_conduits_all_p.geometry.is_empty)
    ].copy()

    existing_conduits_all_p = existing_conduits_all_p[
        existing_conduits_all_p.geometry.geom_type.isin(["LineString", "MultiLineString"])
    ].copy()

    study_union = muni_boundaries_p.geometry.union_all().buffer(1000)

    existing_conduits_all_p = existing_conduits_all_p[
        existing_conduits_all_p.intersects(study_union)
    ].copy()

    print("Existing conduits for final map:")
    print(f"Rows: {len(existing_conduits_all_p):,}")


    # ------------------------------------------------------------
    # 10.7.2 Define final network assignment at demand-node level
    # ------------------------------------------------------------

    final_load_nodes_p_h = demand_nodes_p.copy()

    # Ensure boolean fields exist
    for col in [
        "already_covered_by_existing_auto",
        "already_covered_by_existing_manual",
        "already_covered_by_existing",
        "requires_new_network",
        "is_urban_demand_node",
    ]:
        if col not in final_load_nodes_p_h.columns:
            final_load_nodes_p_h[col] = False

        final_load_nodes_p_h[col] = final_load_nodes_p_h[col].fillna(False).astype(bool)


    # ------------------------------------------------------------
    # Robust new-network coverage lookup
    # ------------------------------------------------------------
    # node_id alone is not unique.
    # Use node_id + municipality + component.

    new_coverage_lookup = municipal_demand_coverage_p.copy()

    new_coverage_lookup["covered_by_new_network"] = (
        new_coverage_lookup["covered_by_new_network"]
        .fillna(False)
        .astype(bool)
    )

    new_coverage_lookup = (
        new_coverage_lookup
        .groupby(["node_id", "municipality", "component"], as_index=False)
        .agg(
            covered_by_new_network=("covered_by_new_network", "max")
        )
    )

    final_load_nodes_p_h = final_load_nodes_p_h.merge(
        new_coverage_lookup,
        on=["node_id", "municipality", "component"],
        how="left"
    )

    final_load_nodes_p_h["covered_by_new_network"] = (
        final_load_nodes_p_h["covered_by_new_network"]
        .fillna(False)
        .astype(bool)
    )



    # ------------------------------------------------------------
    # Assignment logic
    # ------------------------------------------------------------

    # Existing / WWTP1 assignment:
    # all south municipality demand nodes + north existing-covered nodes.
    final_load_nodes_p_h["assigned_to_existing_WWTP1"] = (
        final_load_nodes_p_h["municipality"].isin(SOUTH_MUNICIPALITIES)
        | (
            final_load_nodes_p_h["municipality"].isin(NORTH_MUNICIPALITIES)
            & final_load_nodes_p_h["already_covered_by_existing"]
        )
    )

    # New / WWTP2 assignment -- north:
    # nodes covered by the new network, excluding existing-covered nodes.
    final_load_nodes_p_h["assigned_to_new_WWTP2"] = (
        final_load_nodes_p_h["municipality"].isin(NORTH_MUNICIPALITIES)
        & final_load_nodes_p_h["covered_by_new_network"]
        & (~final_load_nodes_p_h["assigned_to_existing_WWTP1"])
    )

    # New / WWTP2 assignment -- south opportunistic (Block 12.2):
    # Southern nodes that fall under or near new pipes crossing the south.
    # These are removed from WWTP1 and added to WWTP2.
    if "south_opportunistic_node_ids_h" in dir() and len(south_opportunistic_node_ids_h) > 0:

        south_opp_mask = (
            final_load_nodes_p_h["node_id"].astype(int).isin(south_opportunistic_node_ids_h)
            & final_load_nodes_p_h["is_urban_demand_node"].fillna(False).astype(bool)
        )

        # Remove from WWTP1
        final_load_nodes_p_h.loc[south_opp_mask, "assigned_to_existing_WWTP1"] = False

        # Add to WWTP2
        final_load_nodes_p_h.loc[south_opp_mask, "assigned_to_new_WWTP2"] = True

        # Tag for diagnostics
        final_load_nodes_p_h["south_opportunistic_WWTP2"] = (
            final_load_nodes_p_h["node_id"].astype(int).isin(south_opportunistic_node_ids_h)
        )

        n_opp = south_opp_mask.sum()
        pop_opp = final_load_nodes_p_h.loc[south_opp_mask, "assigned_population"].sum()
        print(f"South opportunistic nodes reassigned to WWTP2: {n_opp:,} | pop {pop_opp:,.1f}")

    else:
        final_load_nodes_p_h["south_opportunistic_WWTP2"] = False
        print("No south opportunistic nodes found (Block 12.2 not run or no nodes identified).")

    # Only urban nodes are relevant for final service-load accounting.
    final_load_nodes_p_h["load_counted_in_final_summary"] = (
        final_load_nodes_p_h["is_urban_demand_node"]
    )

    # Remaining northern urban demand not covered by either network.
    final_load_nodes_p_h["unassigned_after_new_network"] = (
        final_load_nodes_p_h["municipality"].isin(NORTH_MUNICIPALITIES)
        & final_load_nodes_p_h["is_urban_demand_node"]
        & (~final_load_nodes_p_h["assigned_to_existing_WWTP1"])
        & (~final_load_nodes_p_h["assigned_to_new_WWTP2"])
    )

    final_load_nodes_p_h["final_network_assignment"] = "not_counted_or_unassigned"

    final_load_nodes_p_h.loc[
        final_load_nodes_p_h["assigned_to_existing_WWTP1"]
        & final_load_nodes_p_h["is_urban_demand_node"],
        "final_network_assignment"
    ] = "existing_WWTP1"

    final_load_nodes_p_h.loc[
        final_load_nodes_p_h["assigned_to_new_WWTP2"]
        & final_load_nodes_p_h["is_urban_demand_node"],
        "final_network_assignment"
    ] = "new_WWTP2"

    final_load_nodes_p_h.loc[
        final_load_nodes_p_h["unassigned_after_new_network"],
        "final_network_assignment"
    ] = "unassigned"
    # ------------------------------------------------------------
    # 10.7.3 Check unassigned urban nodes
    # ------------------------------------------------------------

    unassigned_load = final_load_nodes_p_h[
        final_load_nodes_p_h["unassigned_after_new_network"]
    ].copy()

    print("\nUnassigned northern urban demand after corrected merge:")
    print(f"Nodes: {len(unassigned_load):,}")
    print(f"Population: {unassigned_load['assigned_population'].sum():,.3f}")
    print(f"Wastewater: {unassigned_load['wastewater_m3_day'].sum():,.3f} m³/day")

    if len(unassigned_load) > 0:
        display(
            unassigned_load[
                [
                    "node_id",
                    "municipality",
                    "component",
                    "requires_new_network",
                    "covered_by_new_network",
                    "assigned_to_existing_WWTP1",
                    "assigned_to_new_WWTP2",
                    "assigned_population",
                    "wastewater_m3_day",
                ]
            ].sort_values(["municipality", "component", "node_id"])
        )


    # ------------------------------------------------------------
    # 10.7.4 Load summary -- URBAN NODES ONLY
    # ------------------------------------------------------------

    existing_load = final_load_nodes_p_h[
        final_load_nodes_p_h["assigned_to_existing_WWTP1"]
        & final_load_nodes_p_h["is_urban_demand_node"]
    ].copy()

    new_load = final_load_nodes_p_h[
        final_load_nodes_p_h["assigned_to_new_WWTP2"]
        & final_load_nodes_p_h["is_urban_demand_node"]
    ].copy()

    unassigned_load = final_load_nodes_p_h[
        final_load_nodes_p_h["unassigned_after_new_network"]
        & final_load_nodes_p_h["is_urban_demand_node"]
    ].copy()

    network_load_summary = pd.DataFrame([
        {
            "network": "Existing network / WWTP1",
            "definition": "Urban demand in south municipalities + urban north auto-covered + urban north manually excluded",
            "n_demand_nodes": int(existing_load["node_id"].nunique()),
            "n_buildings": int(existing_load["n_buildings"].sum()),
            "population": float(existing_load["assigned_population"].sum()),
            "wastewater_m3_day": float(existing_load["wastewater_m3_day"].sum()),
        },
        {
            "network": "New network / WWTP2",
            "definition": "Urban demand nodes covered by proposed new network, excluding existing-covered nodes",
            "n_demand_nodes": int(new_load["node_id"].nunique()),
            "n_buildings": int(new_load["n_buildings"].sum()),
            "population": float(new_load["assigned_population"].sum()),
            "wastewater_m3_day": float(new_load["wastewater_m3_day"].sum()),
        },
        {
            "network": "Unassigned after new network",
            "definition": "Northern urban demand nodes not assigned to existing or new network",
            "n_demand_nodes": int(unassigned_load["node_id"].nunique()),
            "n_buildings": int(unassigned_load["n_buildings"].sum()) if len(unassigned_load) > 0 else 0,
            "population": float(unassigned_load["assigned_population"].sum()) if len(unassigned_load) > 0 else 0.0,
            "wastewater_m3_day": float(unassigned_load["wastewater_m3_day"].sum()) if len(unassigned_load) > 0 else 0.0,
        },
    ])

    total_assigned_q = network_load_summary.loc[
        network_load_summary["network"].isin(
            ["Existing network / WWTP1", "New network / WWTP2"]
        ),
        "wastewater_m3_day"
    ].sum()

    total_assigned_pop = network_load_summary.loc[
        network_load_summary["network"].isin(
            ["Existing network / WWTP1", "New network / WWTP2"]
        ),
        "population"
    ].sum()

    network_load_summary["population_share_percent"] = (
        network_load_summary["population"] / total_assigned_pop * 100
    ).fillna(0.0)

    network_load_summary["wastewater_share_percent"] = (
        network_load_summary["wastewater_m3_day"] / total_assigned_q * 100
    ).fillna(0.0)

    print("\n=== Final network load summary -- URBAN DEMAND ONLY ===")
    display(network_load_summary)

    network_load_summary.to_csv(
        heuristic_dir / "Step1_final_two_network_load_summary.csv",
        index=False
    )

    print("\nSaved:")
    print(heuristic_dir / "Step1_final_two_network_load_summary.csv")


    # ------------------------------------------------------------
    # 10.7.5 Municipality-level assignment summary -- URBAN ONLY
    # ------------------------------------------------------------

    municipality_final_assignment_summary = (
        final_load_nodes_p_h[
            final_load_nodes_p_h["is_urban_demand_node"]
        ]
        .groupby(["municipality", "final_network_assignment"])
        .agg(
            n_demand_nodes=("node_id", "count"),
            n_buildings=("n_buildings", "sum"),
            population=("assigned_population", "sum"),
            wastewater_m3_day=("wastewater_m3_day", "sum"),
        )
        .reset_index()
        .sort_values(["municipality", "final_network_assignment"])
    )

    print("\n=== Municipality-level final assignment summary -- URBAN DEMAND ONLY ===")
    display(municipality_final_assignment_summary)

    municipality_final_assignment_summary.to_csv(
        heuristic_dir / "Step1_final_assignment_by_municipality.csv",
        index=False
    )

    print("\nSaved:")
    print(heuristic_dir / "Step1_final_assignment_by_municipality.csv")


    # ------------------------------------------------------------
    # 10.7.6 Save final assignment layers
    # ------------------------------------------------------------

    final_assignment_gpkg = heuristic_dir / "Step1_final_two_network_assignment.gpkg"

    if final_assignment_gpkg.exists():
        final_assignment_gpkg.unlink()

    # Full node assignment, including non-urban nodes for diagnostics
    final_load_nodes_p_h.to_file(
        final_assignment_gpkg,
        layer="final_demand_node_assignment_all_nodes",
        driver="GPKG"
    )

    # Urban-only final accounting layers
    existing_load.to_file(
        final_assignment_gpkg,
        layer="urban_demand_existing_WWTP1",
        driver="GPKG"
    )

    new_load.to_file(
        final_assignment_gpkg,
        layer="urban_demand_new_WWTP2",
        driver="GPKG"
    )

    if len(unassigned_load) > 0:
        unassigned_load.to_file(
            final_assignment_gpkg,
            layer="urban_demand_unassigned",
            driver="GPKG"
        )

    existing_conduits_all_p.to_file(
        final_assignment_gpkg,
        layer="existing_conduits_WWTP1_network",
        driver="GPKG"
    )

    new_pipes_p_h.to_file(
        final_assignment_gpkg,
        layer="new_pipes_WWTP2_network",
        driver="GPKG"
    )

    print("\nSaved final assignment GeoPackage:")
    print(final_assignment_gpkg)


    # ------------------------------------------------------------
    # 10.7.7 Interactive map: two networks and urban served population
    # ------------------------------------------------------------

    if not HAS_FOLIUM:
        print("HAS_FOLIUM is False. Map not created.")

    else:
        muni_w = muni_boundaries_p.to_crs(CRS_WGS84)
        urban_w = urban_footprints_p.to_crs(CRS_WGS84)
        existing_w = existing_conduits_all_p.to_crs(CRS_WGS84)
        new_pipes_w = new_pipes_p_h.to_crs(CRS_WGS84)

        existing_nodes_w = existing_load.to_crs(CRS_WGS84)
        new_nodes_w = new_load.to_crs(CRS_WGS84)
        unassigned_w = unassigned_load.to_crs(CRS_WGS84) if len(unassigned_load) > 0 else None

        center = muni_w.geometry.union_all().centroid

        m_two_networks = folium.Map(
            location=[center.y, center.x],
            zoom_start=11,
            tiles="OpenStreetMap"
        )

        Fullscreen().add_to(m_two_networks)
        MiniMap(toggle_display=True).add_to(m_two_networks)

        # --------------------------------------------------------
        # Boundaries
        # --------------------------------------------------------

        fg_bound = folium.FeatureGroup(name="Municipal boundaries", show=True)

        for _, r in muni_w.iterrows():
            is_north = r["municipality"] in NORTH_MUNICIPALITIES
            color = "#0076C2" if is_north else "#777777"

            folium.GeoJson(
                r.geometry,
                tooltip=f"{r['municipality']} boundary",
                style_function=lambda x, color=color: {
                    "fillColor": "#ffffff",
                    "color": color,
                    "weight": 2,
                    "fillOpacity": 0.00,
                }
            ).add_to(fg_bound)

        fg_bound.add_to(m_two_networks)

        # --------------------------------------------------------
        # Urban footprints
        # --------------------------------------------------------

        fg_urban = folium.FeatureGroup(name="Urban footprints", show=False)

        for _, r in urban_w.iterrows():
            folium.GeoJson(
                r.geometry,
                tooltip=f"{r['municipality']} urban footprint",
                style_function=lambda x: {
                    "fillColor": "#B7DDF5",
                    "color": "#0076C2",
                    "weight": 1,
                    "fillOpacity": 0.18,
                }
            ).add_to(fg_urban)

        fg_urban.add_to(m_two_networks)

        # --------------------------------------------------------
        # Existing network / WWTP1 pipes
        # --------------------------------------------------------

        fg_existing_network = folium.FeatureGroup(
            name="Existing network / WWTP1",
            show=True
        )

        for _, r in existing_w.iterrows():
            folium.GeoJson(
                r.geometry,
                tooltip="Existing sewer conduit / WWTP1 network",
                style_function=lambda x: {
                    "color": "#1B5E20",
                    "weight": 3,
                    "opacity": 0.75,
                }
            ).add_to(fg_existing_network)

        fg_existing_network.add_to(m_two_networks)

        # --------------------------------------------------------
        # New network / WWTP2 pipes
        # --------------------------------------------------------

        fg_new_network = folium.FeatureGroup(
            name="New proposed network / WWTP2",
            show=True
        )

        for _, r in new_pipes_w.iterrows():
            popup = (
                f"Municipality: {r['municipality']}<br>"
                f"Length: {r['length_m']:.1f} m<br>"
                f"Cost: €{r['cost_eur']:,.0f}"
            )

            folium.GeoJson(
                r.geometry,
                tooltip="New proposed pipe / WWTP2 network",
                popup=popup,
                style_function=lambda x: {
                    "color": "#003B70",
                    "weight": 4,
                    "opacity": 0.95,
                }
            ).add_to(fg_new_network)

        fg_new_network.add_to(m_two_networks)

        # --------------------------------------------------------
        # Urban demand nodes served by existing / WWTP1
        # --------------------------------------------------------

        fg_existing_nodes = folium.FeatureGroup(
            name="Urban demand served by existing / WWTP1",
            show=False
        )

        for _, r in existing_nodes_w.iterrows():
            pt = r.geometry

            popup = (
                f"<b>{r['municipality']}</b><br>"
                f"Node ID: {r['node_id']}<br>"
                f"Assignment: existing_WWTP1<br>"
                f"Urban demand: {r['is_urban_demand_node']}<br>"
                f"Coverage class: {r.get('coverage_class', '')}<br>"
                f"Population: {r['assigned_population']:.2f}<br>"
                f"Wastewater: {r['wastewater_m3_day']:.3f} m³/day"
            )

            folium.CircleMarker(
                location=[pt.y, pt.x],
                radius=3,
                color="#2E7D32",
                fill=True,
                fill_color="#2E7D32",
                fill_opacity=0.65,
                weight=0.8,
                popup=popup
            ).add_to(fg_existing_nodes)

        fg_existing_nodes.add_to(m_two_networks)

        # --------------------------------------------------------
        # Urban demand nodes served by new / WWTP2
        # --------------------------------------------------------

        fg_new_nodes = folium.FeatureGroup(
            name="Urban demand served by new / WWTP2",
            show=False
        )

        for _, r in new_nodes_w.iterrows():
            pt = r.geometry

            popup = (
                f"<b>{r['municipality']}</b><br>"
                f"Node ID: {r['node_id']}<br>"
                f"Assignment: new_WWTP2<br>"
                f"Urban demand: {r['is_urban_demand_node']}<br>"
                f"Population: {r['assigned_population']:.2f}<br>"
                f"Wastewater: {r['wastewater_m3_day']:.3f} m³/day"
            )

            folium.CircleMarker(
                location=[pt.y, pt.x],
                radius=3,
                color="#003B70",
                fill=True,
                fill_color="#003B70",
                fill_opacity=0.70,
                weight=0.8,
                popup=popup
            ).add_to(fg_new_nodes)

        fg_new_nodes.add_to(m_two_networks)

        # --------------------------------------------------------
        # Optional unassigned urban nodes
        # --------------------------------------------------------

        if unassigned_w is not None and len(unassigned_w) > 0:
            fg_unassigned = folium.FeatureGroup(
                name="Unassigned northern urban demand",
                show=True
            )

            for _, r in unassigned_w.iterrows():
                pt = r.geometry

                popup = (
                    f"<b>{r['municipality']}</b><br>"
                    f"Node ID: {r['node_id']}<br>"
                    f"Assignment: unassigned<br>"
                    f"Population: {r['assigned_population']:.2f}<br>"
                    f"Wastewater: {r['wastewater_m3_day']:.3f} m³/day"
                )

                folium.CircleMarker(
                    location=[pt.y, pt.x],
                    radius=5,
                    color="#E03C31",
                    fill=True,
                    fill_color="#E03C31",
                    fill_opacity=0.85,
                    weight=1,
                    popup=popup
                ).add_to(fg_unassigned)

            fg_unassigned.add_to(m_two_networks)

        # --------------------------------------------------------
        # Population labels
        # --------------------------------------------------------

        fg_labels = folium.FeatureGroup(
            name="Urban load labels",
            show=True
        )

        existing_row = network_load_summary[
            network_load_summary["network"] == "Existing network / WWTP1"
        ].iloc[0]

        new_row = network_load_summary[
            network_load_summary["network"] == "New network / WWTP2"
        ].iloc[0]

        if len(existing_nodes_w) > 0:
            existing_centroid = existing_nodes_w.geometry.union_all().centroid

            existing_label = (
                f"Existing / WWTP1<br>"
                f"Urban pop: {existing_row['population']:,.0f}<br>"
                f"Q: {existing_row['wastewater_m3_day']:,.0f} m³/day"
            )

            folium.Marker(
                location=[existing_centroid.y, existing_centroid.x],
                tooltip="Existing / WWTP1 urban load",
                popup=existing_label,
                icon=folium.DivIcon(
                    html=f"""
                    <div style="
                        font-size: 11pt;
                        font-weight: bold;
                        color: #1B5E20;
                        background-color: rgba(255,255,255,0.90);
                        border: 2px solid #1B5E20;
                        border-radius: 6px;
                        padding: 5px 8px;
                        text-align: center;
                        white-space: nowrap;">
                        {existing_label}
                    </div>
                    """
                )
            ).add_to(fg_labels)

        if len(new_nodes_w) > 0:
            new_centroid = new_nodes_w.geometry.union_all().centroid

            new_label = (
                f"New / WWTP2<br>"
                f"Urban pop: {new_row['population']:,.0f}<br>"
                f"Q: {new_row['wastewater_m3_day']:,.0f} m³/day"
            )

            folium.Marker(
                location=[new_centroid.y, new_centroid.x],
                tooltip="New / WWTP2 urban load",
                
                popup=new_label,
                icon=folium.DivIcon(
                    html=f"""
                    <div style="
                        font-size: 11pt;
                        font-weight: bold;
                        color: #003B70;
                        background-color: rgba(255,255,255,0.90);
                        border: 2px solid #003B70;
                        border-radius: 6px;
                        padding: 5px 8px;
                        text-align: center;
                        white-space: nowrap;">
                        {new_label}
                    </div>
                    """
                )
            ).add_to(fg_labels)

        fg_labels.add_to(m_two_networks)

        # --------------------------------------------------------
        # Legend
        # --------------------------------------------------------

        legend_html = """
        <div style="
            position: fixed;
            bottom: 35px;
            left: 35px;
            width: 290px;
            z-index: 9999;
            background-color: rgba(255,255,255,0.92);
            border: 2px solid #333333;
            border-radius: 6px;
            padding: 10px;
            font-size: 10pt;">
            <b>Urban demand assignment</b><br>
            <span style="color:#1B5E20;font-weight:bold;">━━</span> Existing network / WWTP1<br>
            <span style="color:#003B70;font-weight:bold;">━━</span> New proposed network / WWTP2<br>
            <span style="color:#2E7D32;">●</span> Urban demand served by existing<br>
            <span style="color:#003B70;">●</span> Urban demand served by new network<br>
            <span style="color:#E03C31;">●</span> Unassigned urban demand, if any
        </div>
        """

        m_two_networks.get_root().html.add_child(folium.Element(legend_html))

        folium.LayerControl(collapsed=False).add_to(m_two_networks)

        out_html = heuristic_dir / "Step1_final_two_networks_population_map.html"
        m_two_networks.save(out_html)

        print("\nSaved final two-network population map:")
        print(out_html)
    # ── BLOCK 25: Pipe-based WWTP reassignment and load balancing ─
    print("\n" + "=" * 80)
    print(f"BLOCK 25 [{heuristic_name}] -- PIPE-BASED WWTP REASSIGNMENT AND LOAD BALANCING")
    print("=" * 80)

    # ============================================================
    # BLOCK 14 -- PIPE-BASED WWTP REASSIGNMENT AND LOAD BALANCING
    # ============================================================
    # Purpose:
    # Assign existing sewer pipes either to WWTP1 or WWTP2.
    #
    # Logic:
    # - Existing conduits are initially assumed to drain to WWTP1.
    # - The new network created in the northern municipalities drains to WWTP2.
    # - Stage A: all existing pipes located in the five northern municipalities
    #   are reassigned to WWTP2.
    # - Stage B: if WWTP2 is still below 50% of total urban population, additional
    #   existing pipes are reassigned to WWTP2 progressively from the northern
    #   boundary outward.
    #
    # Demand association:
    # - The algorithm estimates the population associated with each existing pipe
    #   by buffering the full pipe geometry and capturing all urban demand road
    #   nodes within that buffer. Each demand node is assigned to its nearest pipe.
    #
    # Orphan node fix (14.11b):
    # - After pipe-based assignment, some WWTP1 nodes may have no pipe association
    #   at all (no pipe within PIPE_SERVICE_BUFFER_M). These are reassigned to
    #   the nearest WWTP2 pipe using unbounded nearest-neighbour lookup, but only
    #   if the nearest WWTP2 pipe is closer than the nearest WWTP1 pipe.
    #   This prevents distant isolated nodes from being incorrectly switched.
    #
    # Output:
    # - pipe_assignment_p:                  all existing pipes with final WWTP assignment
    # - selected_existing_pipes_to_WWTP2_p_h: existing pipes reassigned to WWTP2
    # - remaining_existing_pipes_WWTP1_p_h:   existing pipes remaining assigned to WWTP1
    # - balanced_load_nodes_p_h:              demand nodes with final assignment
    # - transfer_summary, balance_stage_summary, pipe_assignment_summary
    # - GPKG and CSV outputs

    # ------------------------------------------------------------
    # 14.0 Input checks
    # ------------------------------------------------------------

    # (var checks omitted — guaranteed by heuristic loop context)
    required_node_cols = [
        "node_id",
        "municipality",
        "assigned_population",
        "wastewater_m3_day",
        "is_urban_demand_node",
        "assigned_to_existing_WWTP1",
        "assigned_to_new_WWTP2",
        "geometry",
    ]

    for c in required_node_cols:
        if c not in final_load_nodes_p_h.columns:
            raise RuntimeError(f"Column '{c}' missing from final_load_nodes_p_h.")

    required_pipe_cols = [
        "existing_conduit_id",
        "existing_conduit_name",
        "FromNode",
        "ToNode",
        "road_node_from",
        "road_node_to",
        "geometry",
    ]

    for c in required_pipe_cols:
        if c not in conduits_road_helper_p.columns:
            raise RuntimeError(f"Column '{c}' missing from conduits_road_helper_p. Run Block 13 first.")


    # ------------------------------------------------------------
    # 14.1 Parameters
    # ------------------------------------------------------------

    WWTP2_LATLON = (34.978087, 33.640878)
    WWTP2_LONLAT = (WWTP2_LATLON[1], WWTP2_LATLON[0])

    NORTH_MUNICIPALITIES_TARGET = [
        "Aradippou",
        "Oroklini",
        "Kellia",
        "Pyla",
        "Livadia",
    ]

    TARGET_WWTP2_POPULATION_SHARE = 0.40

    # Buffer along full pipe geometry for demand node association.
    # Increase if too few nodes are associated.
    PIPE_SERVICE_BUFFER_M = 100.0

    # Buffer used to classify pipes as belonging to northern municipalities.
    NORTH_PIPE_BUFFER_M = 50.0

    # Minimum pipe-associated population to be considered in Stage B.
    MIN_STAGE_B_PIPE_POPULATION = 0.0

    # Orphan node fix (14.11b):
    # An unassociated WWTP1 node is switched to WWTP2 only if the nearest
    # WWTP2 pipe is closer than the nearest WWTP1 pipe by at least this ratio.
    # 1.0 means switch if nearest WWTP2 pipe is equally close or closer.
    # 1.2 means switch only if nearest WWTP2 pipe is at least 20% closer.
    ORPHAN_WWTP2_PROXIMITY_RATIO = 1.0

    # Maximum absolute distance to nearest WWTP2 pipe for an orphan node to be switched.
    # Prevents distant isolated nodes in deep south from being incorrectly reassigned.
    ORPHAN_MAX_DIST_TO_WWTP2_PIPE_M = 500.0


    # ------------------------------------------------------------
    # 14.2 Helper functions
    # ------------------------------------------------------------

    def share_percent(value, total):
        if total <= 0:
            return 0.0
        return value / total * 100.0


    def get_road_geom(node_id):
        row = road_nodes_p.loc[road_nodes_p["node_id"] == int(node_id), "geometry"]
        if len(row) == 0:
            return None
        return row.iloc[0]


    # ------------------------------------------------------------
    # 14.3 Prepare main geometries
    # ------------------------------------------------------------

    north_boundary_p = muni_boundaries_p[
        muni_boundaries_p["municipality"].isin(NORTH_MUNICIPALITIES_TARGET)
    ].copy()

    if len(north_boundary_p) == 0:
        raise RuntimeError("No northern municipality boundaries found. Check municipality names.")

    north_union_p = north_boundary_p.geometry.union_all()
    north_union_buffered_p = north_union_p.buffer(NORTH_PIPE_BUFFER_M)
    north_boundary_line_p = north_union_p.boundary

    wwtp2_point_p = gpd.GeoSeries(
        [Point(WWTP2_LONLAT[0], WWTP2_LONLAT[1])],
        crs=CRS_WGS84
    ).to_crs(CRS_PROJECTED).iloc[0]

    print("Northern municipalities assigned first to WWTP2:")
    print(NORTH_MUNICIPALITIES_TARGET)
    print(f"Northern area: {north_union_p.area / 1e6:.2f} km²")
    print(f"North pipe buffer: {NORTH_PIPE_BUFFER_M:.1f} m")


    # ------------------------------------------------------------
    # 14.4 Prepare demand nodes
    # ------------------------------------------------------------

    balanced_load_nodes_p_h = final_load_nodes_p_h.copy()
    balanced_load_nodes_p_h = balanced_load_nodes_p_h.to_crs(CRS_PROJECTED)

    balanced_load_nodes_p_h["load_eligible"] = (
        balanced_load_nodes_p_h["is_urban_demand_node"].fillna(False).astype(bool)
    )

    balanced_load_nodes_p_h["initial_WWTP"] = "UNASSIGNED"

    balanced_load_nodes_p_h.loc[
        balanced_load_nodes_p_h["assigned_to_existing_WWTP1"] & balanced_load_nodes_p_h["load_eligible"],
        "initial_WWTP"
    ] = "WWTP1"

    balanced_load_nodes_p_h.loc[
        balanced_load_nodes_p_h["assigned_to_new_WWTP2"] & balanced_load_nodes_p_h["load_eligible"],
        "initial_WWTP"
    ] = "WWTP2"

    initial_wwtp1_mask = balanced_load_nodes_p_h["initial_WWTP"] == "WWTP1"
    initial_wwtp2_mask = balanced_load_nodes_p_h["initial_WWTP"] == "WWTP2"

    P1_initial = float(balanced_load_nodes_p_h.loc[initial_wwtp1_mask, "assigned_population"].sum())
    P2_initial = float(balanced_load_nodes_p_h.loc[initial_wwtp2_mask, "assigned_population"].sum())
    Q1_initial = float(balanced_load_nodes_p_h.loc[initial_wwtp1_mask, "wastewater_m3_day"].sum())
    Q2_initial = float(balanced_load_nodes_p_h.loc[initial_wwtp2_mask, "wastewater_m3_day"].sum())

    P_total = P1_initial + P2_initial
    P2_target = TARGET_WWTP2_POPULATION_SHARE * P_total

    initial_stats = pd.DataFrame([{
        "stage": "Initial",
        "WWTP1_population": P1_initial,
        "WWTP2_population": P2_initial,
        "WWTP1_share_percent": share_percent(P1_initial, P_total),
        "WWTP2_share_percent": share_percent(P2_initial, P_total),
        "WWTP2_target_population": P2_target,
        "WWTP2_population_gap_to_50_percent": max(0.0, P2_target - P2_initial),
        "WWTP1_wastewater_m3_day": Q1_initial,
        "WWTP2_wastewater_m3_day": Q2_initial,
        "wastewater_imbalance_m3_day": abs(Q1_initial - Q2_initial),
    }])

    print("\n=== Initial load balance ===")
    display(initial_stats)


    # ------------------------------------------------------------
    # 14.5 Build sewer endpoint-road-node layer (kept for GPKG export)
    # ------------------------------------------------------------

    endpoint_rows = []

    for _, r in conduits_road_helper_p.iterrows():
        conduit_id = int(r["existing_conduit_id"])
        conduit_name = str(r.get("existing_conduit_name", conduit_id))

        if pd.notna(r["road_node_from"]):
            rn = int(r["road_node_from"])
            geom = get_road_geom(rn)
            if geom is not None:
                endpoint_rows.append({
                    "existing_conduit_id": conduit_id,
                    "existing_conduit_name": conduit_name,
                    "endpoint_type": "from",
                    "sewer_node_name": str(r.get("FromNode", "")),
                    "endpoint_road_node_id": rn,
                    "geometry": geom,
                })

        if pd.notna(r["road_node_to"]):
            rn = int(r["road_node_to"])
            geom = get_road_geom(rn)
            if geom is not None:
                endpoint_rows.append({
                    "existing_conduit_id": conduit_id,
                    "existing_conduit_name": conduit_name,
                    "endpoint_type": "to",
                    "sewer_node_name": str(r.get("ToNode", "")),
                    "endpoint_road_node_id": rn,
                    "geometry": geom,
                })

    conduit_endpoint_road_nodes_p = gpd.GeoDataFrame(
        endpoint_rows,
        geometry="geometry",
        crs=CRS_PROJECTED
    ).drop_duplicates(
        ["existing_conduit_id", "endpoint_type", "endpoint_road_node_id"]
    ).copy()

    print("\nConduit endpoint-road-node layer:")
    print(f"Endpoint rows: {len(conduit_endpoint_road_nodes_p):,}")
    print(f"Unique conduits represented: {conduit_endpoint_road_nodes_p['existing_conduit_id'].nunique():,}")


    # ------------------------------------------------------------
    # 14.6 Associate WWTP1 urban demand nodes to existing pipes
    # via buffer along full pipe geometry
    # ------------------------------------------------------------

    pipe_buffers_p = conduits_road_helper_p[
        ["existing_conduit_id", "existing_conduit_name", "geometry"]
    ].copy()
    pipe_buffers_p["geometry"] = pipe_buffers_p.geometry.buffer(PIPE_SERVICE_BUFFER_M)
    pipe_buffers_p = gpd.GeoDataFrame(pipe_buffers_p, geometry="geometry", crs=CRS_PROJECTED)

    existing_demand_nodes_p = balanced_load_nodes_p_h[
        balanced_load_nodes_p_h["load_eligible"]
        & (balanced_load_nodes_p_h["initial_WWTP"] == "WWTP1")
    ].copy()
    existing_demand_nodes_p = gpd.GeoDataFrame(
        existing_demand_nodes_p, geometry="geometry", crs=CRS_PROJECTED
    )

    node_pipe_join_p = gpd.sjoin(
        existing_demand_nodes_p,
        pipe_buffers_p,
        how="left",
        predicate="within"
    )

    node_pipe_join_p = node_pipe_join_p.drop(columns=["index_right"], errors="ignore")

    pipe_geom_lookup = dict(zip(
        conduits_road_helper_p["existing_conduit_id"].astype(int),
        conduits_road_helper_p.geometry
    ))

    node_pipe_join_p["dist_to_pipe"] = node_pipe_join_p.apply(
        lambda r: r.geometry.distance(pipe_geom_lookup[int(r["existing_conduit_id"])])
        if pd.notna(r["existing_conduit_id"]) else np.inf,
        axis=1
    )

    node_pipe_join_p = (
        node_pipe_join_p
        .sort_values("dist_to_pipe")
        .drop_duplicates("node_id", keep="first")
    )

    node_pipe_join_p["has_pipe_association"] = node_pipe_join_p["existing_conduit_id"].notna()
    associated_node_pipe_p = node_pipe_join_p[node_pipe_join_p["has_pipe_association"]].copy()
    associated_node_pipe_p["existing_conduit_id"] = associated_node_pipe_p["existing_conduit_id"].astype(int)

    print("\n=== Demand-to-pipe association ===")
    print(f"WWTP1 urban demand nodes: {len(existing_demand_nodes_p):,}")
    print(f"Associated to existing pipes: {len(associated_node_pipe_p):,}")
    print(f"Associated population: {associated_node_pipe_p['assigned_population'].sum():,.1f}")
    print(f"Pipe service buffer: {PIPE_SERVICE_BUFFER_M:.0f} m")

    demand_pipe_diagnostics = pd.DataFrame([{
        "WWTP1_urban_nodes": len(existing_demand_nodes_p),
        "WWTP1_urban_population": float(existing_demand_nodes_p["assigned_population"].sum()),
        "associated_nodes": len(associated_node_pipe_p),
        "associated_population": float(associated_node_pipe_p["assigned_population"].sum()),
        "unassociated_nodes": len(existing_demand_nodes_p) - len(associated_node_pipe_p),
        "unassociated_population": float(
            existing_demand_nodes_p["assigned_population"].sum()
            - associated_node_pipe_p["assigned_population"].sum()
        ),
        "pipe_service_buffer_m": PIPE_SERVICE_BUFFER_M,
    }])

    display(demand_pipe_diagnostics)


    # ------------------------------------------------------------
    # 14.7 Aggregate associated demand to pipe level
    # ------------------------------------------------------------

    pipe_demand_stats = (
        associated_node_pipe_p
        .groupby(["existing_conduit_id", "existing_conduit_name"])
        .agg(
            associated_demand_nodes=("node_id", "count"),
            associated_population=("assigned_population", "sum"),
            associated_wastewater_m3_day=("wastewater_m3_day", "sum"),
            mean_node_to_pipe_distance_m=("dist_to_pipe", "mean"),
        )
        .reset_index()
    )

    pipe_assignment_p = conduits_road_helper_p.copy()

    pipe_assignment_p = pipe_assignment_p.merge(
        pipe_demand_stats,
        on=["existing_conduit_id", "existing_conduit_name"],
        how="left"
    )

    for c in [
        "associated_demand_nodes",
        "associated_population",
        "associated_wastewater_m3_day",
        "mean_node_to_pipe_distance_m",
    ]:
        pipe_assignment_p[c] = pipe_assignment_p[c].fillna(0.0)


    # ------------------------------------------------------------
    # 14.8 Stage A pipe reassignment:
    # all existing pipes in northern municipalities -> WWTP2
    # ------------------------------------------------------------

    road_node_geom_lookup = dict(
        zip(road_nodes_p["node_id"].astype(int), road_nodes_p.geometry)
    )

    def endpoint_in_north(row):
        geoms = []

        if pd.notna(row.get("road_node_from", None)):
            g = road_node_geom_lookup.get(int(row["road_node_from"]))
            if g is not None:
                geoms.append(g)

        if pd.notna(row.get("road_node_to", None)):
            g = road_node_geom_lookup.get(int(row["road_node_to"]))
            if g is not None:
                geoms.append(g)

        if len(geoms) == 0:
            return False

        return any(g.within(north_union_buffered_p) or g.intersects(north_union_buffered_p) for g in geoms)


    pipe_assignment_p["intersects_north_area"] = pipe_assignment_p.geometry.intersects(north_union_p)
    pipe_assignment_p["intersects_north_buffer"] = pipe_assignment_p.geometry.intersects(north_union_buffered_p)
    pipe_assignment_p["centroid_in_north_area"] = pipe_assignment_p.geometry.centroid.within(north_union_p)
    pipe_assignment_p["endpoint_in_north_area"] = pipe_assignment_p.apply(endpoint_in_north, axis=1)

    pipe_assignment_p["stage_A_pipe_to_WWTP2"] = (
        pipe_assignment_p["intersects_north_area"]
        | pipe_assignment_p["intersects_north_buffer"]
        | pipe_assignment_p["centroid_in_north_area"]
        | pipe_assignment_p["endpoint_in_north_area"]
    )

    P_stage_A_pipe_transfer = float(
        pipe_assignment_p.loc[pipe_assignment_p["stage_A_pipe_to_WWTP2"], "associated_population"].sum()
    )
    Q_stage_A_pipe_transfer = float(
        pipe_assignment_p.loc[pipe_assignment_p["stage_A_pipe_to_WWTP2"], "associated_wastewater_m3_day"].sum()
    )

    P1_after_A = P1_initial - P_stage_A_pipe_transfer
    P2_after_A = P2_initial + P_stage_A_pipe_transfer
    Q1_after_A = Q1_initial - Q_stage_A_pipe_transfer
    Q2_after_A = Q2_initial + Q_stage_A_pipe_transfer

    stage_A_summary = pd.DataFrame([{
        "stage": "Stage A: existing pipes in northern municipalities reassigned to WWTP2",
        "n_pipes_reassigned": int(pipe_assignment_p["stage_A_pipe_to_WWTP2"].sum()),
        "population_reassigned_this_stage": P_stage_A_pipe_transfer,
        "wastewater_reassigned_this_stage_m3_day": Q_stage_A_pipe_transfer,
        "WWTP1_population": P1_after_A,
        "WWTP2_population": P2_after_A,
        "WWTP1_share_percent": share_percent(P1_after_A, P_total),
        "WWTP2_share_percent": share_percent(P2_after_A, P_total),
        "WWTP2_population_gap_to_50_percent": max(0.0, P2_target - P2_after_A),
    }])

    print("\n=== Stage A pipe reassignment ===")
    display(stage_A_summary)

    print("Stage A pipe checks:")
    print(f"Intersects north area: {pipe_assignment_p['intersects_north_area'].sum():,}")
    print(f"Intersects north buffer: {pipe_assignment_p['intersects_north_buffer'].sum():,}")
    print(f"Centroid in north area: {pipe_assignment_p['centroid_in_north_area'].sum():,}")
    print(f"Endpoint in north area: {pipe_assignment_p['endpoint_in_north_area'].sum():,}")
    print(f"Final Stage A pipes to WWTP2: {pipe_assignment_p['stage_A_pipe_to_WWTP2'].sum():,}")


    # ------------------------------------------------------------
    # 14.9 Stage B:
    # progressively reassign remaining pipes closest to northern boundary
    # ------------------------------------------------------------

    pipe_assignment_p["distance_to_north_boundary_m"] = pipe_assignment_p.geometry.distance(
        north_boundary_line_p
    )

    pipe_assignment_p["distance_to_WWTP2_m"] = pipe_assignment_p.geometry.centroid.distance(
        wwtp2_point_p
    )

    stage_B_candidates_p = pipe_assignment_p[
        (~pipe_assignment_p["stage_A_pipe_to_WWTP2"])
        & (pipe_assignment_p["associated_population"] > MIN_STAGE_B_PIPE_POPULATION)
    ].copy()

    stage_B_candidates_p = stage_B_candidates_p.sort_values(
        ["distance_to_north_boundary_m", "distance_to_WWTP2_m", "associated_population"],
        ascending=[True, True, False]
    ).copy()

    print("\n=== Stage B pipe candidates ===")
    print(f"Candidate pipes: {len(stage_B_candidates_p):,}")
    print(f"Candidate associated population: {stage_B_candidates_p['associated_population'].sum():,.1f}")
    print(f"Population still needed after Stage A: {max(0.0, P2_target - P2_after_A):,.1f}")

    iteration_rows = []

    iteration_rows.append({
        "iteration": 0,
        "stage": "After Stage A",
        "selected_pipe_id": None,
        "selected_pipe_name": None,
        "population_reassigned_this_iteration": 0.0,
        "cumulative_stage_B_population": 0.0,
        "WWTP1_population": P1_after_A,
        "WWTP2_population": P2_after_A,
        "WWTP1_share_percent": share_percent(P1_after_A, P_total),
        "WWTP2_share_percent": share_percent(P2_after_A, P_total),
        "distance_to_north_boundary_m": 0.0,
    })

    selected_stage_B_pipe_ids = []
    cumulative_stage_B_pop = 0.0
    cumulative_stage_B_q = 0.0

    P1_current = P1_after_A
    P2_current = P2_after_A
    Q1_current = Q1_after_A
    Q2_current = Q2_after_A

    for _, pipe in stage_B_candidates_p.iterrows():
        if P2_current >= P2_target:
            break

        pipe_pop = float(pipe["associated_population"])
        pipe_q = float(pipe["associated_wastewater_m3_day"])

        if pipe_pop <= 0:
            continue

        selected_stage_B_pipe_ids.append(int(pipe["existing_conduit_id"]))

        cumulative_stage_B_pop += pipe_pop
        cumulative_stage_B_q += pipe_q

        P1_current -= pipe_pop
        P2_current += pipe_pop
        Q1_current -= pipe_q
        Q2_current += pipe_q

        iteration_rows.append({
            "iteration": len(iteration_rows),
            "stage": "Stage B boundary expansion",
            "selected_pipe_id": int(pipe["existing_conduit_id"]),
            "selected_pipe_name": pipe["existing_conduit_name"],
            "population_reassigned_this_iteration": pipe_pop,
            "cumulative_stage_B_population": cumulative_stage_B_pop,
            "WWTP1_population": P1_current,
            "WWTP2_population": P2_current,
            "WWTP1_share_percent": share_percent(P1_current, P_total),
            "WWTP2_share_percent": share_percent(P2_current, P_total),
            "distance_to_north_boundary_m": float(pipe["distance_to_north_boundary_m"]),
        })

    stage_B_iterations_df = pd.DataFrame(iteration_rows)

    pipe_assignment_p["stage_B_pipe_to_WWTP2"] = (
        pipe_assignment_p["existing_conduit_id"].isin(selected_stage_B_pipe_ids)
    )

    print("\n=== Stage B iteration summary preview ===")
    display(stage_B_iterations_df.tail(20))


    # ------------------------------------------------------------
    # 14.10 Final pipe assignment
    # ------------------------------------------------------------

    pipe_assignment_p["pipe_final_assignment"] = "WWTP1_existing"
    pipe_assignment_p["pipe_reassignment_reason"] = "WWTP1_existing"

    pipe_assignment_p.loc[
        pipe_assignment_p["stage_A_pipe_to_WWTP2"], "pipe_final_assignment"
    ] = "WWTP2_reassigned"

    pipe_assignment_p.loc[
        pipe_assignment_p["stage_A_pipe_to_WWTP2"], "pipe_reassignment_reason"
    ] = "Stage_A_northern_pipes"

    pipe_assignment_p.loc[
        pipe_assignment_p["stage_B_pipe_to_WWTP2"], "pipe_final_assignment"
    ] = "WWTP2_reassigned"

    pipe_assignment_p.loc[
        pipe_assignment_p["stage_B_pipe_to_WWTP2"], "pipe_reassignment_reason"
    ] = "Stage_B_boundary_expansion"

    pipe_assignment_p.loc[
        pipe_assignment_p["stage_A_pipe_to_WWTP2"] & pipe_assignment_p["stage_B_pipe_to_WWTP2"],
        "pipe_reassignment_reason"
    ] = "Stage_A_and_Stage_B"

    selected_existing_pipes_to_WWTP2_p_h = pipe_assignment_p[
        pipe_assignment_p["pipe_final_assignment"] == "WWTP2_reassigned"
    ].copy()

    remaining_existing_pipes_WWTP1_p_h = pipe_assignment_p[
        pipe_assignment_p["pipe_final_assignment"] == "WWTP1_existing"
    ].copy()

    P_stage_B_pipe_transfer = float(
        pipe_assignment_p.loc[pipe_assignment_p["stage_B_pipe_to_WWTP2"], "associated_population"].sum()
    )
    Q_stage_B_pipe_transfer = float(
        pipe_assignment_p.loc[pipe_assignment_p["stage_B_pipe_to_WWTP2"], "associated_wastewater_m3_day"].sum()
    )

    P_pipe_transfer_total = P_stage_A_pipe_transfer + P_stage_B_pipe_transfer
    Q_pipe_transfer_total = Q_stage_A_pipe_transfer + Q_stage_B_pipe_transfer

    P1_final = P1_initial - P_pipe_transfer_total
    P2_final = P2_initial + P_pipe_transfer_total
    Q1_final = Q1_initial - Q_pipe_transfer_total
    Q2_final = Q2_initial + Q_pipe_transfer_total


    # ------------------------------------------------------------
    # 14.11 Final node assignment based on associated pipe assignment
    # ------------------------------------------------------------

    pipe_id_to_assignment = dict(
        zip(
            pipe_assignment_p["existing_conduit_id"].astype(int),
            pipe_assignment_p["pipe_final_assignment"]
        )
    )

    associated_node_pipe_p["associated_pipe_assignment"] = (
        associated_node_pipe_p["existing_conduit_id"].astype(int).map(pipe_id_to_assignment)
    )

    wwtp2_pipe_node_ids = set(
        associated_node_pipe_p.loc[
            associated_node_pipe_p["associated_pipe_assignment"] == "WWTP2_reassigned",
            "node_id"
        ].astype(int).tolist()
    )

    balanced_load_nodes_p_h["associated_existing_pipe_to_WWTP2"] = (
        balanced_load_nodes_p_h["node_id"].astype(int).isin(wwtp2_pipe_node_ids)
    )

    balanced_load_nodes_p_h["assigned_to_new_WWTP2_balanced"] = (
        balanced_load_nodes_p_h["assigned_to_new_WWTP2"]
        | balanced_load_nodes_p_h["associated_existing_pipe_to_WWTP2"]
    )

    balanced_load_nodes_p_h["assigned_to_existing_WWTP1_balanced"] = (
        balanced_load_nodes_p_h["assigned_to_existing_WWTP1"]
        & (~balanced_load_nodes_p_h["associated_existing_pipe_to_WWTP2"])
    )

    balanced_load_nodes_p_h["stage_A_pipe_transfer_node"] = False
    balanced_load_nodes_p_h["stage_B_pipe_transfer_node"] = False

    stage_A_pipe_ids = set(
        pipe_assignment_p.loc[
            pipe_assignment_p["stage_A_pipe_to_WWTP2"], "existing_conduit_id"
        ].astype(int).tolist()
    )

    stage_B_pipe_ids = set(
        pipe_assignment_p.loc[
            pipe_assignment_p["stage_B_pipe_to_WWTP2"], "existing_conduit_id"
        ].astype(int).tolist()
    )

    stage_A_node_ids = set(
        associated_node_pipe_p.loc[
            associated_node_pipe_p["existing_conduit_id"].astype(int).isin(stage_A_pipe_ids),
            "node_id"
        ].astype(int).tolist()
    )

    stage_B_node_ids = set(
        associated_node_pipe_p.loc[
            associated_node_pipe_p["existing_conduit_id"].astype(int).isin(stage_B_pipe_ids),
            "node_id"
        ].astype(int).tolist()
    )

    balanced_load_nodes_p_h.loc[
        balanced_load_nodes_p_h["node_id"].astype(int).isin(stage_A_node_ids),
        "stage_A_pipe_transfer_node"
    ] = True

    balanced_load_nodes_p_h.loc[
        balanced_load_nodes_p_h["node_id"].astype(int).isin(stage_B_node_ids),
        "stage_B_pipe_transfer_node"
    ] = True


    # ------------------------------------------------------------
    # 14.11b Orphan node fix:
    # WWTP1 nodes with no pipe association are reassigned to WWTP2
    # if their nearest WWTP2 pipe is closer than their nearest WWTP1 pipe.
    #
    # This handles groups of nodes in areas where the pipe buffer did
    # not reach them (e.g. sparse pipe network, wide streets).
    # Works on whole groups because every node independently finds
    # its own nearest pipe -- no reliance on neighbours being WWTP2.
    # ------------------------------------------------------------

    # Nodes that have no pipe association at all
    associated_node_ids = set(associated_node_pipe_p["node_id"].astype(int).tolist())

    orphan_mask = (
        balanced_load_nodes_p_h["load_eligible"]
        & (balanced_load_nodes_p_h["initial_WWTP"] == "WWTP1")
        & (~balanced_load_nodes_p_h["node_id"].astype(int).isin(associated_node_ids))
    )

    orphan_nodes_p = balanced_load_nodes_p_h[orphan_mask].copy()

    print(f"\n=== Orphan node fix (14.11b) ===")
    print(f"WWTP1 urban nodes with no pipe association: {len(orphan_nodes_p):,}")
    print(f"Population: {orphan_nodes_p['assigned_population'].sum():,.1f}")

    orphan_wwtp2_node_ids = set()

    if len(orphan_nodes_p) > 0:

        wwtp2_pipes = pipe_assignment_p[
            pipe_assignment_p["pipe_final_assignment"] == "WWTP2_reassigned"
        ].copy()

        wwtp1_pipes = pipe_assignment_p[
            pipe_assignment_p["pipe_final_assignment"] == "WWTP1_existing"
        ].copy()

        if len(wwtp2_pipes) == 0:
            print("No WWTP2 pipes available for orphan reassignment. Skipping.")
        else:
            # Build arrays of pipe geometries for vectorised distance calculation
            wwtp2_pipe_geoms = list(wwtp2_pipes.geometry)
            wwtp2_pipe_ids = list(wwtp2_pipes["existing_conduit_id"].astype(int))

            wwtp1_pipe_geoms = list(wwtp1_pipes.geometry)

            switched = 0
            switched_pop = 0.0
            orphan_pipe_rows = []

            for _, node in orphan_nodes_p.iterrows():
                node_geom = node.geometry

                # Distance to nearest WWTP2 pipe
                dist_to_wwtp2 = min(node_geom.distance(pg) for pg in wwtp2_pipe_geoms)

                # Distance to nearest WWTP1 pipe (if any)
                if len(wwtp1_pipe_geoms) > 0:
                    dist_to_wwtp1 = min(node_geom.distance(pg) for pg in wwtp1_pipe_geoms)
                else:
                    dist_to_wwtp1 = np.inf

                # Switch only if WWTP2 pipe is closer (by ratio threshold)
                if dist_to_wwtp2 <= dist_to_wwtp1 * ORPHAN_WWTP2_PROXIMITY_RATIO and dist_to_wwtp2 <= ORPHAN_MAX_DIST_TO_WWTP2_PIPE_M:
                    nid = int(node["node_id"])
                    orphan_wwtp2_node_ids.add(nid)
                    switched += 1
                    switched_pop += float(node["assigned_population"])

                    # Find which WWTP2 pipe is nearest for diagnostics
                    nearest_idx = int(np.argmin([node_geom.distance(pg) for pg in wwtp2_pipe_geoms]))
                    orphan_pipe_rows.append({
                        "node_id": nid,
                        "municipality": node["municipality"],
                        "assigned_population": float(node["assigned_population"]),
                        "dist_to_nearest_wwtp2_pipe_m": dist_to_wwtp2,
                        "dist_to_nearest_wwtp1_pipe_m": dist_to_wwtp1,
                        "nearest_wwtp2_pipe_id": wwtp2_pipe_ids[nearest_idx],
                    })

            print(f"Orphan nodes switched to WWTP2: {switched:,}")
            print(f"Population switched: {switched_pop:,.1f}")

            if len(orphan_pipe_rows) > 0:
                orphan_diagnostics_df = pd.DataFrame(orphan_pipe_rows)
                print("\nOrphan reassignment diagnostics (first 20 rows):")
                display(orphan_diagnostics_df.head(20))

        # Apply orphan fix to balanced_load_nodes_p_h
        balanced_load_nodes_p_h["orphan_node_switched_to_WWTP2"] = (
            balanced_load_nodes_p_h["node_id"].astype(int).isin(orphan_wwtp2_node_ids)
        )

        balanced_load_nodes_p_h.loc[
            balanced_load_nodes_p_h["orphan_node_switched_to_WWTP2"],
            "assigned_to_new_WWTP2_balanced"
        ] = True

        balanced_load_nodes_p_h.loc[
            balanced_load_nodes_p_h["orphan_node_switched_to_WWTP2"],
            "assigned_to_existing_WWTP1_balanced"
        ] = False

        # Update population counters
        orphan_pop_switched = float(
            balanced_load_nodes_p_h.loc[
                balanced_load_nodes_p_h["orphan_node_switched_to_WWTP2"],
                "assigned_population"
            ].sum()
        )
        orphan_q_switched = float(
            balanced_load_nodes_p_h.loc[
                balanced_load_nodes_p_h["orphan_node_switched_to_WWTP2"],
                "wastewater_m3_day"
            ].sum()
        )

        P1_final -= orphan_pop_switched
        P2_final += orphan_pop_switched
        Q1_final -= orphan_q_switched
        Q2_final += orphan_q_switched

        print(f"\nFinal WWTP2 population after orphan fix: {P2_final:,.1f} ({share_percent(P2_final, P_total):.1f}%)")

    else:
        balanced_load_nodes_p_h["orphan_node_switched_to_WWTP2"] = False
        print("No orphan nodes found.")


    # ------------------------------------------------------------
    # 14.12 Summaries
    # ------------------------------------------------------------

    balance_stage_summary = pd.DataFrame([
        {
            "stage": "Initial",
            "WWTP1_population": P1_initial,
            "WWTP2_population": P2_initial,
            "WWTP1_share_percent": share_percent(P1_initial, P_total),
            "WWTP2_share_percent": share_percent(P2_initial, P_total),
            "WWTP2_population_gap_to_50_percent": max(0.0, P2_target - P2_initial),
            "WWTP1_wastewater_m3_day": Q1_initial,
            "WWTP2_wastewater_m3_day": Q2_initial,
            "wastewater_imbalance_m3_day": abs(Q1_initial - Q2_initial),
        },
        {
            "stage": "After Stage A northern existing pipes",
            "WWTP1_population": P1_after_A,
            "WWTP2_population": P2_after_A,
            "WWTP1_share_percent": share_percent(P1_after_A, P_total),
            "WWTP2_share_percent": share_percent(P2_after_A, P_total),
            "WWTP2_population_gap_to_50_percent": max(0.0, P2_target - P2_after_A),
            "WWTP1_wastewater_m3_day": Q1_after_A,
            "WWTP2_wastewater_m3_day": Q2_after_A,
            "wastewater_imbalance_m3_day": abs(Q1_after_A - Q2_after_A),
        },
        {
            "stage": "After Stage B boundary expansion pipes",
            "WWTP1_population": P1_final - orphan_pop_switched if "orphan_pop_switched" in dir() else P1_final,
            "WWTP2_population": P2_final - orphan_pop_switched if "orphan_pop_switched" in dir() else P2_final,
            "WWTP1_share_percent": share_percent(P1_final - (orphan_pop_switched if "orphan_pop_switched" in dir() else 0), P_total),
            "WWTP2_share_percent": share_percent(P2_final - (orphan_pop_switched if "orphan_pop_switched" in dir() else 0), P_total),
            "WWTP2_population_gap_to_50_percent": max(0.0, P2_target - (P2_final - (orphan_pop_switched if "orphan_pop_switched" in dir() else 0))),
            "WWTP1_wastewater_m3_day": Q1_final - (orphan_q_switched if "orphan_q_switched" in dir() else 0),
            "WWTP2_wastewater_m3_day": Q2_final - (orphan_q_switched if "orphan_q_switched" in dir() else 0),
            "wastewater_imbalance_m3_day": abs((Q1_final - (orphan_q_switched if "orphan_q_switched" in dir() else 0)) - (Q2_final - (orphan_q_switched if "orphan_q_switched" in dir() else 0))),
        },
        {
            "stage": "After orphan node fix (14.11b)",
            "WWTP1_population": P1_final,
            "WWTP2_population": P2_final,
            "WWTP1_share_percent": share_percent(P1_final, P_total),
            "WWTP2_share_percent": share_percent(P2_final, P_total),
            "WWTP2_population_gap_to_50_percent": max(0.0, P2_target - P2_final),
            "WWTP1_wastewater_m3_day": Q1_final,
            "WWTP2_wastewater_m3_day": Q2_final,
            "wastewater_imbalance_m3_day": abs(Q1_final - Q2_final),
        },
    ])

    pipe_assignment_summary = (
        pipe_assignment_p
        .groupby(["pipe_final_assignment", "pipe_reassignment_reason"])
        .agg(
            n_pipes=("existing_conduit_id", "count"),
            associated_population=("associated_population", "sum"),
            associated_wastewater_m3_day=("associated_wastewater_m3_day", "sum"),
        )
        .reset_index()
    )

    transfer_summary = pd.DataFrame([{
        "target_WWTP2_population_share": TARGET_WWTP2_POPULATION_SHARE,
        "target_WWTP2_population": P2_target,
        "stage_A_pipe_transfer_population": P_stage_A_pipe_transfer,
        "stage_B_pipe_transfer_population": P_stage_B_pipe_transfer,
        "orphan_node_transfer_population": float(orphan_pop_switched) if "orphan_pop_switched" in dir() else 0.0,
        "total_pipe_transfer_population": P_pipe_transfer_total,
        "stage_A_pipe_transfer_wastewater_m3_day": Q_stage_A_pipe_transfer,
        "stage_B_pipe_transfer_wastewater_m3_day": Q_stage_B_pipe_transfer,
        "n_stage_A_pipes_to_WWTP2": int(pipe_assignment_p["stage_A_pipe_to_WWTP2"].sum()),
        "n_stage_B_pipes_to_WWTP2": int(pipe_assignment_p["stage_B_pipe_to_WWTP2"].sum()),
        "n_orphan_nodes_switched": len(orphan_wwtp2_node_ids),
        "n_existing_pipes_assigned_to_WWTP2": len(selected_existing_pipes_to_WWTP2_p_h),
        "n_existing_pipes_remaining_WWTP1": len(remaining_existing_pipes_WWTP1_p_h),
        "WWTP1_population_before": P1_initial,
        "WWTP2_population_before": P2_initial,
        "WWTP1_population_after": P1_final,
        "WWTP2_population_after": P2_final,
        "WWTP1_share_before_percent": share_percent(P1_initial, P_total),
        "WWTP2_share_before_percent": share_percent(P2_initial, P_total),
        "WWTP1_share_after_percent": share_percent(P1_final, P_total),
        "WWTP2_share_after_percent": share_percent(P2_final, P_total),
        "total_population_before": P_total,
        "total_population_after": P1_final + P2_final,
        "population_difference_after_minus_before": (P1_final + P2_final) - P_total,
    }])

    print("\n=== Balance stage summary ===")
    display(balance_stage_summary)

    print("\n=== Pipe assignment summary ===")
    display(pipe_assignment_summary)

    print("\n=== Final transfer summary ===")
    display(transfer_summary)

    print("\nPopulation conservation check:")
    print(f"Before: {P_total:,.6f}")
    print(f"After:  {P1_final + P2_final:,.6f}")
    print(f"Diff:   {(P1_final + P2_final) - P_total:,.12f}")


    # ------------------------------------------------------------
    # 14.13 Save outputs
    # ------------------------------------------------------------

    out_gpkg = heuristic_dir / "Block14_pipe_based_WWTP_reassignment.gpkg"

    if out_gpkg.exists():
        out_gpkg.unlink()

    pipe_assignment_p.to_file(out_gpkg, layer="all_existing_pipes_assignment", driver="GPKG")
    selected_existing_pipes_to_WWTP2_p_h.to_file(out_gpkg, layer="existing_pipes_reassigned_to_WWTP2", driver="GPKG")
    remaining_existing_pipes_WWTP1_p_h.to_file(out_gpkg, layer="existing_pipes_remaining_WWTP1", driver="GPKG")
    balanced_load_nodes_p_h.to_file(out_gpkg, layer="balanced_demand_nodes", driver="GPKG")
    associated_node_pipe_p.to_file(out_gpkg, layer="demand_nodes_associated_to_existing_pipes", driver="GPKG")
    conduit_endpoint_road_nodes_p.to_file(out_gpkg, layer="conduit_endpoint_road_nodes", driver="GPKG")

    if "sewer_to_road_connectors_p" in globals():
        sewer_to_road_connectors_p.to_file(out_gpkg, layer="sewer_to_road_connectors", driver="GPKG")

    balance_stage_summary.to_csv(heuristic_dir / "Block14_balance_stage_summary.csv", index=False)
    pipe_assignment_summary.to_csv(heuristic_dir / "Block14_pipe_assignment_summary.csv", index=False)
    transfer_summary.to_csv(heuristic_dir / "Block14_transfer_summary.csv", index=False)
    stage_B_iterations_df.to_csv(heuristic_dir / "Block14_stage_B_pipe_iteration_summary.csv", index=False)
    pipe_assignment_p.drop(columns="geometry").to_csv(heuristic_dir / "Block14_all_existing_pipes_assignment.csv", index=False)
    selected_existing_pipes_to_WWTP2_p_h.drop(columns="geometry").to_csv(heuristic_dir / "Block14_existing_pipes_reassigned_to_WWTP2.csv", index=False)
    remaining_existing_pipes_WWTP1_p_h.drop(columns="geometry").to_csv(heuristic_dir / "Block14_existing_pipes_remaining_WWTP1.csv", index=False)

    print("\nSaved:")
    print(out_gpkg)
    print(heuristic_dir / "Block14_balance_stage_summary.csv")
    print(heuristic_dir / "Block14_pipe_assignment_summary.csv")
    print(heuristic_dir / "Block14_transfer_summary.csv")
    print(heuristic_dir / "Block14_stage_B_pipe_iteration_summary.csv")
    print(heuristic_dir / "Block14_existing_pipes_reassigned_to_WWTP2.csv")
    # ── BLOCK 25.5: Final WWTP pipe assignment map ────────────────
    print("\n" + "=" * 80)
    print(f"BLOCK 25.5 [{heuristic_name}] -- FINAL WWTP PIPE ASSIGNMENT MAP")
    print("=" * 80)

    # ============================================================
    # BLOCK 14.5 -- SIMPLE FINAL WWTP PIPE ASSIGNMENT MAP
    # ============================================================
    # Purpose:
    # Create a simple map with:
    # - Blue: existing pipes remaining assigned to WWTP1
    # - Orange: existing pipes reassigned to WWTP2
    # - Red: new network created for WWTP2
    # - Optional demand nodes and WWTP load labels

    if not HAS_FOLIUM:
        raise ImportError("folium is not installed. Install with: pip install folium")

    # (var checks omitted — guaranteed by heuristic loop context)
    # ------------------------------------------------------------
    # 14.5.1 Prepare layers
    # ------------------------------------------------------------

    muni_w = muni_boundaries_p.to_crs(CRS_WGS84)
    pipes_wwtp1_w = remaining_existing_pipes_WWTP1_p_h.to_crs(CRS_WGS84)
    pipes_wwtp2_w = selected_existing_pipes_to_WWTP2_p_h.to_crs(CRS_WGS84)
    new_pipes_w = new_pipes_p_h.to_crs(CRS_WGS84)
    nodes_w = balanced_load_nodes_p_h.to_crs(CRS_WGS84)

    nodes_wwtp1_w = nodes_w[
        nodes_w["assigned_to_existing_WWTP1_balanced"]
        & nodes_w["load_eligible"]
    ].copy()

    nodes_wwtp2_w = nodes_w[
        nodes_w["assigned_to_new_WWTP2_balanced"]
        & nodes_w["load_eligible"]
    ].copy()

    stage_A_nodes_w = nodes_w[
        nodes_w["stage_A_pipe_transfer_node"]
        & nodes_w["load_eligible"]
    ].copy()

    stage_B_nodes_w = nodes_w[
        nodes_w["stage_B_pipe_transfer_node"]
        & nodes_w["load_eligible"]
    ].copy()

    center = muni_w.geometry.union_all().centroid

    m_final = folium.Map(
        location=[center.y, center.x],
        zoom_start=11,
        tiles="OpenStreetMap"
    )

    Fullscreen().add_to(m_final)
    MiniMap(toggle_display=True).add_to(m_final)


    # ------------------------------------------------------------
    # 14.5.2 Boundaries
    # ------------------------------------------------------------

    fg_boundaries = folium.FeatureGroup(name="Municipal boundaries", show=True)

    for _, r in muni_w.iterrows():
        is_north = r["municipality"] in NORTH_MUNICIPALITIES_TARGET

        folium.GeoJson(
            r.geometry,
            tooltip=f"{r['municipality']} boundary",
            style_function=lambda x, is_north=is_north: {
                "fillColor": "#ffffff",
                "color": "#444444" if not is_north else "#D32F2F",
                "weight": 2 if is_north else 1,
                "fillOpacity": 0.00,
            }
        ).add_to(fg_boundaries)

    fg_boundaries.add_to(m_final)


    # ------------------------------------------------------------
    # 14.5.3 Existing pipes remaining WWTP1 -- blue
    # ------------------------------------------------------------

    fg_wwtp1 = folium.FeatureGroup(
        name="Existing pipes remaining assigned to WWTP1",
        show=True
    )

    for _, r in pipes_wwtp1_w.iterrows():
        popup = (
            f"Pipe ID: {r.get('existing_conduit_id', '')}<br>"
            f"Name: {r.get('existing_conduit_name', '')}<br>"
            f"Assignment: WWTP1<br>"
            f"Associated population: {r.get('associated_population', 0):,.1f}<br>"
            f"Associated wastewater: {r.get('associated_wastewater_m3_day', 0):,.2f} m³/day"
        )

        folium.GeoJson(
            r.geometry,
            tooltip="Existing pipe assigned to WWTP1",
            popup=popup,
            style_function=lambda x: {
                "color": "#1565C0",
                "weight": 3,
                "opacity": 0.75,
            }
        ).add_to(fg_wwtp1)

    fg_wwtp1.add_to(m_final)


    # ------------------------------------------------------------
    # 14.5.4 Existing pipes reassigned WWTP2 -- orange
    # ------------------------------------------------------------

    fg_wwtp2_existing = folium.FeatureGroup(
        name="Existing pipes reassigned to WWTP2",
        show=True
    )

    for _, r in pipes_wwtp2_w.iterrows():
        popup = (
            f"Pipe ID: {r.get('existing_conduit_id', '')}<br>"
            f"Name: {r.get('existing_conduit_name', '')}<br>"
            f"Assignment: WWTP2<br>"
            f"Reason: {r.get('pipe_reassignment_reason', '')}<br>"
            f"Associated population: {r.get('associated_population', 0):,.1f}<br>"
            f"Associated wastewater: {r.get('associated_wastewater_m3_day', 0):,.2f} m³/day"
        )

        folium.GeoJson(
            r.geometry,
            tooltip="Existing pipe reassigned to WWTP2",
            popup=popup,
            style_function=lambda x: {
                "color": "#F57C00",
                "weight": 4.5,
                "opacity": 0.95,
            }
        ).add_to(fg_wwtp2_existing)

    fg_wwtp2_existing.add_to(m_final)


    # ------------------------------------------------------------
    # 14.5.5 New network created for WWTP2 -- red
    # ------------------------------------------------------------

    fg_new = folium.FeatureGroup(
        name="New network created for WWTP2",
        show=True
    )

    for _, r in new_pipes_w.iterrows():
        popup = (
            f"New pipe<br>"
            f"Municipality: {r.get('municipality', '')}<br>"
            f"Length: {r.get('length_m', 0):,.1f} m<br>"
            f"Cost: €{r.get('cost_eur', 0):,.0f}"
        )

        folium.GeoJson(
            r.geometry,
            tooltip="New pipe created for WWTP2",
            popup=popup,
            style_function=lambda x: {
                "color": "#C62828",
                "weight": 4,
                "opacity": 0.95,
            }
        ).add_to(fg_new)

    fg_new.add_to(m_final)


    # ------------------------------------------------------------
    # 14.5.6 Optional demand node layers
    # ------------------------------------------------------------

    fg_nodes_A = folium.FeatureGroup(
        name="Demand nodes reassigned via Stage A pipes",
        show=False
    )

    for _, r in stage_A_nodes_w.iterrows():
        pt = r.geometry

        popup = (
            f"<b>{r['municipality']}</b><br>"
            f"Node ID: {r['node_id']}<br>"
            f"Reassigned through Stage A pipe<br>"
            f"Population: {r['assigned_population']:.2f}<br>"
            f"Wastewater: {r['wastewater_m3_day']:.3f} m³/day"
        )

        folium.CircleMarker(
            location=[pt.y, pt.x],
            radius=3.5,
            color="#F57C00",
            fill=True,
            fill_color="#F57C00",
            fill_opacity=0.75,
            weight=0.8,
            popup=popup,
        ).add_to(fg_nodes_A)

    fg_nodes_A.add_to(m_final)


    fg_nodes_B = folium.FeatureGroup(
        name="Demand nodes reassigned via Stage B pipes",
        show=False
    )

    for _, r in stage_B_nodes_w.iterrows():
        pt = r.geometry

        popup = (
            f"<b>{r['municipality']}</b><br>"
            f"Node ID: {r['node_id']}<br>"
            f"Reassigned through Stage B pipe<br>"
            f"Population: {r['assigned_population']:.2f}<br>"
            f"Wastewater: {r['wastewater_m3_day']:.3f} m³/day"
        )

        folium.CircleMarker(
            location=[pt.y, pt.x],
            radius=4.5,
            color="#FFB300",
            fill=True,
            fill_color="#FFB300",
            fill_opacity=0.90,
            weight=1,
            popup=popup,
        ).add_to(fg_nodes_B)

    fg_nodes_B.add_to(m_final)


    fg_nodes_wwtp1 = folium.FeatureGroup(
        name="Demand nodes finally assigned to WWTP1",
        show=False
    )

    for _, r in nodes_wwtp1_w.iterrows():
        pt = r.geometry

        folium.CircleMarker(
            location=[pt.y, pt.x],
            radius=2.5,
            color="#1565C0",
            fill=True,
            fill_color="#1565C0",
            fill_opacity=0.45,
            weight=0.5,
        ).add_to(fg_nodes_wwtp1)

    fg_nodes_wwtp1.add_to(m_final)


    fg_nodes_wwtp2 = folium.FeatureGroup(
        name="Demand nodes finally assigned to WWTP2",
        show=False
    )

    for _, r in nodes_wwtp2_w.iterrows():
        pt = r.geometry

        folium.CircleMarker(
            location=[pt.y, pt.x],
            radius=2.5,
            color="#F57C00",
            fill=True,
            fill_color="#F57C00",
            fill_opacity=0.45,
            weight=0.5,
        ).add_to(fg_nodes_wwtp2)

    fg_nodes_wwtp2.add_to(m_final)


    # ------------------------------------------------------------
    # 14.5.7 WWTP load labels
    # ------------------------------------------------------------

    fg_labels = folium.FeatureGroup(name="WWTP population balance labels", show=True)

    final_row = balance_stage_summary[
        balance_stage_summary["stage"] == "After Stage B boundary expansion pipes"
    ].iloc[0]

    initial_row = balance_stage_summary[
        balance_stage_summary["stage"] == "Initial"
    ].iloc[0]

    label_1 = (
        f"WWTP1 South<br>"
        f"Initial: {initial_row['WWTP1_population']:,.0f} "
        f"({initial_row['WWTP1_share_percent']:.1f}%)<br>"
        f"Final: {final_row['WWTP1_population']:,.0f} "
        f"({final_row['WWTP1_share_percent']:.1f}%)"
    )

    label_2 = (
        f"WWTP2 North<br>"
        f"Initial: {initial_row['WWTP2_population']:,.0f} "
        f"({initial_row['WWTP2_share_percent']:.1f}%)<br>"
        f"Final: {final_row['WWTP2_population']:,.0f} "
        f"({final_row['WWTP2_share_percent']:.1f}%)"
    )

    if len(nodes_wwtp1_w) > 0:
        c1 = nodes_wwtp1_w.geometry.union_all().centroid

        folium.Marker(
            location=[c1.y, c1.x],
            tooltip="WWTP1 population balance",
            popup=label_1,
            icon=folium.DivIcon(
                html=f"""
                <div style="
                    font-size: 10pt;
                    font-weight: bold;
                    color: #1565C0;
                    background-color: rgba(255,255,255,0.92);
                    border: 2px solid #1565C0;
                    border-radius: 6px;
                    padding: 5px 8px;
                    text-align: center;
                    white-space: nowrap;">
                    {label_1}
                </div>
                """
            )
        ).add_to(fg_labels)

    folium.Marker(
        location=[WWTP2_LATLON[0], WWTP2_LATLON[1]],
        tooltip="WWTP2 population balance",
        popup=label_2,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 10pt;
                font-weight: bold;
                color: #C62828;
                background-color: rgba(255,255,255,0.92);
                border: 2px solid #C62828;
                border-radius: 6px;
                padding: 5px 8px;
                text-align: center;
                white-space: nowrap;">
                {label_2}
            </div>
            """
        )
    ).add_to(fg_labels)

    folium.Marker(
        location=[WWTP2_LATLON[0], WWTP2_LATLON[1]],
        popup="Proposed WWTP2 North",
        icon=folium.Icon(color="red", icon="tint")
    ).add_to(fg_labels)

    fg_labels.add_to(m_final)


    # ------------------------------------------------------------
    # 14.5.8 Legend
    # ------------------------------------------------------------

    legend_html = """
    <div style="
        position: fixed;
        bottom: 35px;
        left: 35px;
        width: 360px;
        z-index: 9999;
        background-color: rgba(255,255,255,0.94);
        border: 2px solid #333333;
        border-radius: 6px;
        padding: 10px;
        font-size: 10pt;">
        <b>Final WWTP pipe assignment</b><br>
        <span style="color:#1565C0;font-weight:bold;">━━</span> Existing pipes remaining with WWTP1<br>
        <span style="color:#F57C00;font-weight:bold;">━━</span> Existing pipes reassigned to WWTP2<br>
        <span style="color:#C62828;font-weight:bold;">━━</span> New network created for WWTP2<br>
        <span style="color:#F57C00;">●</span> Demand nodes served by reassigned pipes<br>
    </div>
    """

    m_final.get_root().html.add_child(folium.Element(legend_html))

    folium.LayerControl(collapsed=False).add_to(m_final)

    out_html = heuristic_dir / "Step1_Block14_simple_final_WWTP_pipe_assignment_map.html"
    m_final.save(out_html)

    print("Saved simple final WWTP pipe assignment map:")
    print(out_html)
    # ── BLOCK 26: Statistics, cost summary, map and QGIS export ──
    print("\n" + "=" * 80)
    print(f"BLOCK 26 [{heuristic_name}] -- STATISTICS, COST SUMMARY, MAP AND QGIS EXPORT")
    print("=" * 80)

    # BLOCK 15 -- STATISTICS, COST SUMMARY, MAP AND QGIS EXPORT
    # ============================================================

    # (var checks omitted — guaranteed by heuristic loop context)
    print("=" * 80)
    print(f"BLOCK 15 [{heuristic_name}] -- STATISTICS, COST SUMMARY, MAP AND QGIS EXPORT")
    print("=" * 80)


    # ------------------------------------------------------------
    # 15.0 Helpers
    # ------------------------------------------------------------

    def _pct(part, total):
        if total is None or total == 0:
            return 0.0
        return 100.0 * float(part) / float(total)


    def _get_all_municipalities():
        """
        Full municipality list used in all tables.
        This ensures that municipalities with zero length or zero assigned population
        are still printed.
        """
        munis = []

        if "ALL_MUNICIPALITIES" in globals():
            munis.extend(list(ALL_MUNICIPALITIES))

        if "muni_boundaries_p" in globals() and "municipality" in muni_boundaries_p.columns:
            munis.extend(muni_boundaries_p["municipality"].dropna().astype(str).unique().tolist())

        if "final_load_nodes_p_h" in globals() and "municipality" in final_load_nodes_p_h.columns:
            munis.extend(final_load_nodes_p_h["municipality"].dropna().astype(str).unique().tolist())

        if "balanced_load_nodes_p_h" in globals() and "municipality" in balanced_load_nodes_p_h.columns:
            munis.extend(balanced_load_nodes_p_h["municipality"].dropna().astype(str).unique().tolist())

        # preserve order, remove duplicates
        out = []
        for m in munis:
            if m not in out:
                out.append(m)

        return out


    def _prepare_node_assignment_table(nodes_gdf, stage_label, wwtp1_col, wwtp2_col, assignment_col):
        """
        Build a clean node-level WWTP assignment table.

        This is intentionally node-derived, not summary-derived.
        It avoids discrepancies between printed statistics and the actual final
        node assignment flags.
        """
        nodes = nodes_gdf.copy()

        if nodes.crs is None:
            nodes = gpd.GeoDataFrame(nodes, geometry="geometry", crs=CRS_PROJECTED)
        else:
            nodes = nodes.to_crs(CRS_PROJECTED)

        required_cols = [
            "node_id",
            "municipality",
            "assigned_population",
            "wastewater_m3_day",
            wwtp1_col,
            wwtp2_col,
        ]

        for c in required_cols:
            if c not in nodes.columns:
                raise RuntimeError(f"Block 15: column '{c}' missing in {stage_label} nodes.")

        if "is_urban_demand_node" in nodes.columns:
            nodes["load_eligible"] = nodes["is_urban_demand_node"].fillna(False).astype(bool)
        elif "load_eligible" in nodes.columns:
            nodes["load_eligible"] = nodes["load_eligible"].fillna(False).astype(bool)
        else:
            raise RuntimeError(
                f"Block 15: neither 'is_urban_demand_node' nor 'load_eligible' found in {stage_label} nodes."
            )

        nodes[wwtp1_col] = nodes[wwtp1_col].fillna(False).astype(bool)
        nodes[wwtp2_col] = nodes[wwtp2_col].fillna(False).astype(bool)

        both_mask = nodes["load_eligible"] & nodes[wwtp1_col] & nodes[wwtp2_col]

        if bool(both_mask.any()):
            display(
                nodes.loc[
                    both_mask,
                    ["node_id", "municipality", "assigned_population", wwtp1_col, wwtp2_col]
                ].head(30)
            )
            raise RuntimeError(
                f"Block 15: some load-eligible nodes are assigned to both WWTP1 and WWTP2 in {stage_label}."
            )

        nodes[assignment_col] = "UNASSIGNED"

        nodes.loc[
            nodes["load_eligible"] & nodes[wwtp1_col],
            assignment_col
        ] = "WWTP_1_SOUTH"

        nodes.loc[
            nodes["load_eligible"] & nodes[wwtp2_col],
            assignment_col
        ] = "WWTP_2_NORTH"

        return nodes[nodes["load_eligible"]].copy()


    _ALL_MUNIS = _get_all_municipalities()
    _WWTP_LABELS = ["WWTP_1_SOUTH", "WWTP_2_NORTH"]

    # ============================================================
    # 15.1 Pumping / storage stations
    # ============================================================
    # Prefer the deduplicated storages.gpkg layer for visualisation.
    # Fallback to pumps.gpkg only if storages is not available.
    #
    # This layer is used only for maps and QGIS outputs.
    # It does NOT affect routing, Dijkstra, Prim-Steiner, WWTP assignment,
    # pipe length, cost, or demand-node allocation.

    pumping_stations_p = None

    if "storages" in local_gdfs:
        _ps = local_gdfs["storages"].copy()
        _source_layer = "storages"

    elif "pumps" in local_gdfs:
        _ps = local_gdfs["pumps"].copy()
        _source_layer = "pumps"

    else:
        _ps = None
        _source_layer = None

    if _ps is not None and len(_ps) > 0:
        if _ps.crs is None:
            _ps = _ps.set_crs(CRS_PROJECTED)

        pumping_stations_p = _ps.to_crs(CRS_PROJECTED)

        # Safety deduplication for visualisation:
        # if several records have the same Name, keep one point.
        if "Name" in pumping_stations_p.columns:
            pumping_stations_p = (
                pumping_stations_p
                .sort_values("Name")
                .drop_duplicates(subset=["Name"], keep="first")
                .copy()
            )

        for _col in ["Name", "name", "NAME", "ID", "id", "pump_id"]:
            if _col in pumping_stations_p.columns:
                pumping_stations_p["ps_name"] = pumping_stations_p[_col].astype(str)
                break
        else:
            pumping_stations_p["ps_name"] = [
                f"PS-{i + 1}" for i in range(len(pumping_stations_p))
            ]

        print(
            f"\n✓ Pumping/storage stations loaded from '{_source_layer}': "
            f"{len(pumping_stations_p)} features"
        )

    else:
        print("\n⚠ No storages/pumps layer found in local_gdfs — pumping-station layer skipped.")

    # ============================================================
    # § 1  IMPLEMENTATION COST
    #       New pipes split by every municipality polygon
    # ============================================================

    print("\n" + "─" * 80)
    print("§ 1  IMPLEMENTATION COST  (new pipes only)")
    print("     Costs split by municipality territory traversed")
    print("     All municipalities are shown, including zero-length municipalities")
    print("─" * 80)

    _np = new_pipes_p_h.copy()

    if _np.crs is None:
        _np = _np.set_crs(CRS_PROJECTED)
    else:
        _np = _np.to_crs(CRS_PROJECTED)

    _np = _np[_np.geometry.notna() & (~_np.geometry.is_empty)].copy()

    if "length_m" not in _np.columns:
        _np["length_m"] = _np.geometry.length
    else:
        _np["length_m"] = _np["length_m"].fillna(_np.geometry.length)

    if "cost_eur" not in _np.columns:
        _np["cost_eur"] = _np["length_m"] * PIPE_COST_EUR_PER_M
    else:
        _np["cost_eur"] = _np["cost_eur"].fillna(_np["length_m"] * PIPE_COST_EUR_PER_M)

    total_new_length_m = float(_np["length_m"].sum())
    total_new_length_km = total_new_length_m / 1000.0
    total_new_cost_eur = float(total_new_length_m * PIPE_COST_EUR_PER_M)

    print(f"\n  New pipe total length : {total_new_length_km:>10.2f} km")
    print(f"  Unit cost             : €{PIPE_COST_EUR_PER_M:>10.2f} / m")
    print(f"  Total implementation  : €{total_new_cost_eur:>14,.0f}")
    print(f"                          €{total_new_cost_eur / 1e6:>10.3f} M")

    _muni_poly = muni_boundaries_p[["municipality", "geometry"]].copy()

    if _muni_poly.crs is None:
        _muni_poly = _muni_poly.set_crs(CRS_PROJECTED)
    else:
        _muni_poly = _muni_poly.to_crs(CRS_PROJECTED)

    _pipes_split = gpd.overlay(
        _np[["geometry"]],
        _muni_poly,
        how="intersection",
        keep_geom_type=False,
    )

    _pipes_split = _pipes_split[
        _pipes_split.geometry.notna() & (~_pipes_split.geometry.is_empty)
    ].copy()

    _pipes_split["seg_length_m"] = _pipes_split.geometry.length
    _pipes_split["seg_cost_eur"] = _pipes_split["seg_length_m"] * PIPE_COST_EUR_PER_M

    _cost_split_raw = (
        _pipes_split
        .groupby("municipality", as_index=False)
        .agg(
            length_m=("seg_length_m", "sum"),
            cost_eur=("seg_cost_eur", "sum"),
            n_segments=("seg_length_m", "count"),
        )
    )

    block15_cost_by_municipality = (
        pd.DataFrame({"municipality": _ALL_MUNIS})
        .merge(_cost_split_raw, on="municipality", how="left")
    )

    for _c in ["length_m", "cost_eur", "n_segments"]:
        block15_cost_by_municipality[_c] = block15_cost_by_municipality[_c].fillna(0.0)

    block15_cost_by_municipality["length_km"] = (
        block15_cost_by_municipality["length_m"] / 1000.0
    )

    block15_cost_by_municipality["share_percent"] = (
        block15_cost_by_municipality["cost_eur"] / total_new_cost_eur * 100.0
        if total_new_cost_eur > 0 else 0.0
    )

    captured_length_m = float(block15_cost_by_municipality["length_m"].sum())
    outside_length_m = total_new_length_m - captured_length_m
    outside_cost_eur = outside_length_m * PIPE_COST_EUR_PER_M

    if abs(outside_length_m) < 1e-6:
        outside_length_m = 0.0
        outside_cost_eur = 0.0

    outside_row = pd.DataFrame([{
        "municipality": "(outside any polygon)",
        "length_m": outside_length_m,
        "cost_eur": outside_cost_eur,
        "n_segments": np.nan,
        "length_km": outside_length_m / 1000.0,
        "share_percent": _pct(outside_cost_eur, total_new_cost_eur),
    }])

    block15_cost_table = pd.concat(
        [block15_cost_by_municipality, outside_row],
        ignore_index=True,
    )

    print(f"\n  {'Municipality':<26} {'Length km':>10} {'Cost €':>15} {'Share %':>9}")
    print(f"  {'─' * 26} {'─' * 10} {'─' * 15} {'─' * 9}")

    for _, _r in block15_cost_table.iterrows():
        print(
            f"  {_r['municipality']:<26} "
            f"{_r['length_km']:>10.2f} "
            f"{_r['cost_eur']:>15,.0f} "
            f"{_r['share_percent']:>9.1f}"
        )

    print(f"  {'─' * 26} {'─' * 10} {'─' * 15} {'─' * 9}")
    print(
        f"  {'TOTAL':<26} "
        f"{total_new_length_km:>10.2f} "
        f"{total_new_cost_eur:>15,.0f} "
        f"{100.0:>9.1f}"
    )

    print("\n  Consistency check:")
    print(f"    Length from original new_pipes_p_h : {total_new_length_km:,.3f} km")
    print(f"    Length inside municipalities     : {captured_length_m / 1000.0:,.3f} km")
    print(f"    Length outside polygons          : {outside_length_m / 1000.0:,.3f} km")
    print(f"    Reconstructed total              : {(captured_length_m + outside_length_m) / 1000.0:,.3f} km")

    if outside_length_m < -1.0:
        print("    ⚠ WARNING: municipality polygons may overlap; captured length exceeds original length.")

    _len_w1_km = remaining_existing_pipes_WWTP1_p_h.to_crs(CRS_PROJECTED).geometry.length.sum() / 1000.0
    _len_w2_km = selected_existing_pipes_to_WWTP2_p_h.to_crs(CRS_PROJECTED).geometry.length.sum() / 1000.0

    print(f"\n  Existing pipes reused (no additional implementation cost):")
    print(
        f"    WWTP1 retained : {len(remaining_existing_pipes_WWTP1_p_h):>5} pipes  "
        f"{_len_w1_km:>7.2f} km"
    )
    print(
        f"    WWTP2 reused   : {len(selected_existing_pipes_to_WWTP2_p_h):>5} pipes  "
        f"{_len_w2_km:>7.2f} km"
    )

    block15_cost_table.to_csv(
        heuristic_dir / f"Block15_{heuristic_name}_new_pipe_cost_by_municipality.csv",
        index=False,
    )


    # ============================================================
    # § 2  POPULATION DISTRIBUTION
    #       Two stages:
    #       - post-Block-12.5: final two-network assignment before pipe balancing
    #       - post-Block-14: final assignment after pipe-based reassignment
    # ============================================================

    print("\n" + "─" * 80)
    print("§ 2  POPULATION DISTRIBUTION")
    print(f"     Wastewater: {WASTEWATER_L_PER_CAPITA_DAY:.0f} L/cap/day")
    print(f"     Formula: pop × {WASTEWATER_L_PER_CAPITA_DAY:.0f} / 1000 = m³/day")
    print("─" * 80)

    _nodes_125 = _prepare_node_assignment_table(
        nodes_gdf=final_load_nodes_p_h,
        stage_label="post-Block-12.5",
        wwtp1_col="assigned_to_existing_WWTP1",
        wwtp2_col="assigned_to_new_WWTP2",
        assignment_col="post125_wwtp_assignment",
    )

    _nodes_14 = _prepare_node_assignment_table(
        nodes_gdf=balanced_load_nodes_p_h,
        stage_label="post-Block-14",
        wwtp1_col="assigned_to_existing_WWTP1_balanced",
        wwtp2_col="assigned_to_new_WWTP2_balanced",
        assignment_col="post14_wwtp_assignment",
    )

    muni_total_pop = (
        _nodes_125
        .groupby("municipality")["assigned_population"]
        .sum()
        .reindex(_ALL_MUNIS)
        .fillna(0.0)
    )

    P_reference = float(muni_total_pop.sum())
    Q_reference = P_reference * WASTEWATER_L_PER_CAPITA_DAY / 1000.0

    print(f"\n  Reference urban population: {P_reference:,.0f}")
    print(f"  Reference wastewater load:  {Q_reference:,.0f} m³/day")


    def _build_population_distribution_table(stage_name, nodes, assignment_col):
        rows = []

        for wwtp in _WWTP_LABELS:
            for muni in _ALL_MUNIS:
                m_total = float(muni_total_pop.get(muni, 0.0))

                assigned_pop = float(
                    nodes.loc[
                        (nodes["municipality"] == muni)
                        & (nodes[assignment_col] == wwtp),
                        "assigned_population"
                    ].sum()
                )

                rows.append({
                    "stage": stage_name,
                    "wwtp": wwtp,
                    "municipality": muni,
                    "municipality_total_population": m_total,
                    "population_covered": assigned_pop,
                    "share_of_municipality_percent": _pct(assigned_pop, m_total),
                    "share_of_total_population_percent": _pct(assigned_pop, P_reference),
                    "wastewater_m3_day": assigned_pop * WASTEWATER_L_PER_CAPITA_DAY / 1000.0,
                })

        return pd.DataFrame(rows)


    def _print_wwtp_tables(stage_title, dist_df):
        print(f"\n  {stage_title}")

        stage_total_pop = 0.0
        stage_total_q = 0.0
        wwtp_totals = {}

        for wwtp in _WWTP_LABELS:
            sub = dist_df[dist_df["wwtp"] == wwtp].copy()

            print(f"\n  {wwtp}")
            print(
                f"  {'Municipality':<26} "
                f"{'Pop covered':>13} "
                f"{'% of muni':>10} "
                f"{'% of total':>11} "
                f"{'m³/day':>10}"
            )
            print(
                f"  {'─' * 26} "
                f"{'─' * 13} "
                f"{'─' * 10} "
                f"{'─' * 11} "
                f"{'─' * 10}"
            )

            for _, r in sub.iterrows():
                print(
                    f"  {r['municipality']:<26} "
                    f"{r['population_covered']:>13,.0f} "
                    f"{r['share_of_municipality_percent']:>9.1f}% "
                    f"{r['share_of_total_population_percent']:>10.1f}% "
                    f"{r['wastewater_m3_day']:>10,.0f}"
                )

            sp = float(sub["population_covered"].sum())
            sq = float(sub["wastewater_m3_day"].sum())

            wwtp_totals[wwtp] = {
                "population": sp,
                "wastewater_m3_day": sq,
            }

            stage_total_pop += sp
            stage_total_q += sq

            print(
                f"  {'─' * 26} "
                f"{'─' * 13} "
                f"{'─' * 10} "
                f"{'─' * 11} "
                f"{'─' * 10}"
            )
            print(
                f"  {'→ subtotal':<26} "
                f"{sp:>13,.0f} "
                f"{'':>10} "
                f"{_pct(sp, P_reference):>10.1f}% "
                f"{sq:>10,.0f}"
            )

        print("\n  Stage totals:")
        print(
            f"    WWTP1 South : "
            f"{wwtp_totals['WWTP_1_SOUTH']['population']:>10,.0f}  "
            f"({_pct(wwtp_totals['WWTP_1_SOUTH']['population'], P_reference):.1f}%)  "
            f"{wwtp_totals['WWTP_1_SOUTH']['wastewater_m3_day']:>10,.0f} m³/day"
        )
        print(
            f"    WWTP2 North : "
            f"{wwtp_totals['WWTP_2_NORTH']['population']:>10,.0f}  "
            f"({_pct(wwtp_totals['WWTP_2_NORTH']['population'], P_reference):.1f}%)  "
            f"{wwtp_totals['WWTP_2_NORTH']['wastewater_m3_day']:>10,.0f} m³/day"
        )
        print(
            f"    TOTAL       : "
            f"{stage_total_pop:>10,.0f}  "
            f"({_pct(stage_total_pop, P_reference):.1f}%)  "
            f"{stage_total_q:>10,.0f} m³/day"
        )

        if abs(stage_total_pop - P_reference) > 1e-6:
            print(f"    ⚠ Unassigned / not covered difference: {P_reference - stage_total_pop:+,.6f} persons")

        return (
            wwtp_totals["WWTP_1_SOUTH"]["population"],
            wwtp_totals["WWTP_2_NORTH"]["population"],
            wwtp_totals["WWTP_1_SOUTH"]["wastewater_m3_day"],
            wwtp_totals["WWTP_2_NORTH"]["wastewater_m3_day"],
        )


    block15_population_distribution_post125 = _build_population_distribution_table(
        stage_name="post_block_12_5",
        nodes=_nodes_125,
        assignment_col="post125_wwtp_assignment",
    )

    block15_population_distribution_post14 = _build_population_distribution_table(
        stage_name="post_block_14_pipe_based",
        nodes=_nodes_14,
        assignment_col="post14_wwtp_assignment",
    )

    P1_b125_calc, P2_b125_calc, Q1_b125_calc, Q2_b125_calc = _print_wwtp_tables(
        "── POST-BLOCK-12.5  (before pipe-based reassignment) ──",
        block15_population_distribution_post125,
    )

    P1_b14_calc, P2_b14_calc, Q1_b14_calc, Q2_b14_calc = _print_wwtp_tables(
        "── POST-BLOCK-14  (final, after pipe-based reassignment) ──",
        block15_population_distribution_post14,
    )

    print(f"\n  Transfer relative to post-Block-12.5:")
    print(f"    Δ WWTP1 South : {P1_b14_calc - P1_b125_calc:+,.0f} persons")
    print(f"    Δ WWTP2 North : {P2_b14_calc - P2_b125_calc:+,.0f} persons")

    block15_node_assignment_post14 = _nodes_14[
        [
            "node_id",
            "municipality",
            "assigned_population",
            "wastewater_m3_day",
            "post14_wwtp_assignment",
            "assigned_to_existing_WWTP1_balanced",
            "assigned_to_new_WWTP2_balanced",
        ]
    ].copy()

    if "component" in _nodes_14.columns:
        block15_node_assignment_post14["component"] = _nodes_14["component"]

    block15_node_assignment_post14.to_csv(
        heuristic_dir / f"Block15_{heuristic_name}_node_assignment_from_Block14.csv",
        index=False,
    )

    block15_population_distribution_post125.to_csv(
        heuristic_dir / f"Block15_{heuristic_name}_population_distribution_post_block12_5.csv",
        index=False,
    )

    block15_population_distribution_post14.to_csv(
        heuristic_dir / f"Block15_{heuristic_name}_population_distribution_post_block14.csv",
        index=False,
    )


    # ============================================================
    # § 3  ADDITIONAL KPIs
    # ============================================================

    print("\n" + "─" * 80)
    print("§ 3  ADDITIONAL KPIs")
    print("─" * 80)

    covered_post14_mask = (
        _nodes_14["post14_wwtp_assignment"].isin(_WWTP_LABELS)
    )

    P_covered = float(_nodes_14.loc[covered_post14_mask, "assigned_population"].sum())
    P_uncovered = float(_nodes_14.loc[~covered_post14_mask, "assigned_population"].sum())

    print(
        f"\n  Load-eligible population served   : "
        f"{P_covered:>10,.0f}  ({_pct(P_covered, P_covered + P_uncovered):.1f}%)"
    )
    print(
        f"  Load-eligible population unserved : "
        f"{P_uncovered:>10,.0f}  ({_pct(P_uncovered, P_covered + P_uncovered):.1f}%)"
    )

    Qt_b14 = Q1_b14_calc + Q2_b14_calc

    print(f"\n  Wastewater load, post-Block-14 ({WASTEWATER_L_PER_CAPITA_DAY:.0f} L/cap/day):")
    print(f"    WWTP1 South : {Q1_b14_calc:>10,.0f} m³/day  ({_pct(Q1_b14_calc, Qt_b14):.1f}%)")
    print(f"    WWTP2 North : {Q2_b14_calc:>10,.0f} m³/day  ({_pct(Q2_b14_calc, Qt_b14):.1f}%)")
    print(
        f"    Imbalance   : {abs(Q1_b14_calc - Q2_b14_calc):>10,.0f} m³/day  "
        f"({_pct(abs(Q1_b14_calc - Q2_b14_calc), Qt_b14):.1f}%)"
    )

    if pumping_stations_p is not None:
        print(f"\n  Pumping stations in dataset : {len(pumping_stations_p)}")


    # ============================================================
    # § 3b  POPULATION COVERAGE BY MUNICIPALITY
    #        - urban population  : nodes in _nodes_14 (urban demand nodes)
    #        - total population  : MUNICIPALITY_POPULATION dict (census values)
    # ============================================================

    print("\n" + "─" * 80)
    print("§ 3b  POPULATION COVERAGE BY MUNICIPALITY")
    print("─" * 80)

    print(
        f"\n  {'Municipality':<26} "
        f"{'Urban pop':>10} "
        f"{'Covered (urban)':>15} "
        f"{'% urban':>8} "
        f"{'Total pop':>10} "
        f"{'% total':>8}"
    )
    print(
        f"  {'─' * 26} "
        f"{'─' * 10} "
        f"{'─' * 15} "
        f"{'─' * 8} "
        f"{'─' * 10} "
        f"{'─' * 8}"
    )

    _cov_urban_total  = 0.0
    _ref_urban_total  = 0.0
    _ref_census_total = 0.0

    for muni in _ALL_MUNIS:
        urban_pop = float(muni_total_pop.get(muni, 0.0))

        covered_urban = float(
            _nodes_14.loc[
                (_nodes_14["municipality"] == muni)
                & (_nodes_14["post14_wwtp_assignment"].isin(_WWTP_LABELS)),
                "assigned_population",
            ].sum()
        )

        census_pop = float(MUNICIPALITY_POPULATION.get(muni, 0.0))

        pct_urban  = _pct(covered_urban, urban_pop)  if urban_pop  > 0 else 0.0
        pct_census = _pct(covered_urban, census_pop) if census_pop > 0 else 0.0

        print(
            f"  {muni:<26} "
            f"{urban_pop:>10,.0f} "
            f"{covered_urban:>15,.0f} "
            f"{pct_urban:>7.1f}% "
            f"{census_pop:>10,.0f} "
            f"{pct_census:>7.1f}%"
        )

        _cov_urban_total  += covered_urban
        _ref_urban_total  += urban_pop
        _ref_census_total += census_pop

    print(
        f"  {'─' * 26} "
        f"{'─' * 10} "
        f"{'─' * 15} "
        f"{'─' * 8} "
        f"{'─' * 10} "
        f"{'─' * 8}"
    )
    print(
        f"  {'TOTAL':<26} "
        f"{_ref_urban_total:>10,.0f} "
        f"{_cov_urban_total:>15,.0f} "
        f"{_pct(_cov_urban_total, _ref_urban_total):>7.1f}% "
        f"{_ref_census_total:>10,.0f} "
        f"{_pct(_cov_urban_total, _ref_census_total):>7.1f}%"
    )

    print(
        f"\n  Note: 'Covered' = nodes with a WWTP assignment post-Block-14."
        f"\n  '% urban' uses demand-node urban pop as denominator."
        f"\n  '% total' uses MUNICIPALITY_POPULATION (census) as denominator."
    )


    # ============================================================
    # § 4  MAP
    # ============================================================

    print("\n" + "─" * 80)
    print("§ 4  MAP")
    print("─" * 80)

    if "HAS_FOLIUM" not in globals() or not HAS_FOLIUM:
        print("⚠ folium not installed — skipped.")
    else:
        import folium
        from folium.plugins import MiniMap, Fullscreen

        _muni_w = muni_boundaries_p.to_crs(CRS_WGS84)
        _pipes_w1_w = remaining_existing_pipes_WWTP1_p_h.to_crs(CRS_WGS84)
        _pipes_w2_w = selected_existing_pipes_to_WWTP2_p_h.to_crs(CRS_WGS84)
        _new_w = new_pipes_p_h.to_crs(CRS_WGS84)
        _center = _muni_w.geometry.union_all().centroid

        m15 = folium.Map(
            location=[_center.y, _center.x],
            zoom_start=11,
            tiles="OpenStreetMap"
        )

        Fullscreen().add_to(m15)
        MiniMap(toggle_display=True).add_to(m15)

        _fg_muni = folium.FeatureGroup(name="Municipal boundaries", show=True)

        _north_for_map = (
            NORTH_MUNICIPALITIES_TARGET if "NORTH_MUNICIPALITIES_TARGET" in globals()
            else []
        )

        for _, _r in _muni_w.iterrows():
            _is_north = _r["municipality"] in _north_for_map

            folium.GeoJson(
                _r.geometry,
                tooltip=_r["municipality"],
                style_function=lambda x, is_north=_is_north: {
                    "fillColor": "#ffffff",
                    "fillOpacity": 0.0,
                    "color": "#D32F2F" if is_north else "#444444",
                    "weight": 2 if is_north else 1,
                },
            ).add_to(_fg_muni)

        _fg_muni.add_to(m15)

        _fg_w1 = folium.FeatureGroup(name="Existing pipes → WWTP1", show=True)

        for _, _r in _pipes_w1_w.iterrows():
            folium.GeoJson(
                _r.geometry,
                tooltip="WWTP1 existing pipe",
                popup=(
                    f"Pipe: {_r.get('existing_conduit_id', '')}<br>"
                    f"Associated pop: {_r.get('associated_population', 0):,.0f}"
                ),
                style_function=lambda x: {
                    "color": "#1565C0",
                    "weight": 3,
                    "opacity": 0.75,
                },
            ).add_to(_fg_w1)

        _fg_w1.add_to(m15)

        _fg_w2e = folium.FeatureGroup(name="Existing pipes → WWTP2", show=True)

        for _, _r in _pipes_w2_w.iterrows():
            folium.GeoJson(
                _r.geometry,
                tooltip="WWTP2 reassigned existing pipe",
                popup=(
                    f"Pipe: {_r.get('existing_conduit_id', '')}<br>"
                    f"Reason: {_r.get('pipe_reassignment_reason', '')}<br>"
                    f"Associated pop: {_r.get('associated_population', 0):,.0f}"
                ),
                style_function=lambda x: {
                    "color": "#F57C00",
                    "weight": 4,
                    "opacity": 0.90,
                },
            ).add_to(_fg_w2e)

        _fg_w2e.add_to(m15)

        _fg_new = folium.FeatureGroup(name="New pipes → WWTP2", show=True)

        for _, _r in _new_w.iterrows():
            folium.GeoJson(
                _r.geometry,
                tooltip="New pipe for WWTP2",
                popup=(
                    f"Municipality: {_r.get('municipality', '')}<br>"
                    f"Length: {_r.get('length_m', 0):,.0f} m<br>"
                    f"Cost: €{_r.get('cost_eur', 0):,.0f}"
                ),
                style_function=lambda x: {
                    "color": "#C62828",
                    "weight": 4,
                    "opacity": 0.95,
                },
            ).add_to(_fg_new)

        _fg_new.add_to(m15)

        if pumping_stations_p is not None:
            _ps_w = pumping_stations_p.to_crs(CRS_WGS84)
            _fg_ps = folium.FeatureGroup(name="Pumping stations", show=True)

            for _, _r in _ps_w.iterrows():
                _pt = _r.geometry if _r.geometry.geom_type == "Point" else _r.geometry.centroid

                folium.CircleMarker(
                    location=[_pt.y, _pt.x],
                    radius=8,
                    color="#6A1B9A",
                    fill=True,
                    fill_color="#AB47BC",
                    fill_opacity=0.85,
                    weight=2,
                    tooltip=_r.get("ps_name", "Pumping station"),
                    popup=f"<b>Pumping station</b><br>{_r.get('ps_name', '')}",
                ).add_to(_fg_ps)

            _fg_ps.add_to(m15)

        folium.Marker(
            location=[WWTP2_LATLON[0], WWTP2_LATLON[1]],
            popup="Proposed WWTP2 North",
            icon=folium.Icon(color="red", icon="tint"),
        ).add_to(m15)

        _stats_html = f"""
        <div style="position:fixed;top:10px;right:10px;width:300px;z-index:9999;
            background:rgba(255,255,255,0.95);border:2px solid #333;
            border-radius:6px;padding:10px;font-size:9.5pt;font-family:Arial,sans-serif;">
          <b>Block 15 — Network statistics</b><br><hr style="margin:4px 0">
          <b>Cost: new pipes only</b><br>
          {total_new_length_km:.1f} km &nbsp; €{total_new_cost_eur/1e6:.2f} M<br>
          <hr style="margin:4px 0">
          <b>Population — post-Block-14</b><br>
          WWTP1 South: {P1_b14_calc:,.0f} ({_pct(P1_b14_calc, P_reference):.1f}%)<br>
          WWTP2 North: {P2_b14_calc:,.0f} ({_pct(P2_b14_calc, P_reference):.1f}%)<br>
          <hr style="margin:4px 0">
          <b>Wastewater ({WASTEWATER_L_PER_CAPITA_DAY:.0f} L/cap/day)</b><br>
          WWTP1: {Q1_b14_calc:,.0f} m³/day &nbsp; WWTP2: {Q2_b14_calc:,.0f} m³/day<br>
        </div>
        """

        m15.get_root().html.add_child(folium.Element(_stats_html))

        _legend_html = """
        <div style="position:fixed;bottom:35px;left:35px;width:310px;z-index:9999;
            background:rgba(255,255,255,0.93);border:2px solid #333;
            border-radius:6px;padding:10px;font-size:9.5pt;">
          <b>Legend</b><br>
          <span style="color:#1565C0;font-weight:bold;">━━</span> Existing pipes → WWTP1<br>
          <span style="color:#F57C00;font-weight:bold;">━━</span> Existing pipes → WWTP2<br>
          <span style="color:#C62828;font-weight:bold;">━━</span> New pipes → WWTP2<br>
          <span style="color:#AB47BC;font-size:14pt;">●</span> Pumping station<br>
        </div>
        """

        m15.get_root().html.add_child(folium.Element(_legend_html))

        folium.LayerControl(collapsed=False).add_to(m15)

        _out_map = heuristic_dir / f"Block15_{heuristic_name}_final_network_map.html"
        m15.save(_out_map)

        print(f"\n  ✓ Map saved: {_out_map}")


    # ============================================================
    # § 5  GEOPACKAGE EXPORT FOR QGIS
    # ============================================================

    print("\n" + "─" * 80)
    print("§ 5  GEOPACKAGE EXPORT (for QGIS manual editing)")
    print("─" * 80)

    out_qgis_gpkg = heuristic_dir / f"Block15_{heuristic_name}_network_for_qgis_edit.gpkg"

    if out_qgis_gpkg.exists():
        out_qgis_gpkg.unlink()

    _np_export = new_pipes_p_h.copy()

    if _np_export.crs is None:
        _np_export = _np_export.set_crs(CRS_PROJECTED)
    else:
        _np_export = _np_export.to_crs(CRS_PROJECTED)

    if "length_m" not in _np_export.columns:
        _np_export["length_m"] = _np_export.geometry.length

    if "cost_eur" not in _np_export.columns:
        _np_export["cost_eur"] = _np_export["length_m"] * PIPE_COST_EUR_PER_M

    _np_export.to_file(out_qgis_gpkg, layer="new_pipes_WWTP2", driver="GPKG")

    remaining_existing_pipes_WWTP1_p_h.to_crs(CRS_PROJECTED).to_file(
        out_qgis_gpkg,
        layer="existing_pipes_WWTP1",
        driver="GPKG",
    )

    selected_existing_pipes_to_WWTP2_p_h.to_crs(CRS_PROJECTED).to_file(
        out_qgis_gpkg,
        layer="existing_pipes_WWTP2",
        driver="GPKG",
    )

    balanced_load_nodes_p_h.to_crs(CRS_PROJECTED).to_file(
        out_qgis_gpkg,
        layer="demand_nodes",
        driver="GPKG",
    )

    muni_boundaries_p.to_crs(CRS_PROJECTED).to_file(
        out_qgis_gpkg,
        layer="municipal_boundaries",
        driver="GPKG",
    )

    if pumping_stations_p is not None:
        pumping_stations_p.to_crs(CRS_PROJECTED).to_file(
            out_qgis_gpkg,
            layer="pumping_stations",
            driver="GPKG",
        )

    print(f"\n  ✓ GeoPackage saved: {out_qgis_gpkg}")

    print("\n  Layers:")
    print("    • new_pipes_WWTP2")
    print("    • existing_pipes_WWTP1")
    print("    • existing_pipes_WWTP2")
    print("    • demand_nodes")
    print("    • municipal_boundaries")

    if pumping_stations_p is not None:
        print("    • pumping_stations")


    # ============================================================
    # § 6  CSV OUTPUTS
    # ============================================================

    print("\n" + "─" * 80)
    print("§ 6  CSV OUTPUTS")
    print("─" * 80)

    print("\n  ✓ Saved:")
    print(f"    {heuristic_dir / 'Step1_Block15_new_pipe_cost_by_municipality.csv'}")
    print(f"    {heuristic_dir / 'Step1_Block15_population_distribution_post_block12_5.csv'}")
    print(f"    {heuristic_dir / 'Step1_Block15_population_distribution_post_block14.csv'}")
    print(f"    {heuristic_dir / 'Step1_Block15_node_assignment_from_Block14.csv'}")

    print("\n" + "=" * 80)
    print("BLOCK 15 complete.")
    print("=" * 80)
    print(f"\n✓ Heuristic '{heuristic_name}' complete. Outputs in: {heuristic_dir}")

print("\n" + "=" * 80)
print("ALL HEURISTICS COMPLETE.")
print("=" * 80)



################################################################################
# HEURISTIC: rooted_dijkstra
################################################################################

BLOCK 24 [rooted_dijkstra] -- FINAL TWO-NETWORK MAP AND WWTP LOAD SUMMARY
Existing conduits for final map:
Rows: 6,379
No south opportunistic nodes found (Block 12.2 not run or no nodes identified).

Unassigned northern urban demand after corrected merge:
Nodes: 1
Population: 0.107
Wastewater: 0.013 m³/day


C:\Users\lucam\AppData\Local\Temp\ipykernel_14652\1526405493.py:181: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,node_id,municipality,component,requires_new_network,covered_by_new_network,assigned_to_existing_WWTP1,assigned_to_new_WWTP2,assigned_population,wastewater_m3_day
20421,65284,Pyla,16,True,False,False,False,0.10744,0.012893



=== Final network load summary -- URBAN DEMAND ONLY ===


,network,definition,n_demand_nodes,n_buildings,population,wastewater_m3_day,population_share_percent,wastewater_share_percent
0,Existing network / WWTP1,Urban demand in south municipalities + urban n...,12271,30606,76666.271948,9199.952634,70.252665,70.252665
1,New network / WWTP2,Urban demand nodes covered by proposed new net...,7482,18035,32463.071916,3895.568630,29.747335,29.747335
2,Unassigned after new network,Northern urban demand nodes not assigned to ex...,1,1,0.107440,0.012893,0.000098,0.000098


...


=== Pipe assignment summary ===


,pipe_final_assignment,pipe_reassignment_reason,n_pipes,associated_population,associated_wastewater_m3_day
0,WWTP1_existing,WWTP1_existing,4287,46696.209437,5603.545132
1,WWTP2_reassigned,Stage_A_northern_pipes,2092,11419.672258,1370.360671



=== Final transfer summary ===


,target_WWTP2_population_share,target_WWTP2_population,stage_A_pipe_transfer_population,stage_B_pipe_transfer_population,orphan_node_transfer_population,total_pipe_transfer_population,stage_A_pipe_transfer_wastewater_m3_day,stage_B_pipe_transfer_wastewater_m3_day,n_stage_A_pipes_to_WWTP2,n_stage_B_pipes_to_WWTP2,...,WWTP2_population_before,WWTP1_population_after,WWTP2_population_after,WWTP1_share_before_percent,WWTP2_share_before_percent,WWTP1_share_after_percent,WWTP2_share_after_percent,total_population_before,total_population_after,population_difference_after_minus_before
0,0.4,43651.737546,11419.672258,0.0,1137.201723,11419.672258,1370.360671,0.0,2092,0,...,32463.071916,64109.397966,45019.945898,70.252665,29.747335,58.746251,41.253749,109129.343864,109129.343864,0.0



Population conservation check:
Before: 109,129.343864
After:  109,129.343864
Diff:   0.000000000000

Saved:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\heuristic_prim_steiner\Block14_pipe_based_WWTP_reassignment.gpkg
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\heuristic_prim_steiner\Block14_balance_stage_summary.csv
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\heuristic_prim_steiner\Block14_pipe_assignment_summary.csv
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\heuristic_prim_steiner\Block14_transfer_summary.csv
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\heuristic_prim_steiner\Block14_stage_B_pipe_iteration_summary.csv
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\NetworkNonBudgeted\heuristic_prim_steiner\Block14_existing_pipes_reassigned_to_WWTP2.csv

BLOCK 25.5 [prim_steiner] -- FINAL WWTP PIPE ASSIGNMENT MAP
Saved simple fina

---
## BLOCK 24 — Cross-Heuristic Final Summary

Reads the per-heuristic downstream outputs once the remaining pipeline has been run for
all three heuristics and builds a consolidated comparison table for the report.


In [24]:
# ============================================================
# BLOCK 24 -- CROSS-HEURISTIC FINAL SUMMARY
# ============================================================

print("=" * 80)
print("BLOCK 24 -- CROSS-HEURISTIC FINAL SUMMARY")
print("=" * 80)

def _pct(part, total):
    if not total or total == 0:
        return 0.0
    return 100.0 * float(part) / float(total)

HEURISTIC_DATA = {
    "rooted_dijkstra": {
        "pipes":   new_pipes_A_p,
        "summary": new_network_summary_A,
    },
    "prim_steiner": {
        "pipes":   new_pipes_B_p,
        "summary": new_network_summary_B,
    },
}

# Exact filenames written by the Block 24-26 loop
def _h_files(h_name):
    d = HEURISTIC_DIRS[h_name]
    return {
        "cost":   d / f"Block15_{h_name}_new_pipe_cost_by_municipality.csv",
        "pop14":  d / f"Block15_{h_name}_population_distribution_post_block14.csv",
        "pop125": d / f"Block15_{h_name}_population_distribution_post_block12_5.csv",
    }

# ── § 1  NEW PIPE NETWORK — ROUTING COMPARISON ───────────────
print("\n" + "─" * 80)
print("§ 1  NEW PIPE NETWORK — ROUTING COMPARISON")
print("     'Required nodes cov%' = required population covered / total required population")
print("─" * 80)

print(
    f"\n  {'Heuristic':<20} "
    f"{'Length km':>10} {'Cost €':>15} "
    f"{'Req.pop cov%':>14} {'vs baseline':>12}"
)
print(f"  {'─'*20} {'─'*10} {'─'*15} {'─'*14} {'─'*12}")

routing_rows = []
for h_name, h_data in HEURISTIC_DATA.items():
    pipes  = h_data["pipes"]
    summ   = h_data["summary"]
    length_km = float(pipes["length_m"].sum()) / 1000.0
    cost_eur  = float(pipes["cost_eur"].sum())
    if len(summ) > 0 and "required_population_covered" in summ.columns \
            and "required_population" in summ.columns:
        cov_pct = _pct(
            float(summ["required_population_covered"].sum()),
            float(summ["required_population"].sum())
        )
    else:
        cov_pct = float(summ["required_node_coverage_percent"].mean()) if len(summ) else 0.0
    routing_rows.append({
        "heuristic": h_name, "length_km": length_km,
        "cost_eur": cost_eur, "coverage_pct": cov_pct,
    })

baseline_len  = routing_rows[0]["length_km"]
baseline_cost = routing_rows[0]["cost_eur"]
for r in routing_rows:
    vs = _pct(r["length_km"] - baseline_len, baseline_len)
    sign = "+" if vs >= 0 else ""
    print(
        f"  {r['heuristic']:<20} "
        f"{r['length_km']:>10.2f} {r['cost_eur']:>15,.0f} "
        f"{r['coverage_pct']:>13.1f}% {sign}{vs:>10.1f}%"
    )

# ── § 2  PRE vs POST BALANCING (per heuristic) ───────────────
print("\n" + "─" * 80)
print("§ 2  PRE vs POST BALANCING — WWTP SPLIT")
print("─" * 80)

for h_name in HEURISTIC_DATA:
    f = _h_files(h_name)
    if not f["pop14"].exists():
        print(f"\n  {h_name}: {f['pop14'].name} not found.")
        continue
    df = pd.read_csv(f["pop14"])
    pre  = df[df["stage"] == "post_block_12_5"]
    post = df[df["stage"] == "post_block_14_pipe_based"]

    def _split(d):
        w1 = float(d.loc[d["wwtp"] == "WWTP_1_SOUTH", "population_covered"].sum())
        w2 = float(d.loc[d["wwtp"] == "WWTP_2_NORTH", "population_covered"].sum())
        return w1, w2

    w1p, w2p = _split(pre)
    w1o, w2o = _split(post)
    tot = w1p + w2p

    print(f"\n  {h_name}")
    print(f"  {'Stage':<35} {'WWTP1':>10} {'WWTP2':>10} {'WWTP2%':>7}")
    print(f"  {'─'*35} {'─'*10} {'─'*10} {'─'*7}")
    print(f"  {'Pre-balancing  (post Block 12.5)':<35} {w1p:>10,.0f} {w2p:>10,.0f} {_pct(w2p,tot):>6.1f}%")
    print(f"  {'Post-balancing (post Block 25)':<35} {w1o:>10,.0f} {w2o:>10,.0f} {_pct(w2o,tot):>6.1f}%")
    print(f"  {'Δ WWTP2':<35} {w2o-w2p:>+10,.0f}")

# ── § 3  POPULATION POST-BALANCING ───────────────────────────
print("\n" + "─" * 80)
print("§ 3  POPULATION AND WWTP LOAD — POST BALANCING")
print("─" * 80)

print(
    f"\n  {'Heuristic':<20} "
    f"{'WWTP1 pop':>10} {'WWTP2 pop':>10} {'WWTP2%':>7} "
    f"{'WWTP1 m³/d':>11} {'WWTP2 m³/d':>11} {'Imbalance':>10}"
)
print(f"  {'─'*20} {'─'*10} {'─'*10} {'─'*7} {'─'*11} {'─'*11} {'─'*10}")

pop_rows = []
for h_name in HEURISTIC_DATA:
    f = _h_files(h_name)
    if not f["pop14"].exists():
        print(f"  {h_name:<20} -- {f['pop14'].name} not found --")
        continue
    df = pd.read_csv(f["pop14"])
    post = df[df["stage"] == "post_block_14_pipe_based"]
    w1p = float(post.loc[post["wwtp"] == "WWTP_1_SOUTH", "population_covered"].sum())
    w2p = float(post.loc[post["wwtp"] == "WWTP_2_NORTH", "population_covered"].sum())
    w1q = float(post.loc[post["wwtp"] == "WWTP_1_SOUTH", "wastewater_m3_day"].sum())
    w2q = float(post.loc[post["wwtp"] == "WWTP_2_NORTH", "wastewater_m3_day"].sum())
    imb = abs(w1q - w2q)
    print(
        f"  {h_name:<20} "
        f"{w1p:>10,.0f} {w2p:>10,.0f} {_pct(w2p,w1p+w2p):>6.1f}% "
        f"{w1q:>11,.0f} {w2q:>11,.0f} {imb:>10,.0f}"
    )
    pop_rows.append({
        "heuristic": h_name,
        "WWTP1_pop": w1p, "WWTP2_pop": w2p,
        "WWTP2_share_pct": _pct(w2p, w1p+w2p),
        "WWTP1_q": w1q, "WWTP2_q": w2q,
    })

# ── § 4  COVERAGE BY MUNICIPALITY — SIDE BY SIDE ─────────────
print("\n" + "─" * 80)
print("§ 4  COVERAGE BY MUNICIPALITY — SIDE BY SIDE (post-Block-25)")
print("─" * 80)

cov_data = {}
for h_name in HEURISTIC_DATA:
    f = _h_files(h_name)
    if f["pop14"].exists():
        df = pd.read_csv(f["pop14"])
        post = df[df["stage"] == "post_block_14_pipe_based"]
        cov_data[h_name] = post.groupby("municipality").agg(
            population_covered=("population_covered", "sum"),
            municipality_total=("municipality_total_population", "first"),
        )

if len(cov_data) == 2:
    h_a, h_b = list(cov_data.keys())
    all_munis = sorted(set(cov_data[h_a].index) | set(cov_data[h_b].index))
    print(f"\n  (A) = {h_a}")
    print(f"  (B) = {h_b}")
    print(
        f"\n  {'Municipality':<26} {'Urban pop':>10} "
        f"{'Cov(A)':>9} {'%(A)':>6} "
        f"{'Cov(B)':>9} {'%(B)':>6} {'Δ pop':>8}"
    )
    print(f"  {'─'*26} {'─'*10} {'─'*9} {'─'*6} {'─'*9} {'─'*6} {'─'*8}")
    for muni in all_munis:
        ra = cov_data[h_a].loc[muni] if muni in cov_data[h_a].index else None
        rb = cov_data[h_b].loc[muni] if muni in cov_data[h_b].index else None
        urban = float(ra["municipality_total"]) if ra is not None else 0.0
        ca = float(ra["population_covered"]) if ra is not None else 0.0
        cb = float(rb["population_covered"]) if rb is not None else 0.0
        d  = cb - ca
        sign = "+" if d >= 0 else ""
        print(
            f"  {muni:<26} {urban:>10,.0f} "
            f"{ca:>9,.0f} {_pct(ca,urban):>5.1f}% "
            f"{cb:>9,.0f} {_pct(cb,urban):>5.1f}% "
            f"{sign}{d:>7,.0f}"
        )

# ── § 5  COST BY MUNICIPALITY — SIDE BY SIDE ─────────────────
print("\n" + "─" * 80)
print("§ 5  NEW PIPE COST BY MUNICIPALITY")
print("─" * 80)

cost_data = {}
for h_name in HEURISTIC_DATA:
    f = _h_files(h_name)
    if f["cost"].exists():
        cost_data[h_name] = pd.read_csv(f["cost"]).set_index("municipality")

if len(cost_data) == 2:
    h_a, h_b = list(cost_data.keys())
    all_munis_c = sorted(set(cost_data[h_a].index) | set(cost_data[h_b].index))
    print(f"\n  (A) = {h_a}")
    print(f"  (B) = {h_b}")
    print(
        f"\n  {'Municipality':<26} "
        f"{'km(A)':>7} {'Cost €(A)':>13} "
        f"{'km(B)':>7} {'Cost €(B)':>13} "
        f"{'Δkm':>7} {'Δ%':>6}"
    )
    print(f"  {'─'*26} {'─'*7} {'─'*13} {'─'*7} {'─'*13} {'─'*7} {'─'*6}")
    tot_ka = tot_kb = tot_ca = tot_cb = 0.0
    for muni in all_munis_c:
        ra = cost_data[h_a].loc[muni] if muni in cost_data[h_a].index else None
        rb = cost_data[h_b].loc[muni] if muni in cost_data[h_b].index else None
        ka = float(ra["length_km"]) if ra is not None else 0.0
        kb = float(rb["length_km"]) if rb is not None else 0.0
        ca = float(ra["cost_eur"])  if ra is not None else 0.0
        cb = float(rb["cost_eur"])  if rb is not None else 0.0
        dk = kb - ka
        dp = _pct(dk, ka) if ka > 0 else 0.0
        sign = "+" if dk >= 0 else ""
        print(
            f"  {muni:<26} "
            f"{ka:>7.2f} {ca:>13,.0f} "
            f"{kb:>7.2f} {cb:>13,.0f} "
            f"{sign}{dk:>6.2f} {sign}{dp:>4.1f}%"
        )
        tot_ka += ka; tot_kb += kb; tot_ca += ca; tot_cb += cb
    dk_tot = tot_kb - tot_ka
    dp_tot = _pct(dk_tot, tot_ka) if tot_ka > 0 else 0.0
    sign = "+" if dk_tot >= 0 else ""
    print(f"  {'─'*26} {'─'*7} {'─'*13} {'─'*7} {'─'*13} {'─'*7} {'─'*6}")
    print(
        f"  {'TOTAL':<26} "
        f"{tot_ka:>7.2f} {tot_ca:>13,.0f} "
        f"{tot_kb:>7.2f} {tot_cb:>13,.0f} "
        f"{sign}{dk_tot:>6.2f} {sign}{dp_tot:>4.1f}%"
    )

# ── Save ──────────────────────────────────────────────────────
if routing_rows:
    final_df = pd.DataFrame(routing_rows)
    if pop_rows:
        final_df = final_df.merge(pd.DataFrame(pop_rows), on="heuristic", how="left")
    out_path = shared_path("final_cross_heuristic_summary.csv")
    final_df.to_csv(out_path, index=False)
    print(f"\nSaved: {out_path}")

print("\n" + "=" * 80)
print("BLOCK 24 complete.")
print("=" * 80)

BLOCK 24 -- CROSS-HEURISTIC FINAL SUMMARY

────────────────────────────────────────────────────────────────────────────────
§ 1  NEW PIPE NETWORK — ROUTING COMPARISON
     'Required nodes cov%' = required population covered / total required population
────────────────────────────────────────────────────────────────────────────────

  Heuristic             Length km          Cost €   Req.pop cov%  vs baseline
  ──────────────────── ────────── ─────────────── ────────────── ────────────
  rooted_dijkstra          525.76     210,305,835         100.0% +       0.0%
  prim_steiner             414.70     165,879,694         100.0%      -21.1%

────────────────────────────────────────────────────────────────────────────────
§ 2  PRE vs POST BALANCING — WWTP SPLIT
────────────────────────────────────────────────────────────────────────────────

  rooted_dijkstra
  Stage                                    WWTP1      WWTP2  WWTP2%
  ─────────────────────────────────── ────────── ────────── ─────